In [1]:
import os
import subprocess
import psutil
import shutil
import torch

print("=" * 60)
print("RAIOS KAGGLE COMPUTE CERTIFICATION")
print("=" * 60)

print("\n[CPU]")
print("Logical CPUs:", os.cpu_count())

print("\n[RAM]")
ram = psutil.virtual_memory()
print(f"Total RAM: {ram.total / 1024**3:.2f} GB")
print(f"Available RAM: {ram.available / 1024**3:.2f} GB")

print("\n[STORAGE]")
disk = shutil.disk_usage("/kaggle/working")
print(f"Total: {disk.total / 1024**3:.2f} GB")
print(f"Free:  {disk.free / 1024**3:.2f} GB")

print("\n[CUDA]")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"\nGPU {i}: {props.name}")
    print(f"VRAM: {props.total_memory / 1024**3:.2f} GB")
    print(f"Compute capability: {props.major}.{props.minor}")

print("\n[NVIDIA-SMI]")
subprocess.run(["nvidia-smi"])

print("\n" + "=" * 60)
print("CERTIFICATION COMPLETE")
print("=" * 60)

RAIOS KAGGLE COMPUTE CERTIFICATION

[CPU]
Logical CPUs: 4

[RAM]
Total RAM: 31.35 GB
Available RAM: 29.80 GB

[STORAGE]
Total: 19.52 GB
Free:  19.50 GB

[CUDA]
PyTorch: 2.10.0+cu128
CUDA available: True
GPU count: 2

GPU 0: Tesla T4
VRAM: 14.56 GB
Compute capability: 7.5

GPU 1: Tesla T4
VRAM: 14.56 GB
Compute capability: 7.5

[NVIDIA-SMI]
Sun Aug 16 22:33:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======

In [2]:
import os
import shutil
import subprocess

print("=" * 60)
print("RAIOS KAGGLE PRE-MODEL GATE")
print("=" * 60)

print("\n[DISK]")
disk = shutil.disk_usage("/kaggle/working")
print(f"Working total: {disk.total / 1024**3:.2f} GB")
print(f"Working free : {disk.free / 1024**3:.2f} GB")

print("\n[HOME CACHE]")
home = shutil.disk_usage("/root")
print(f"/root total: {home.total / 1024**3:.2f} GB")
print(f"/root free : {home.free / 1024**3:.2f} GB")

print("\n[HF CACHE]")
print(os.environ.get("HF_HOME", "HF_HOME not set"))
print(os.environ.get("TRANSFORMERS_CACHE", "TRANSFORMERS_CACHE not set"))

print("\n[INTERNET TEST]")
result = subprocess.run(
    ["python", "-c", "import requests; print(requests.get('https://huggingface.co', timeout=10).status_code)"],
    capture_output=True,
    text=True
)
print(result.stdout.strip() or result.stderr.strip())

print("\n" + "=" * 60)
print("PRE-MODEL GATE COMPLETE")
print("=" * 60)


RAIOS KAGGLE PRE-MODEL GATE

[DISK]
Working total: 19.52 GB
Working free : 19.50 GB

[HOME CACHE]
/root total: 8062.39 GB
/root free : 1026.79 GB

[HF CACHE]
HF_HOME not set
TRANSFORMERS_CACHE not set

[INTERNET TEST]
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 198, in _new_conn
    sock = connection.create_connection(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/urllib3/util/connection.py", line 60, in create_connection
    for res in socket.getaddrinfo(host, port, family, socket.SOCK_STREAM):
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 978, in getaddrinfo
    for res in _socket.getaddrinfo(host, port, family, type, proto, flags):
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
socket.gaierror: [Errno -3] Temporary failure in name resolution

The above exception was the dire

In [3]:
import requests

r = requests.get("https://huggingface.co", timeout=15)

print("HTTP:", r.status_code)
print("INTERNET_PASS:", r.status_code == 200)

HTTP: 200
INTERNET_PASS: True


In [4]:
import os, subprocess, textwrap, json, shutil, sys, pathlib, time

print("=" * 70)
print("RAIOS LLAMA.CPP CUDA BUILD GATE")
print("=" * 70)

work = "/kaggle/working"
repo = f"{work}/llama.cpp"

# 1) Tools check
print("\n[1] TOOLCHAIN")
for cmd in ["git", "cmake", "ninja", "g++"]:
    p = shutil.which(cmd)
    print(f"{cmd}: {p or 'MISSING'}")

# 2) Clone only if missing
print("\n[2] LLAMA.CPP SOURCE")
if not os.path.exists(repo):
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/ggml-org/llama.cpp", repo],
        check=True,
    )
    print("clone: PASS")
else:
    print("clone: SKIP (already exists)")

# 3) Configure CUDA build
print("\n[3] CMAKE CONFIGURE")
build = f"{repo}/build"
os.makedirs(build, exist_ok=True)

subprocess.run(
    [
        "cmake",
        "-S", repo,
        "-B", build,
        "-DGGML_CUDA=ON",
        "-DCMAKE_BUILD_TYPE=Release",
    ],
    check=True,
)

print("configure: PASS")

# 4) Build only the server + minimal CLI
print("\n[4] BUILD")
subprocess.run(
    [
        "cmake",
        "--build", build,
        "--config", "Release",
        "-j", str(min(os.cpu_count() or 2, 4)),
        "--target", "llama-server", "llama-cli",
    ],
    check=True,
)

print("build: PASS")

# 5) Locate binaries
print("\n[5] BINARIES")
candidates = [
    f"{build}/bin/llama-server",
    f"{build}/bin/llama-cli",
]

for p in candidates:
    print(p, "=>", os.path.exists(p))

server = f"{build}/bin/llama-server"
cli = f"{build}/bin/llama-cli"

if not os.path.exists(server):
    raise RuntimeError("STOP: llama-server was not built.")

# 6) Version / CUDA signals
print("\n[6] LLAMA SERVER VERSION")
r = subprocess.run([server, "--version"], capture_output=True, text=True)
print((r.stdout + r.stderr).strip())

print("\n[7] GPU VISIBILITY")
subprocess.run(["nvidia-smi", "-L"], check=True)

print("\n" + "=" * 70)
print("LLAMA_CPP_CUDA_BUILD_PASS")
print("=" * 70)

RAIOS LLAMA.CPP CUDA BUILD GATE

[1] TOOLCHAIN
git: /usr/bin/git
cmake: /usr/local/bin/cmake
ninja: /usr/local/bin/ninja
g++: /usr/bin/g++

[2] LLAMA.CPP SOURCE


Cloning into '/kaggle/working/llama.cpp'...


clone: PASS

[3] CMAKE CONFIGURE
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- llama.cpp version: 0.1.0-dev
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD


CMAKE_BUILD_TYPE=Release


-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Found OpenMP_C: -fopenmp (found version "4.5")
-- Found OpenMP_CXX: -fopenmp (found version "4.5")
-- Found OpenMP: TRUE (found version "4.5")
-- Including CPU backend
-- x86 detected
-- Adding CPU backend variant ggml-cpu: -march=native 
-- Found CUDAToolkit: /usr/local/cuda/targets/x86_64-linux/include (found version "12.8.93")
-- CUDA Toolkit found
-- The CUDA compiler identification is NVIDIA 12.8.93 with host compiler GNU 11.4.0
-- Detecting CUDA compiler ABI info
-- Detecting CUDA compiler ABI info - done
-- Check for working CUDA compiler: /usr/local/cuda/bin/nvcc - skipped
-- Detecting CUDA compile features
-- Detecting CUDA compile features - done
-- Using CMAKE_CUDA_ARCHITECTURES=75-real CMAKE_CUDA_ARCHITECTURES_NATIVE=7

CMake Error at ggml/src/ggml-cuda/CMakeLists.txt:182 (target_link_libraries):
  Target "ggml-cuda" links to:

    CUDA::cuda_driver

  but the target was not found.  Possible reasons include:

    * There is a typo in the target name.
    * A find_package call is missing for an IMPORTED target.
    * An ALIAS target is missing.



CMake Generate step failed.  Build files cannot be regenerated correctly.


CalledProcessError: Command '['cmake', '-S', '/kaggle/working/llama.cpp', '-B', '/kaggle/working/llama.cpp/build', '-DGGML_CUDA=ON', '-DCMAKE_BUILD_TYPE=Release']' returned non-zero exit status 1.

In [5]:
import os, shutil, subprocess, sys, textwrap

print("=" * 70)
print("RAIOS LLAMA.CPP PREBUILT RECOVERY")
print("=" * 70)

# Clean only failed local build artifacts
repo = "/kaggle/working/llama.cpp"
build = f"{repo}/build"

if os.path.exists(build):
    shutil.rmtree(build)
    print("Removed failed build directory.")

print("\n[1] PYTHON")
print(sys.version)

print("\n[2] GPU")
subprocess.run(["nvidia-smi", "-L"], check=True)

print("\n[3] TRY PREBUILT LLAMA-CPP-PYTHON CUDA WHEEL")
cmd = [
    sys.executable, "-m", "pip", "install",
    "--upgrade",
    "--extra-index-url",
    "https://abetlen.github.io/llama-cpp-python/whl/cu124",
    "llama-cpp-python[server]"
]

r = subprocess.run(cmd, text=True)

print("\nInstall exit:", r.returncode)

if r.returncode != 0:
    print("PREBUILT_INSTALL_FAIL")
else:
    print("PREBUILT_INSTALL_PASS")

print("\n[4] IMPORT CHECK")

try:
    import llama_cpp
    print("llama_cpp version:", llama_cpp.__version__)
    print("IMPORT_PASS")
except Exception as e:
    print("IMPORT_FAIL:", repr(e))

print("\n" + "=" * 70)
print("RECOVERY CHECK COMPLETE")
print("=" * 70)

RAIOS LLAMA.CPP PREBUILT RECOVERY
Removed failed build directory.

[1] PYTHON
3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

[2] GPU
GPU 0: Tesla T4 (UUID: GPU-21e00db4-c36a-d7a6-577c-735cbb9fd8c7)
GPU 1: Tesla T4 (UUID: GPU-934811de-5d4a-b9c2-38a4-9a70814934b4)

[3] TRY PREBUILT LLAMA-CPP-PYTHON CUDA WHEEL
Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 GB 528.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.5 MB/s eta 0:00:00

Install exit: 0
PREBUILT_INSTALL_PASS

[4] IMPORT CHECK
llama_cpp version: 0.3.34
IMPORT_PASS

RECOVERY CHECK COMPLETE


In [7]:
import subprocess
import llama_cpp

print("=" * 70)
print("RAIOS LLAMA_CPP GPU BACKEND CERTIFICATION")
print("=" * 70)

print("\n[1] VERSION")
print("llama_cpp:", llama_cpp.__version__)

print("\n[2] SYSTEM INFO")
try:
    print(llama_cpp.llama_print_system_info().decode())
except Exception as e:
    print("SYSTEM_INFO_ERROR:", repr(e))

print("\n[3] NVIDIA")
subprocess.run(["nvidia-smi", "-L"], check=True)

print("\n[4] BACKEND VERDICT")

info = ""
try:
    info = llama_cpp.llama_print_system_info().decode().lower()
except:
    pass

cuda_signals = [
    "cuda",
    "cublas",
    "ggml_cuda",
]

gpu_backend = any(x in info for x in cuda_signals)

print("GPU_BACKEND_VISIBLE:", gpu_backend)

if gpu_backend:
    print("LLAMA_CPP_GPU_BACKEND_PASS")
else:
    print("LLAMA_CPP_GPU_BACKEND_NOT_CONFIRMED")

print("=" * 70)

RAIOS LLAMA_CPP GPU BACKEND CERTIFICATION

[1] VERSION
llama_cpp: 0.3.34

[2] SYSTEM INFO
CUDA : ARCHS = 600,610,700,750,800,860,890,900 | FORCE_MMQ = 1 | USE_GRAPHS = 1 | CPU : SSE3 = 1 | SSSE3 = 1 | AVX = 1 | AVX2 = 1 | F16C = 1 | FMA = 1 | BMI2 = 1 | LLAMAFILE = 1 | REPACK = 1 | 

[3] NVIDIA
GPU 0: Tesla T4 (UUID: GPU-21e00db4-c36a-d7a6-577c-735cbb9fd8c7)
GPU 1: Tesla T4 (UUID: GPU-934811de-5d4a-b9c2-38a4-9a70814934b4)

[4] BACKEND VERDICT
GPU_BACKEND_VISIBLE: True
LLAMA_CPP_GPU_BACKEND_PASS


In [8]:
import os
import subprocess
from huggingface_hub import hf_hub_download

print("=" * 70)
print("RAIOS QWEN3.6-35B-A3B MODEL ACQUISITION")
print("=" * 70)

repo = "ggml-org/Qwen3.6-35B-A3B-GGUF"

# Keep the huge model away from /kaggle/working
cache_dir = "/root/.cache/huggingface"

print("\n[1] CACHE")
print("Cache:", cache_dir)

print("\n[2] DISCOVER Q4_K_M FILE")

from huggingface_hub import list_repo_files

files = list_repo_files(repo)

matches = [
    f for f in files
    if "Q4_K_M" in f and f.endswith(".gguf")
]

print("Matches:")
for f in matches:
    print(" ", f)

if not matches:
    raise RuntimeError("STOP: No Q4_K_M GGUF found.")

# Prefer main model, avoid MTP / DFlash helper models if present
candidates = [
    f for f in matches
    if "mtp" not in f.lower()
    and "dflash" not in f.lower()
]

if not candidates:
    raise RuntimeError(
        "STOP: Only auxiliary Q4_K_M files found; do not download blindly."
    )

model_file = candidates[0]

print("\nSelected:")
print(model_file)

print("\n[3] DOWNLOAD")
print("This is the large step (~20.4 GB).")

path = hf_hub_download(
    repo_id=repo,
    filename=model_file,
    cache_dir=cache_dir,
)

print("\nMODEL_PATH:")
print(path)

print("\n[4] VERIFY SIZE")

size_gb = os.path.getsize(path) / 1024**3

print(f"Size: {size_gb:.2f} GB")

if size_gb < 15:
    raise RuntimeError(
        "STOP: Downloaded file is unexpectedly small."
    )

print("\n[5] GPU STATE")
subprocess.run(["nvidia-smi"], check=True)

print("\n" + "=" * 70)
print("MODEL_ACQUISITION_PASS")
print("=" * 70)

RAIOS QWEN3.6-35B-A3B MODEL ACQUISITION

[1] CACHE
Cache: /root/.cache/huggingface

[2] DISCOVER Q4_K_M FILE


Matches:
  Qwen3.6-35B-A3B-Q4_K_M.gguf

Selected:
Qwen3.6-35B-A3B-Q4_K_M.gguf

[3] DOWNLOAD
This is the large step (~20.4 GB).


Qwen3.6-35B-A3B-Q4_K_M.gguf:   0%|          | 0.00/20.4G [00:00<?, ?B/s]


MODEL_PATH:
/root/.cache/huggingface/models--ggml-org--Qwen3.6-35B-A3B-GGUF/snapshots/baec3ebee244827cda0f4557eafa8b28f7545fa6/Qwen3.6-35B-A3B-Q4_K_M.gguf

[4] VERIFY SIZE
Size: 19.02 GB

[5] GPU STATE
Sun Aug 16 23:14:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8            

In [9]:
import os, subprocess, inspect, time
from llama_cpp import Llama, LLAMA_SPLIT_MODE_LAYER

print("=" * 72)
print("RAIOS EXECUTION PROFILE 01 — MODEL LOAD + FIRST INFERENCE")
print("=" * 72)

MODEL = (
    "/root/.cache/huggingface/"
    "models--ggml-org--Qwen3.6-35B-A3B-GGUF/"
    "snapshots/baec3ebee244827cda0f4557eafa8b28f7545fa6/"
    "Qwen3.6-35B-A3B-Q4_K_M.gguf"
)

print("\n[1] MODEL")
print("Exists:", os.path.exists(MODEL))
print("Size GB:", round(os.path.getsize(MODEL) / 1024**3, 2))

print("\n[2] API CHECK")
sig = inspect.signature(Llama.__init__)
print("tensor_split supported:", "tensor_split" in sig.parameters)
print("split_mode supported:", "split_mode" in sig.parameters)

if "tensor_split" not in sig.parameters:
    raise RuntimeError("STOP: tensor_split unavailable")

print("\n[3] GPU BEFORE LOAD")
subprocess.run(["nvidia-smi"], check=True)

print("\n[4] LOADING MODEL...")
start = time.time()

llm = Llama(
    model_path=MODEL,

    # Put model layers on GPUs
    n_gpu_layers=-1,

    # Split across both T4s
    split_mode=LLAMA_SPLIT_MODE_LAYER,
    tensor_split=[0.5, 0.5],

    # Keep first certification deliberately small
    n_ctx=4096,

    # Kaggle CPU
    n_threads=4,
    n_threads_batch=4,

    # Reduce initial memory pressure
    n_batch=256,

    verbose=True,
)

print(f"\nMODEL_LOAD_SECONDS: {time.time() - start:.2f}")

print("\n[5] GPU AFTER LOAD")
subprocess.run(["nvidia-smi"], check=True)

print("\n[6] FIRST INFERENCE")

start = time.time()

result = llm.create_chat_completion(
    messages=[
        {
            "role": "system",
            "content":
                "You are a software engineering agent. "
                "Answer accurately and concisely."
        },
        {
            "role": "user",
            "content":
                "A repository contains a function that should return "
                "the sum of two integers. Write only the Python function."
        }
    ],
    temperature=0,
    max_tokens=128,
)

elapsed = time.time() - start

print("\n--- MODEL RESPONSE ---")
print(result["choices"][0]["message"]["content"])

usage = result.get("usage", {})

print("\n[7] METRICS")
print("Inference seconds:", round(elapsed, 2))
print("Usage:", usage)

print("\n[8] GPU AFTER INFERENCE")
subprocess.run(["nvidia-smi"], check=True)

print("\n" + "=" * 72)
print("EXECUTION_PROFILE_01_BASIC_INFERENCE_PASS")
print("=" * 72)

RAIOS EXECUTION PROFILE 01 — MODEL LOAD + FIRST INFERENCE

[1] MODEL
Exists: True
Size GB: 19.02

[2] API CHECK
tensor_split supported: True
split_mode supported: True

[3] GPU BEFORE LOAD
Sun Aug 16 23:25:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             10W /   70W |

llama_model_loader: loaded meta data with 43 key-value pairs and 733 tensors from /root/.cache/huggingface/models--ggml-org--Qwen3.6-35B-A3B-GGUF/snapshots/baec3ebee244827cda0f4557eafa8b28f7545fa6/Qwen3.6-35B-A3B-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen35moe
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_k i32              = 20
llama_model_loader: - kv   3:                     general.sampling.top_p f32              = 0.950000
llama_model_loader: - kv   4:                      general.sampling.temp f32              = 1.000000
llama_model_loader: - kv   5:                               general.name str              = Qwen3.6-35B-A3B
llama_model_loader: - kv   6:                


[4] LOADING MODEL...


llama_model_loader: - kv  35:                      tokenizer.ggml.merges arr[str,247587]  = ["Ġ Ġ", "ĠĠ ĠĠ", "i n", "Ġ t",...
llama_model_loader: - kv  36:                tokenizer.ggml.eos_token_id u32              = 248046
llama_model_loader: - kv  37:            tokenizer.ggml.padding_token_id u32              = 248044
llama_model_loader: - kv  38:                tokenizer.ggml.bos_token_id u32              = 248044
llama_model_loader: - kv  39:               tokenizer.ggml.add_bos_token bool             = false
llama_model_loader: - kv  40:                    tokenizer.chat_template str              = {%- set image_count = namespace(value...
llama_model_loader: - kv  41:               general.quantization_version u32              = 2
llama_model_loader: - kv  42:                          general.file_type u32              = 15
llama_model_loader: - type  f32:  301 tensors
llama_model_loader: - type q8_0:  310 tensors
llama_model_loader: - type q4_K:  121 tensors
llama_model_loader:


MODEL_LOAD_SECONDS: 65.24

[5] GPU AFTER LOAD
Sun Aug 16 23:26:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P0             29W /   70W |   10159MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+

llama_perf_context_print:        load time =     859.49 ms
llama_perf_context_print: prompt eval time =     857.68 ms /    49 tokens (   17.50 ms per token,    57.13 tokens per second)
llama_perf_context_print:        eval time =    2263.53 ms /   127 runs   (   17.82 ms per token,    56.11 tokens per second)
llama_perf_context_print:       total time =    3221.04 ms /   176 tokens
llama_perf_context_print:    graphs reused =        126



--- MODEL RESPONSE ---
Here's a thinking process:

1.  **Analyze User Input:**
   - **Task:** Write a Python function that returns the sum of two integers.
   - **Constraint:** "Write only the Python function."
   - **Context:** A repository contains a function (implies standard coding task).

2.  **Identify Key Requirements:**
   - Language: Python
   - Functionality: Sum of two integers
   - Output: Only the function code (no explanations, no extra text)

3.  **Draft the Function:**
   ```python
   def add(a, b

[7] METRICS
Inference seconds: 3.23
Usage: {'prompt_tokens': 49, 'completion_tokens': 128, 'total_tokens': 177}

[8] GPU AFTER INFERENCE
Sun Aug 16 23:26:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persis

In [10]:
import os
import time
import json
import traceback
import subprocess
from llama_cpp import Llama, LLAMA_SPLIT_MODE_LAYER

STEP = "EXECUTION_PROFILE_01_MODEL_LOAD"
OUT = "/kaggle/working/raios-results"
os.makedirs(OUT, exist_ok=True)

MODEL = (
    "/root/.cache/huggingface/"
    "models--ggml-org--Qwen3.6-35B-A3B-GGUF/"
    "snapshots/baec3ebee244827cda0f4557eafa8b28f7545fa6/"
    "Qwen3.6-35B-A3B-Q4_K_M.gguf"
)

def gpu_state():
    try:
        raw = subprocess.check_output(
            [
                "nvidia-smi",
                "--query-gpu=index,name,memory.used,memory.total,utilization.gpu",
                "--format=csv,noheader,nounits",
            ],
            text=True,
        ).strip().splitlines()

        return [
            {
                "index": int(parts[0].strip()),
                "name": parts[1].strip(),
                "used_mb": int(parts[2].strip()),
                "total_mb": int(parts[3].strip()),
                "util_pct": int(parts[4].strip()),
            }
            for line in raw
            for parts in [line.split(",")]
        ]
    except Exception as e:
        return [{"error": str(e)}]

report = {
    "step": STEP,
    "status": "FAIL",
    "model": "Qwen3.6-35B-A3B-Q4_K_M",
    "model_size_gb": None,
    "context": 4096,
    "gpu_before": gpu_state(),
    "gpu_after_load": None,
    "gpu_after_inference": None,
    "load_seconds": None,
    "inference_seconds": None,
    "response": None,
    "usage": None,
    "error": None,
}

try:
    report["model_size_gb"] = round(
        os.path.getsize(MODEL) / 1024**3, 2
    )

    start = time.time()

    llm = Llama(
        model_path=MODEL,
        n_gpu_layers=-1,
        split_mode=LLAMA_SPLIT_MODE_LAYER,
        tensor_split=[0.5, 0.5],
        n_ctx=4096,
        n_threads=4,
        n_threads_batch=4,
        n_batch=256,

        # IMPORTANT:
        # prevents thousands of llama.cpp diagnostic lines
        verbose=False,
    )

    report["load_seconds"] = round(time.time() - start, 2)
    report["gpu_after_load"] = gpu_state()

    start = time.time()

    result = llm.create_chat_completion(
        messages=[
            {
                "role": "system",
                "content":
                    "You are a software engineering agent. "
                    "Answer accurately and concisely."
            },
            {
                "role": "user",
                "content":
                    "Write only a Python function named add that "
                    "returns the sum of two integers."
            },
        ],
        temperature=0,
        max_tokens=128,
    )

    report["inference_seconds"] = round(time.time() - start, 2)
    report["gpu_after_inference"] = gpu_state()
    report["response"] = result["choices"][0]["message"]["content"]
    report["usage"] = result.get("usage", {})
    report["status"] = "PASS"

except Exception as e:
    report["error"] = {
        "type": type(e).__name__,
        "message": str(e),
        "traceback_tail": traceback.format_exc().splitlines()[-12:],
    }

# Full structured evidence
json_path = f"{OUT}/{STEP}.json"

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

# =========================================================
# ONLY THIS SMALL BLOCK NEEDS TO BE SENT TO CHATGPT
# =========================================================

print("\n" + "=" * 64)
print("RAIOS SHARE CAPSULE")
print("=" * 64)

print("STEP:", report["step"])
print("STATUS:", report["status"])
print("MODEL:", report["model"])
print("MODEL_SIZE_GB:", report["model_size_gb"])
print("CONTEXT:", report["context"])
print("LOAD_SECONDS:", report["load_seconds"])
print("INFERENCE_SECONDS:", report["inference_seconds"])

if report["gpu_after_load"]:
    for gpu in report["gpu_after_load"]:
        if "error" not in gpu:
            print(
                f"GPU{gpu['index']}: "
                f"{gpu['used_mb']}/{gpu['total_mb']} MB"
            )

if report["status"] == "PASS":
    print("RESPONSE:", report["response"])
    print("USAGE:", report["usage"])
else:
    print("ERROR_TYPE:", report["error"]["type"])
    print("ERROR:", report["error"]["message"])
    print("TRACEBACK_TAIL:")
    for line in report["error"]["traceback_tail"]:
        print(line)

print("FULL_RESULT:", json_path)

print("=" * 64)
print("END SHARE CAPSULE")
print("=" * 64)


RAIOS SHARE CAPSULE
STEP: EXECUTION_PROFILE_01_MODEL_LOAD
STATUS: FAIL
MODEL: Qwen3.6-35B-A3B-Q4_K_M
MODEL_SIZE_GB: 19.02
CONTEXT: 4096
LOAD_SECONDS: None
INFERENCE_SECONDS: None
ERROR_TYPE: ValueError
ERROR: Failed to load model from file: /root/.cache/huggingface/models--ggml-org--Qwen3.6-35B-A3B-GGUF/snapshots/baec3ebee244827cda0f4557eafa8b28f7545fa6/Qwen3.6-35B-A3B-Q4_K_M.gguf
TRACEBACK_TAIL:
Traceback (most recent call last):
  File "/tmp/ipykernel_58/3792499668.py", line 67, in <cell line: 0>
    llm = Llama(
          ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/llama_cpp/llama.py", line 381, in __init__
    internals.LlamaModel(
  File "/usr/local/lib/python3.12/dist-packages/llama_cpp/_internals.py", line 62, in __init__
    raise ValueError(f"Failed to load model from file: {path_model}")
ValueError: Failed to load model from file: /root/.cache/huggingface/models--ggml-org--Qwen3.6-35B-A3B-GGUF/snapshots/baec3ebee244827cda0f4557eafa8b28f7545fa6/Qwen3.6-35B-A3B-Q4_K

In [1]:
from pathlib import Path

print("=" * 80)
print("RAIOS INPUT DISCOVERY — READ ONLY")
print("=" * 80)

root = Path("/kaggle/input")

if not root.exists():
    raise RuntimeError("/kaggle/input not found")

for dataset in sorted(root.iterdir()):
    print(f"\nDATASET: {dataset}")
    print("-" * 80)

    files = [p for p in dataset.rglob("*") if p.is_file()]

    print("FILES:", len(files))

    for p in files[:200]:
        try:
            size = p.stat().st_size
        except Exception:
            size = -1

        print(f"  {p.relative_to(dataset)} | {size} bytes")

    if len(files) > 200:
        print(f"  ... +{len(files)-200} more files")

print()
print("=" * 80)
print("DISCOVERY COMPLETE")
print("READ ONLY")
print("NO MODEL LOAD")
print("NO TRAINING")
print("=" * 80)

RAIOS INPUT DISCOVERY — READ ONLY

DATASET: /kaggle/input/datasets
--------------------------------------------------------------------------------
FILES: 129
  greenylife/greeny-life/MANIFEST.json | 674 bytes
  greenylife/greeny-life/greeny-capabilities.json | 44545 bytes
  greenylife/greeny-life/mission.json | 12361 bytes
  greenylife/greeny-life/MISSION.md | 2436 bytes
  greenylife/greeny-life/greeny-capabilities.md | 7306 bytes
  greenylife/raios-cognitive-state/CURRENT.json | 616 bytes
  greenylife/raios-cognitive-state/dataset-metadata.json | 217 bytes
  greenylife/raios-cognitive-state/RAIOS-STATE-LATEST/RAIOS/RECOVERY-CONTRACT.json | 541 bytes
  greenylife/raios-cognitive-state/RAIOS-STATE-LATEST/RAIOS/DURABLE-MANIFEST.json | 28836 bytes
  greenylife/raios-cognitive-state/RAIOS-STATE-LATEST/RAIOS/SOURCE-OF-TRUTH.json | 171 bytes
  greenylife/raios-cognitive-state/RAIOS-STATE-LATEST/RAIOS/raios-cognitive-factory/runtime/durability_transaction.py | 1988 bytes
  greenylife/raios-c

In [2]:
from __future__ import annotations

from pathlib import Path
from collections import Counter, defaultdict
import json
import re
import math
import hashlib
from datetime import datetime, timezone


print("=" * 88)
print("RAIOS V8 STATE AUDIT + GREENY CAPABILITY ASSIMILATION")
print("CPU ONLY | READ-ONLY INPUTS | NO MODEL | NO TRAINING")
print("=" * 88)


# ============================================================================
# 0. FIXED ROOTS DISCOVERED FROM THE ACTUAL KAGGLE ENVIRONMENT
# ============================================================================

DATASETS_ROOT = Path("/kaggle/input/datasets/greenylife")

RAIOS_DATASET = DATASETS_ROOT / "raios-cognitive-state"
GREENY_DATASET = DATASETS_ROOT / "greeny-life"

RAIOS_ROOT = (
    RAIOS_DATASET
    / "RAIOS-STATE-LATEST"
    / "RAIOS"
)

FACTORY = RAIOS_ROOT / "raios-cognitive-factory"
STATE = FACTORY / "state"

WORK = Path("/kaggle/working")

if not RAIOS_DATASET.exists():
    raise RuntimeError(f"RAIOS dataset missing: {RAIOS_DATASET}")

if not GREENY_DATASET.exists():
    raise RuntimeError(f"Greeny mission dataset missing: {GREENY_DATASET}")

if not STATE.exists():
    raise RuntimeError(f"RAIOS state root missing: {STATE}")

print("\n[PASS] ROOTS")
print("RAIOS_DATASET :", RAIOS_DATASET)
print("GREENY_DATASET:", GREENY_DATASET)
print("STATE         :", STATE)


# ============================================================================
# 1. HELPERS
# ============================================================================

def read_json(path: Path, default=None):
    try:
        return json.loads(path.read_text(encoding="utf-8-sig"))
    except Exception as exc:
        if default is not None:
            return default
        raise RuntimeError(f"Failed JSON read: {path}\n{exc}")


def safe_text(path: Path, max_bytes=2_000_000):
    try:
        if path.stat().st_size > max_bytes:
            return ""
        return path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        return ""


def files_in(path: Path, suffix=None):
    if not path.exists():
        return []
    result = [p for p in path.rglob("*") if p.is_file()]
    if suffix:
        result = [p for p in result if p.suffix.lower() == suffix]
    return sorted(result)


def normalize_tokens(text: str):
    stop = {
        "json", "state", "active", "current", "raios",
        "true", "false", "null", "none",
        "file", "data", "system", "version",
        "export", "import", "function", "const",
    }

    return {
        token
        for token in re.findall(
            r"[a-z][a-z0-9_-]{2,}",
            text.lower()
        )
        if token not in stop and len(token) >= 3
    }


def flatten_keys(value, prefix=""):
    out = []

    if isinstance(value, dict):
        for key, child in value.items():
            path = f"{prefix}.{key}" if prefix else str(key)
            out.append(path)
            out.extend(flatten_keys(child, path))

    elif isinstance(value, list):
        for index, child in enumerate(value[:100]):
            out.extend(flatten_keys(child, prefix))

    return out


def detect_confidence_values(value, path="$"):
    found = []

    if isinstance(value, dict):
        for key, child in value.items():
            child_path = f"{path}.{key}"

            if "confidence" in str(key).lower():
                found.append({
                    "path": child_path,
                    "value": child
                })

            found.extend(
                detect_confidence_values(child, child_path)
            )

    elif isinstance(value, list):
        for index, child in enumerate(value):
            found.extend(
                detect_confidence_values(
                    child,
                    f"{path}[{index}]"
                )
            )

    return found


def confidence_valid(value):
    if value is None:
        return True

    if isinstance(value, bool):
        return False

    if isinstance(value, (int, float)):
        return (
            math.isfinite(float(value))
            and 0.0 <= float(value) <= 1.0
        )

    return False


# ============================================================================
# 2. LOAD AUTHORITY / CONTINUITY FILES
# ============================================================================

authority_files = {
    "CURRENT":
        RAIOS_DATASET / "CURRENT.json",

    "SOURCE_OF_TRUTH":
        RAIOS_ROOT / "SOURCE-OF-TRUTH.json",

    "RECOVERY_CONTRACT":
        RAIOS_ROOT / "RECOVERY-CONTRACT.json",

    "DURABLE_MANIFEST":
        RAIOS_ROOT / "DURABLE-MANIFEST.json",

    "LATEST_CHECKPOINT":
        STATE / "checkpoints" / "LATEST.json",

    "DURABILITY_LATEST":
        STATE / "checkpoints" / "DURABILITY-LATEST.json",

    "DURABILITY_POLICY":
        STATE / "durability-policy" / "ACTIVE.json",

    "ROUTING_POLICY":
        STATE / "routing-policies" / "ACTIVE.json",

    "LEARNING_LEDGER":
        STATE / "manifests" / "learning-ledger.json",

    "COGNITIVE_CAPITAL":
        STATE / "manifests" / "cognitive-capital.json",

    "COGNITIVE_DOCTRINE":
        STATE / "doctrine" / "RAIOS-COGNITIVE-DOCTRINE.json",

    "SEMANTIC_CONTRACT":
        STATE / "doctrine" / "SEMANTIC-STATE-CONTRACT.json",
}

authority = {}

print("\n" + "=" * 88)
print("AUTHORITY / CONTINUITY")
print("=" * 88)

for name, path in authority_files.items():

    if path.exists():
        data = read_json(path, {})
        authority[name] = data

        print(f"\n{name}")
        print("-" * 40)

        preview = json.dumps(
            data,
            ensure_ascii=False,
            indent=2
        )

        print(preview[:2500])

    else:
        authority[name] = None
        print(f"\n{name}: [MISSING] {path}")


# ============================================================================
# 3. INVENTORY COGNITIVE STATE
# ============================================================================

families = [
    "skills",
    "experiences",
    "failures",
    "benchmarks",
    "semantic-cache",
    "semantic-corpus",
    "training-candidates",
    "decisions",
    "evidence",
    "performance",
    "model-profiles",
    "environment-profiles",
    "fingerprints",
    "semantic-atoms",
    "semantic-taxonomy",
    "routing-policies",
    "durability-transactions",
]

inventory = {}

print("\n" + "=" * 88)
print("COGNITIVE STATE INVENTORY")
print("=" * 88)

for family in families:

    path = STATE / family
    items = files_in(path)

    inventory[family] = {
        "count": len(items),
        "files": [
            str(p.relative_to(STATE))
            for p in items
        ]
    }

    print(
        f"{family:26} "
        f"{len(items):>5}"
    )

journal = STATE / "journal" / "events.jsonl"

journal_events = 0

if journal.exists():
    with journal.open(
        "r",
        encoding="utf-8",
        errors="ignore"
    ) as handle:

        for line in handle:
            if line.strip():
                journal_events += 1

print(
    f"{'journal-events':26} "
    f"{journal_events:>5}"
)


# ============================================================================
# 4. CONFIDENCE-SCALE AUDIT
# ============================================================================

print("\n" + "=" * 88)
print("CONFIDENCE NORMALIZATION AUDIT")
print("REQUIRED RANGE = [0,1]")
print("=" * 88)

confidence_findings = []
confidence_anomalies = []

json_files = [
    p
    for p in STATE.rglob("*.json")
    if p.is_file()
]

for path in json_files:

    data = read_json(path, None)

    if data is None:
        continue

    for item in detect_confidence_values(data):

        finding = {
            "file":
                str(path.relative_to(STATE)),

            "json_path":
                item["path"],

            "value":
                item["value"],

            "normalized_valid":
                confidence_valid(item["value"]),
        }

        confidence_findings.append(finding)

        if not finding["normalized_valid"]:
            confidence_anomalies.append(finding)


print(
    "Confidence fields:",
    len(confidence_findings)
)

print(
    "Out-of-range / invalid:",
    len(confidence_anomalies)
)

for anomaly in confidence_anomalies[:100]:
    print(
        "ANOMALY |",
        anomaly["file"],
        "|",
        anomaly["json_path"],
        "=",
        anomaly["value"]
    )


# ============================================================================
# 5. EXPLICITLY AUDIT THE PREVIOUSLY PROBLEMATIC SKILL
# ============================================================================

KNOWN_INVALIDATED_SKILL = (
    "683ca22e97a2cb919ab078321076bd06"
    "b146b1bf69e937a2062d68755e993b65"
)

known_skill_path = (
    STATE
    / "skills"
    / f"{KNOWN_INVALIDATED_SKILL}.json"
)

known_skill = None

print("\n" + "=" * 88)
print("KNOWN SKILL SAFETY CHECK")
print("=" * 88)

if known_skill_path.exists():

    known_skill = read_json(
        known_skill_path,
        {}
    )

    print(
        json.dumps(
            known_skill,
            indent=2,
            ensure_ascii=False
        )
    )

else:
    print(
        "Known skill file not present."
    )


# ============================================================================
# 6. READ ALL SKILLS / EXPERIENCES / TRAINING CANDIDATES
# ============================================================================

def load_family(name):
    root = STATE / name

    result = []

    for path in files_in(root, ".json"):

        data = read_json(path, {})

        result.append({
            "file": str(
                path.relative_to(STATE)
            ),
            "data": data,
            "keys": flatten_keys(data),
            "tokens": normalize_tokens(
                path.name
                + " "
                + json.dumps(
                    data,
                    ensure_ascii=False
                )
            )
        })

    return result


skills = load_family("skills")
experiences = load_family("experiences")
training_candidates = load_family(
    "training-candidates"
)
failures = load_family("failures")
decisions = load_family("decisions")
benchmarks = load_family("benchmarks")
evidence = load_family("evidence")


# ============================================================================
# 7. GREENY MISSION
# ============================================================================

greeny_cap_path = (
    GREENY_DATASET
    / "greeny-capabilities.json"
)

mission_path = (
    GREENY_DATASET
    / "mission.json"
)

greeny = read_json(greeny_cap_path)
mission = read_json(mission_path)

greeny_capabilities = greeny.get(
    "capabilities",
    []
)

print("\n" + "=" * 88)
print("GREENY MISSION")
print("=" * 88)

print(
    "Greeny capabilities:",
    len(greeny_capabilities)
)

print(
    "Greeny source branch:",
    greeny["repository"]["branch"]
)

print(
    "Greeny source HEAD:",
    greeny["repository"]["head"]
)


# ============================================================================
# 8. BUILD A RAIOS CAPABILITY CORPUS FROM ACTUAL STATE
# ============================================================================

raios_objects = []

family_objects = {
    "SKILL": skills,
    "EXPERIENCE": experiences,
    "FAILURE": failures,
    "TRAINING_CANDIDATE":
        training_candidates,
    "DECISION": decisions,
    "BENCHMARK": benchmarks,
    "EVIDENCE": evidence,
}

for family, objects in family_objects.items():

    for obj in objects:

        raios_objects.append({
            "family": family,
            "file": obj["file"],
            "tokens": obj["tokens"],
            "data": obj["data"],
        })


# Also add architecture/state files as reusable foundations.

for name, path in authority_files.items():

    if not path.exists():
        continue

    data = read_json(path, {})

    raios_objects.append({
        "family": "CONTROL_STATE",
        "file": str(
            path.relative_to(RAIOS_ROOT)
            if RAIOS_ROOT in path.parents
            else path.name
        ),
        "tokens": normalize_tokens(
            name
            + " "
            + json.dumps(
                data,
                ensure_ascii=False
            )
        ),
        "data": data,
    })


# ============================================================================
# 9. GREENY -> RAIOS REUSE / EXTEND / GAP MATCHING
#
# IMPORTANT:
# This is candidate generation, NOT semantic proof.
# No candidate becomes a duplicate automatically.
# ============================================================================

comparison = []

for cap in greeny_capabilities:

    greeny_path = cap.get(
        "path",
        ""
    )

    category = cap.get(
        "category",
        "OTHER"
    )

    symbols = cap.get(
        "symbols",
        []
    )

    query = normalize_tokens(
        greeny_path
        + " "
        + category
        + " "
        + " ".join(symbols)
    )

    matches = []

    for obj in raios_objects:

        overlap = (
            query
            & obj["tokens"]
        )

        if not overlap:
            continue

        query_coverage = (
            len(overlap)
            / max(1, len(query))
        )

        symbol_hits = sum(
            1
            for symbol in symbols
            if symbol.lower()
            in obj["tokens"]
        )

        score = (
            query_coverage * 0.85
            + min(symbol_hits, 5) * 0.03
        )

        if score < 0.08:
            continue

        matches.append({
            "family":
                obj["family"],

            "file":
                obj["file"],

            "score":
                round(score, 4),

            "query_coverage":
                round(query_coverage, 4),

            "symbol_hits":
                symbol_hits,

            "overlap":
                sorted(overlap)[:50],
        })

    matches.sort(
        key=lambda x:
            x["score"],
        reverse=True
    )

    best = matches[:8]

    best_score = (
        best[0]["score"]
        if best
        else 0.0
    )

    # Candidate class only.
    # Human / later semantic validation still required.

    if best_score >= 0.70:
        candidate_action = (
            "STRONG_REUSE_CANDIDATE"
        )

    elif best_score >= 0.45:
        candidate_action = (
            "REUSE_OR_COMPOSE_CANDIDATE"
        )

    elif best_score >= 0.25:
        candidate_action = (
            "EXTEND_COMPARE_CANDIDATE"
        )

    elif best_score >= 0.10:
        candidate_action = (
            "WEAK_OVERLAP_REVIEW"
        )

    else:
        candidate_action = (
            "NO_EQUIVALENT_FOUND_YET"
        )

    comparison.append({
        "greeny_path":
            greeny_path,

        "category":
            category,

        "symbols":
            symbols,

        "candidate_action":
            candidate_action,

        "best_score":
            best_score,

        "raios_candidates":
            best,
    })


action_counts = Counter(
    item["candidate_action"]
    for item in comparison
)


# ============================================================================
# 10. DETECT RAIOS FOUNDATIONS THAT MAP TO V9
# ============================================================================

v9_foundations = {
    "FAST_LEARNING":
        bool(experiences),

    "FAILURE_MEMORY":
        bool(failures),

    "SKILL_MEMORY":
        bool(skills),

    "EVIDENCE_MEMORY":
        bool(evidence),

    "DECISION_MEMORY":
        bool(decisions),

    "PERFORMANCE_MEMORY":
        inventory[
            "performance"
        ]["count"] > 0,

    "MODEL_MEMORY":
        inventory[
            "model-profiles"
        ]["count"] > 0,

    "ENVIRONMENT_MEMORY":
        inventory[
            "environment-profiles"
        ]["count"] > 0,

    "SEMANTIC_CACHE":
        inventory[
            "semantic-cache"
        ]["count"] > 0,

    "SEMANTIC_CORPUS":
        inventory[
            "semantic-corpus"
        ]["count"] > 0,

    "TRAINING_CANDIDATES":
        bool(training_candidates),

    "COGNITIVE_JOURNAL":
        journal_events > 0,

    "CHECKPOINTS":
        inventory[
            "durability-transactions"
        ]["count"] > 0
        or (
            STATE / "checkpoints"
        ).exists(),

    "DURABILITY_POLICY":
        authority[
            "DURABILITY_POLICY"
        ] is not None,

    "DURABILITY_TRANSACTION_ENGINE":
        (
            FACTORY
            / "runtime"
            / "durability_transaction.py"
        ).exists(),

    "SEMANTIC_TAXONOMY":
        inventory[
            "semantic-taxonomy"
        ]["count"] > 0,

    "ROUTING_POLICY":
        authority[
            "ROUTING_POLICY"
        ] is not None,
}


# ============================================================================
# 11. REPORT OBJECT
# ============================================================================

report = {
    "schema":
        "raios.v8.greeny-assimilation-audit.v1",

    "generated_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "mode": {
        "cpu_only": True,
        "model_loaded": False,
        "training_executed": False,
        "input_mutated": False,
    },

    "roots": {
        "raios_dataset":
            str(RAIOS_DATASET),

        "raios_root":
            str(RAIOS_ROOT),

        "factory":
            str(FACTORY),

        "state":
            str(STATE),

        "greeny_dataset":
            str(GREENY_DATASET),
    },

    "authority":
        authority,

    "inventory":
        inventory,

    "journal_events":
        journal_events,

    "confidence_audit": {
        "fields_checked":
            len(confidence_findings),

        "anomalies":
            confidence_anomalies,
    },

    "known_invalidated_skill": {
        "id":
            KNOWN_INVALIDATED_SKILL,

        "present":
            known_skill_path.exists(),

        "record":
            known_skill,
    },

    "v9_foundations":
        v9_foundations,

    "greeny": {
        "branch":
            greeny["repository"]["branch"],

        "head":
            greeny["repository"]["head"],

        "capability_count":
            len(greeny_capabilities),
    },

    "action_counts":
        dict(action_counts),

    "comparison":
        comparison,
}


# ============================================================================
# 12. WRITE ONLY TO /kaggle/working
# ============================================================================

json_out = (
    WORK
    / "RAIOS-V8-GREENY-ASSIMILATION-AUDIT.json"
)

json_out.write_text(
    json.dumps(
        report,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


txt_out = (
    WORK
    / "RAIOS-V8-GREENY-ASSIMILATION-AUDIT.txt"
)

lines = []

lines.append("=" * 88)
lines.append(
    "RAIOS V8 + GREENY ASSIMILATION AUDIT"
)
lines.append("=" * 88)

lines.append("")
lines.append(
    f"GREENY_CAPABILITIES={len(greeny_capabilities)}"
)

lines.append(
    f"JOURNAL_EVENTS={journal_events}"
)

lines.append(
    f"CONFIDENCE_FIELDS={len(confidence_findings)}"
)

lines.append(
    f"CONFIDENCE_ANOMALIES={len(confidence_anomalies)}"
)

lines.append(
    f"KNOWN_INVALIDATED_SKILL_PRESENT={known_skill_path.exists()}"
)

lines.append("")
lines.append(
    "----- STATE COUNTS -----"
)

for family, info in inventory.items():
    lines.append(
        f"{family}={info['count']}"
    )

lines.append("")
lines.append(
    "----- V9 FOUNDATIONS -----"
)

for name, present in v9_foundations.items():
    lines.append(
        f"{name}={present}"
    )

lines.append("")
lines.append(
    "----- GREENY MATCH COUNTS -----"
)

for action, count in sorted(
    action_counts.items()
):
    lines.append(
        f"{action}={count}"
    )

lines.append("")
lines.append(
    "----- CONFIDENCE ANOMALIES -----"
)

if confidence_anomalies:

    for item in confidence_anomalies[:200]:

        lines.append(
            f"{item['file']} | "
            f"{item['json_path']} | "
            f"{item['value']}"
        )

else:
    lines.append("NONE")

lines.append("")
lines.append(
    "----- TOP GREENY -> RAIOS CANDIDATES -----"
)

for item in sorted(
    comparison,
    key=lambda x:
        x["best_score"],
    reverse=True
)[:83]:

    lines.append("")
    lines.append(
        f"GREENY={item['greeny_path']}"
    )

    lines.append(
        f"CATEGORY={item['category']}"
    )

    lines.append(
        f"ACTION={item['candidate_action']}"
    )

    lines.append(
        f"SCORE={item['best_score']}"
    )

    for candidate in item[
        "raios_candidates"
    ][:5]:

        lines.append(
            "  -> "
            + candidate["family"]
            + " | "
            + candidate["file"]
            + " | score="
            + str(candidate["score"])
        )


txt_out.write_text(
    "\n".join(lines) + "\n",
    encoding="utf-8"
)


# ============================================================================
# 13. FINAL CONSOLE SUMMARY
# ============================================================================

print("\n" + "=" * 88)
print("AUDIT COMPLETE")
print("=" * 88)

print("\nCOGNITIVE COUNTS")

for family, info in inventory.items():
    print(
        f"{family:26} "
        f"{info['count']:>5}"
    )

print(
    f"{'journal-events':26} "
    f"{journal_events:>5}"
)

print("\nCONFIDENCE")

print(
    "fields checked :",
    len(confidence_findings)
)

print(
    "anomalies      :",
    len(confidence_anomalies)
)

print(
    "known skill present:",
    known_skill_path.exists()
)

print("\nV9 FOUNDATIONS")

for name, present in v9_foundations.items():
    print(
        f"{name:32} "
        f"{'YES' if present else 'NO'}"
    )

print("\nGREENY -> RAIOS")

for action, count in sorted(
    action_counts.items()
):
    print(
        f"{action:32} "
        f"{count}"
    )

print("\nOUTPUTS")

print(json_out)
print(txt_out)

print("""
GPU USED: NO
MODEL LOADED: NO
TRAINING EXECUTED: NO
RAIOS INPUT STATE MUTATED: NO

STOP HERE.
DO NOT RUN OLD MODEL-LOAD CELLS.
DO NOT TRAIN YET.
""")

RAIOS V8 STATE AUDIT + GREENY CAPABILITY ASSIMILATION
CPU ONLY | READ-ONLY INPUTS | NO MODEL | NO TRAINING

[PASS] ROOTS
RAIOS_DATASET : /kaggle/input/datasets/greenylife/raios-cognitive-state
GREENY_DATASET: /kaggle/input/datasets/greenylife/greeny-life
STATE         : /kaggle/input/datasets/greenylife/raios-cognitive-state/RAIOS-STATE-LATEST/RAIOS/raios-cognitive-factory/state

AUTHORITY / CONTINUITY

CURRENT
----------------------------------------
{
  "base_checkpoint": "0062a4f36a600d3ed2187f8e1c704e8e80dcfcc2db74723255e29df5f52a2a14",
  "commit_intent": "INSTALL_DURABILITY_TRANSACTION_ENGINE",
  "created_at": "2026-08-17T06:46:48.894574+00:00",
  "engine_sha256": "90ae4ba4c3de8df3980e9c4ee3dc0cff00f843dbfc2922f5314e8b3be64521a4",
  "manifest_hash": "feca41393895f25fc286046d61432fb72b77167be9234f1fd51daf4be661d242",
  "policy_hash": "ea26793d6fa59c58d14573c35770708feced005be9834048b555125260df00b0",
  "repository_sha": "3d9f58136d318ba07d743e127ab1e433605ce1ea",
  "schema": "raios

In [1]:
from pathlib import Path
from collections import Counter
import json
from datetime import datetime, timezone

print("=" * 92)
print("RAIOS V8.6.2 — GREENY ASSIMILATION PREFLIGHT")
print("CPU ONLY | NO MODEL | NO TRAINING | NO STATE MUTATION")
print("=" * 92)

# Requires `report` from the previous audit cell.
if "report" not in globals():
    audit = Path("/kaggle/working/RAIOS-V8-GREENY-ASSIMILATION-AUDIT.json")
    if not audit.exists():
        raise RuntimeError("Previous assimilation audit not found.")
    report = json.loads(audit.read_text(encoding="utf-8"))

comparison = report["comparison"]

ORDER = [
    "STRONG_REUSE_CANDIDATE",
    "REUSE_OR_COMPOSE_CANDIDATE",
    "EXTEND_COMPARE_CANDIDATE",
    "WEAK_OVERLAP_REVIEW",
    "NO_EQUIVALENT_FOUND_YET",
]

# ------------------------------------------------------------------
# 1. FULL 83-CAPABILITY MAP
# ------------------------------------------------------------------

print("\nCAPABILITY MAP")
print("-" * 92)

for action in ORDER:
    group = [x for x in comparison if x["candidate_action"] == action]

    print(f"\n### {action} ({len(group)})")

    for item in sorted(group, key=lambda x: x["best_score"], reverse=True):
        print(f"\nGREENY : {item['greeny_path']}")
        print(f"TYPE   : {item['category']}")
        print(f"SCORE  : {item['best_score']}")

        candidates = item.get("raios_candidates", [])

        if not candidates:
            print("RAIOS  : NONE")
            continue

        for candidate in candidates[:3]:
            print(
                "RAIOS  : "
                f"{candidate['family']} | "
                f"{candidate['file']} | "
                f"score={candidate['score']}"
            )

# ------------------------------------------------------------------
# 2. PROMOTION SAFETY / QUARANTINE PLAN
# ------------------------------------------------------------------

BAD_SKILL = (
    "683ca22e97a2cb919ab078321076bd06"
    "b146b1bf69e937a2062d68755e993b65"
)

actual_confidence_anomalies = []

for x in report["confidence_audit"]["anomalies"]:
    v = x.get("value")

    # Only runtime numeric values are treated as actual violations.
    if isinstance(v, bool):
        continue

    if isinstance(v, (int, float)) and not (0.0 <= float(v) <= 1.0):
        actual_confidence_anomalies.append(x)

affected_files = sorted({
    x["file"] for x in actual_confidence_anomalies
})

quarantine = {
    "schema": "raios.quarantine-plan.v1",
    "created_at": datetime.now(timezone.utc).isoformat(),

    "mode": "PLAN_ONLY",

    "reason": "CONFIDENCE_DOMAIN_VIOLATION",

    "invariant": {
        "confidence_min": 0.0,
        "confidence_max": 1.0,
        "reject_non_normalized_numeric_confidence": True,
    },

    "invalidated_skill": {
        "content_hash": BAD_SKILL,
        "skill_id": "RAIOS.REPOSITORY_ANALYSIS.MICRO.v1",
        "current_record_status": "PROMOTED",
        "effective_status": "QUARANTINED_PENDING_REVALIDATION",
    },

    "actual_numeric_anomalies": actual_confidence_anomalies,
    "affected_files": affected_files,

    "required_before_repromotion": [
        "install confidence normalization boundary",
        "install schema validation for confidence float[0,1]",
        "invalidate legacy promotion result",
        "rerun repository-analysis replay set",
        "recompute aggregate confidence from normalized replay values",
        "rerun promotion gate",
        "persist through durability transaction",
        "verify remote readback before final promotion",
    ],

    "state_mutated": False,
}

qpath = Path("/kaggle/working/RAIOS-V8.6.2-QUARANTINE-PLAN.json")
qpath.write_text(
    json.dumps(quarantine, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

# ------------------------------------------------------------------
# 3. ASSIMILATION DECISION QUEUE
# ------------------------------------------------------------------

queue = []

for item in comparison:
    candidates = item.get("raios_candidates", [])

    queue.append({
        "greeny_path": item["greeny_path"],
        "category": item["category"],
        "lexical_classification": item["candidate_action"],
        "lexical_score": item["best_score"],
        "top_raios_candidate": candidates[0] if candidates else None,

        # Nothing is promoted automatically.
        "semantic_verification": "PENDING",
        "final_action": "UNDECIDED",
        "allowed_final_actions": [
            "REUSE",
            "COMPOSE",
            "EXTEND",
            "IMPORT_AS_NEW",
            "IGNORE",
        ],
    })

queue_path = Path("/kaggle/working/RAIOS-V8.6.2-ASSIMILATION-QUEUE.json")
queue_path.write_text(
    json.dumps(queue, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

# ------------------------------------------------------------------
# 4. SUMMARY
# ------------------------------------------------------------------

counts = Counter(x["candidate_action"] for x in comparison)

print("\n" + "=" * 92)
print("V8.6.2 PREFLIGHT SUMMARY")
print("=" * 92)

print(f"TOTAL_GREENY_CAPABILITIES       : {len(comparison)}")

for name in ORDER:
    print(f"{name:31}: {counts.get(name, 0)}")

print()
print(f"NUMERIC_CONFIDENCE_VIOLATIONS   : {len(actual_confidence_anomalies)}")
print(f"AFFECTED_FILES                  : {len(affected_files)}")
print(f"INVALIDATED_SKILL               : {BAD_SKILL[:16]}...")
print("EFFECTIVE_SKILL_STATUS          : QUARANTINED_PENDING_REVALIDATION")

print("\nOUTPUTS")
print(queue_path)
print(qpath)

print("""
NEXT GATE:
  SEMANTIC_REUSE_VERIFICATION

GPU USED       : NO
MODEL LOADED   : NO
TRAINING       : NO
STATE MUTATED  : NO

DO NOT TRAIN.
DO NOT PROMOTE.
DO NOT COPY THE 83 CAPABILITIES INTO RAIOS YET.
""")

RAIOS V8.6.2 — GREENY ASSIMILATION PREFLIGHT
CPU ONLY | NO MODEL | NO TRAINING | NO STATE MUTATION


RuntimeError: Previous assimilation audit not found.

In [3]:
from pathlib import Path
import json

print("=" * 88)
print("RAIOS — RECOVER ASSIMILATION AUDIT")
print("CPU ONLY | NO MODEL | NO TRAINING")
print("=" * 88)

candidates = [
    Path("/kaggle/working/RAIOS-V8-GREENY-ASSIMILATION-AUDIT.json"),
]

# Search Kaggle inputs as well
input_root = Path("/kaggle/input")

if input_root.exists():
    candidates.extend(
        input_root.rglob("RAIOS-V8-GREENY-ASSIMILATION-AUDIT.json")
    )

found = None

for p in candidates:
    if p.exists():
        found = p
        break

if found:
    print(f"\n[FOUND] {found}")
    
    report = json.loads(found.read_text(encoding="utf-8"))
    
    print("[PASS] Audit loaded into variable: report")
    
    comparison = report.get("comparison", [])
    
    print(f"[PASS] Comparison records: {len(comparison)}")
    
    if len(comparison) != 83:
        print(f"[WARN] Expected 83 Greeny capabilities, found {len(comparison)}")
    else:
        print("[PASS] All 83 Greeny capabilities recovered")

else:
    print("\n[NOT FOUND]")
    print("The previous /kaggle/working audit disappeared with the old session.")
    print()
    print("NEXT_ACTION = REBUILD_AUDIT_FROM_PERSISTENT_INPUTS")
    print("Do NOT use GPU.")
    print("Do NOT train.")

RAIOS — RECOVER ASSIMILATION AUDIT
CPU ONLY | NO MODEL | NO TRAINING

[NOT FOUND]
The previous /kaggle/working audit disappeared with the old session.

NEXT_ACTION = REBUILD_AUDIT_FROM_PERSISTENT_INPUTS
Do NOT use GPU.
Do NOT train.


In [4]:
from __future__ import annotations

from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import json
import re
import math

print("=" * 96)
print("RAIOS V8.6.2 — PERSISTENT RECOVERY + GREENY ASSIMILATION PREFLIGHT")
print("CPU ONLY | NO MODEL | NO TRAINING | NO INPUT STATE MUTATION")
print("=" * 96)

# =============================================================================
# 0. PERSISTENT INPUT ROOTS
# =============================================================================

DATASETS_ROOT = Path("/kaggle/input/datasets/greenylife")

RAIOS_DATASET = DATASETS_ROOT / "raios-cognitive-state"
GREENY_DATASET = DATASETS_ROOT / "greeny-life"

RAIOS_ROOT = RAIOS_DATASET / "RAIOS-STATE-LATEST" / "RAIOS"
FACTORY = RAIOS_ROOT / "raios-cognitive-factory"
STATE = FACTORY / "state"

WORK = Path("/kaggle/working")

required = [
    RAIOS_DATASET,
    GREENY_DATASET,
    RAIOS_ROOT,
    FACTORY,
    STATE,
]

for path in required:
    if not path.exists():
        raise RuntimeError(f"Required persistent input missing: {path}")

print("\n[PASS] Persistent inputs")
print("RAIOS :", RAIOS_DATASET)
print("GREENY:", GREENY_DATASET)
print("STATE :", STATE)


# =============================================================================
# 1. HELPERS
# =============================================================================

def read_json(path: Path, default=None):
    try:
        return json.loads(
            path.read_text(
                encoding="utf-8-sig"
            )
        )
    except Exception:
        if default is not None:
            return default
        raise


def files_in(path: Path, suffix=None):
    if not path.exists():
        return []

    result = [
        p for p in path.rglob("*")
        if p.is_file()
    ]

    if suffix:
        result = [
            p for p in result
            if p.suffix.lower() == suffix
        ]

    return sorted(result)


def normalize_tokens(text: str) -> set[str]:

    stop = {
        "json", "state", "active", "current",
        "raios", "true", "false", "null",
        "none", "file", "data", "system",
        "version", "export", "import",
        "function", "const", "string",
        "number", "interface", "class",
    }

    return {
        token
        for token in re.findall(
            r"[a-z][a-z0-9_-]{2,}",
            text.lower()
        )
        if token not in stop
    }


def numeric_confidence_findings(value, path="$"):

    findings = []

    if isinstance(value, dict):

        for key, child in value.items():

            child_path = f"{path}.{key}"

            if "confidence" in str(key).lower():

                # Only numeric runtime values are treated
                # as confidence values.
                if (
                    isinstance(child, (int, float))
                    and not isinstance(child, bool)
                ):
                    findings.append({
                        "path": child_path,
                        "value": float(child),
                    })

            findings.extend(
                numeric_confidence_findings(
                    child,
                    child_path
                )
            )

    elif isinstance(value, list):

        for index, child in enumerate(value):

            findings.extend(
                numeric_confidence_findings(
                    child,
                    f"{path}[{index}]"
                )
            )

    return findings


def confidence_valid(value: float) -> bool:

    return (
        math.isfinite(value)
        and 0.0 <= value <= 1.0
    )


def load_family(name: str):

    root = STATE / name
    result = []

    for path in files_in(root, ".json"):

        data = read_json(path, {})

        raw = json.dumps(
            data,
            ensure_ascii=False
        )

        result.append({
            "file":
                str(path.relative_to(STATE)),

            "data":
                data,

            "tokens":
                normalize_tokens(
                    path.name + " " + raw
                )
        })

    return result


# =============================================================================
# 2. CERTIFIED V8 STATE
# =============================================================================

current = read_json(
    RAIOS_DATASET / "CURRENT.json"
)

source_truth = read_json(
    RAIOS_ROOT / "SOURCE-OF-TRUTH.json"
)

recovery = read_json(
    RAIOS_ROOT / "RECOVERY-CONTRACT.json"
)

latest_checkpoint = read_json(
    STATE / "checkpoints" / "LATEST.json"
)

durability_latest = read_json(
    STATE
    / "checkpoints"
    / "DURABILITY-LATEST.json"
)

durability_policy = read_json(
    STATE
    / "durability-policy"
    / "ACTIVE.json"
)

routing_policy = read_json(
    STATE
    / "routing-policies"
    / "ACTIVE.json"
)

learning_ledger = read_json(
    STATE
    / "manifests"
    / "learning-ledger.json"
)

print("\n" + "=" * 96)
print("CERTIFIED V8 BASELINE")
print("=" * 96)

print(
    "Repository SHA :",
    source_truth.get("sha")
)

print(
    "Branch         :",
    source_truth.get("branch")
)

print(
    "Resume point   :",
    recovery.get("resume_point")
)

print(
    "Checkpoint     :",
    latest_checkpoint.get("hash")
)

print(
    "Durability     :",
    durability_latest.get("status")
)

print(
    "Remote readback:",
    durability_latest.get(
        "remote_readback_verified"
    )
)

print(
    "Exact tree     :",
    durability_latest.get(
        "exact_tree_verified"
    )
)


# =============================================================================
# 3. RAIOS COGNITIVE FAMILIES
# =============================================================================

family_names = [
    "skills",
    "experiences",
    "failures",
    "benchmarks",
    "training-candidates",
    "decisions",
    "evidence",
    "performance",
    "model-profiles",
    "environment-profiles",
    "semantic-cache",
    "semantic-corpus",
    "semantic-atoms",
    "semantic-taxonomy",
]

families = {
    name: load_family(name)
    for name in family_names
}

journal_file = (
    STATE
    / "journal"
    / "events.jsonl"
)

journal_events = 0

if journal_file.exists():

    with journal_file.open(
        "r",
        encoding="utf-8",
        errors="ignore"
    ) as handle:

        journal_events = sum(
            1
            for line in handle
            if line.strip()
        )


print("\n" + "=" * 96)
print("COGNITIVE STATE COUNTS")
print("=" * 96)

for name in family_names:
    print(
        f"{name:26} "
        f"{len(families[name]):>5}"
    )

print(
    f"{'journal-events':26} "
    f"{journal_events:>5}"
)


# =============================================================================
# 4. CONFIDENCE NORMALIZATION AUDIT
# =============================================================================

confidence_fields = []
confidence_violations = []

for path in STATE.rglob("*.json"):

    if not path.is_file():
        continue

    data = read_json(path, None)

    if data is None:
        continue

    for item in numeric_confidence_findings(data):

        finding = {
            "file":
                str(path.relative_to(STATE)),

            "json_path":
                item["path"],

            "value":
                item["value"],
        }

        confidence_fields.append(
            finding
        )

        if not confidence_valid(
            item["value"]
        ):
            confidence_violations.append(
                finding
            )


print("\n" + "=" * 96)
print("CONFIDENCE AUDIT")
print("=" * 96)

print(
    "Numeric confidence fields:",
    len(confidence_fields)
)

print(
    "Actual violations        :",
    len(confidence_violations)
)

for item in confidence_violations:
    print(
        "VIOLATION |",
        item["file"],
        "|",
        item["json_path"],
        "=",
        item["value"]
    )


# =============================================================================
# 5. INVALIDATED SKILL SAFETY
# =============================================================================

BAD_SKILL = (
    "683ca22e97a2cb919ab078321076bd06"
    "b146b1bf69e937a2062d68755e993b65"
)

bad_skill_path = (
    STATE
    / "skills"
    / f"{BAD_SKILL}.json"
)

bad_skill = (
    read_json(
        bad_skill_path,
        {}
    )
    if bad_skill_path.exists()
    else None
)

print("\n" + "=" * 96)
print("INVALIDATED SKILL SAFETY")
print("=" * 96)

print(
    "Skill present:",
    bad_skill_path.exists()
)

if bad_skill:

    print(
        "Skill ID     :",
        bad_skill.get("skill_id")
    )

    print(
        "Stored status:",
        bad_skill.get("status")
    )

    print(
        "Stored avg confidence:",
        bad_skill.get(
            "metrics",
            {}
        ).get(
            "avg_confidence"
        )
    )

    print(
        "Effective status:",
        "QUARANTINED_PENDING_REVALIDATION"
    )


# =============================================================================
# 6. LOAD GREENY CAPABILITY BRIDGE
# =============================================================================

greeny = read_json(
    GREENY_DATASET
    / "greeny-capabilities.json"
)

mission = read_json(
    GREENY_DATASET
    / "mission.json"
)

greeny_capabilities = greeny.get(
    "capabilities",
    []
)

if len(greeny_capabilities) != 83:
    raise RuntimeError(
        f"Expected 83 Greeny capabilities, "
        f"found {len(greeny_capabilities)}"
    )

print("\n" + "=" * 96)
print("GREENY CAPABILITY BRIDGE")
print("=" * 96)

print(
    "Capabilities:",
    len(greeny_capabilities)
)

print(
    "Branch      :",
    greeny["repository"]["branch"]
)

print(
    "HEAD        :",
    greeny["repository"]["head"]
)


# =============================================================================
# 7. BUILD RAIOS CAPABILITY CORPUS
# =============================================================================

raios_objects = []

family_alias = {
    "skills": "SKILL",
    "experiences": "EXPERIENCE",
    "failures": "FAILURE",
    "benchmarks": "BENCHMARK",
    "training-candidates": "TRAINING_CANDIDATE",
    "decisions": "DECISION",
    "evidence": "EVIDENCE",
    "performance": "PERFORMANCE",
    "model-profiles": "MODEL_PROFILE",
    "environment-profiles": "ENVIRONMENT_PROFILE",
    "semantic-cache": "SEMANTIC_CACHE",
    "semantic-corpus": "SEMANTIC_CORPUS",
    "semantic-atoms": "SEMANTIC_ATOM",
    "semantic-taxonomy": "SEMANTIC_TAXONOMY",
}

for family_name, alias in family_alias.items():

    for obj in families[family_name]:

        raios_objects.append({
            "family": alias,
            "file": obj["file"],
            "tokens": obj["tokens"],
        })


# Add important control-plane objects.

control_objects = {
    "ROUTING_POLICY": routing_policy,
    "DURABILITY_POLICY": durability_policy,
    "LEARNING_LEDGER": learning_ledger,
    "RECOVERY_CONTRACT": recovery,
}

for name, data in control_objects.items():

    raios_objects.append({
        "family": "CONTROL_STATE",
        "file": name,
        "tokens": normalize_tokens(
            name
            + " "
            + json.dumps(
                data,
                ensure_ascii=False
            )
        ),
    })


# =============================================================================
# 8. GREENY -> RAIOS CANDIDATE MATCHING
# =============================================================================

comparison = []

for cap in greeny_capabilities:

    greeny_path = cap.get(
        "path",
        ""
    )

    category = cap.get(
        "category",
        "OTHER"
    )

    symbols = cap.get(
        "symbols",
        []
    )

    query = normalize_tokens(
        greeny_path
        + " "
        + category
        + " "
        + " ".join(symbols)
    )

    matches = []

    for obj in raios_objects:

        overlap = (
            query
            & obj["tokens"]
        )

        if not overlap:
            continue

        query_coverage = (
            len(overlap)
            / max(
                1,
                len(query)
            )
        )

        symbol_hits = sum(
            1
            for symbol in symbols
            if symbol.lower()
            in obj["tokens"]
        )

        score = (
            query_coverage * 0.85
            + min(
                symbol_hits,
                5
            ) * 0.03
        )

        if score < 0.08:
            continue

        matches.append({
            "family":
                obj["family"],

            "file":
                obj["file"],

            "score":
                round(
                    score,
                    4
                ),

            "query_coverage":
                round(
                    query_coverage,
                    4
                ),

            "symbol_hits":
                symbol_hits,

            "overlap":
                sorted(
                    overlap
                )[:40],
        })

    matches.sort(
        key=lambda item:
            item["score"],
        reverse=True
    )

    best = matches[:8]

    best_score = (
        best[0]["score"]
        if best
        else 0.0
    )

    if best_score >= 0.70:

        candidate_action = (
            "STRONG_REUSE_CANDIDATE"
        )

    elif best_score >= 0.45:

        candidate_action = (
            "REUSE_OR_COMPOSE_CANDIDATE"
        )

    elif best_score >= 0.25:

        candidate_action = (
            "EXTEND_COMPARE_CANDIDATE"
        )

    elif best_score >= 0.10:

        candidate_action = (
            "WEAK_OVERLAP_REVIEW"
        )

    else:

        candidate_action = (
            "NO_EQUIVALENT_FOUND_YET"
        )

    comparison.append({
        "greeny_path":
            greeny_path,

        "category":
            category,

        "symbols":
            symbols,

        "candidate_action":
            candidate_action,

        "best_score":
            best_score,

        "raios_candidates":
            best,

        "semantic_verification":
            "PENDING",

        "final_action":
            "UNDECIDED",
    })


counts = Counter(
    item["candidate_action"]
    for item in comparison
)


# =============================================================================
# 9. V9 FOUNDATION REUSE MAP
# =============================================================================

v9_foundations = {

    "FAST_LEARNING":
        len(
            families[
                "experiences"
            ]
        ) > 0,

    "FAILURE_MEMORY":
        len(
            families[
                "failures"
            ]
        ) > 0,

    "SKILL_MEMORY":
        len(
            families[
                "skills"
            ]
        ) > 0,

    "EVIDENCE_MEMORY":
        len(
            families[
                "evidence"
            ]
        ) > 0,

    "DECISION_MEMORY":
        len(
            families[
                "decisions"
            ]
        ) > 0,

    "PERFORMANCE_MEMORY":
        len(
            families[
                "performance"
            ]
        ) > 0,

    "MODEL_MEMORY":
        len(
            families[
                "model-profiles"
            ]
        ) > 0,

    "ENVIRONMENT_MEMORY":
        len(
            families[
                "environment-profiles"
            ]
        ) > 0,

    "SEMANTIC_CACHE":
        len(
            families[
                "semantic-cache"
            ]
        ) > 0,

    "SEMANTIC_CORPUS":
        len(
            families[
                "semantic-corpus"
            ]
        ) > 0,

    "SEMANTIC_ATOMS":
        len(
            families[
                "semantic-atoms"
            ]
        ) > 0,

    "SEMANTIC_TAXONOMY":
        len(
            families[
                "semantic-taxonomy"
            ]
        ) > 0,

    "TRAINING_CANDIDATES":
        len(
            families[
                "training-candidates"
            ]
        ) > 0,

    "COGNITIVE_JOURNAL":
        journal_events > 0,

    "CHECKPOINTS":
        (
            STATE
            / "checkpoints"
        ).exists(),

    "DURABILITY_POLICY":
        bool(
            durability_policy
        ),

    "DURABILITY_TRANSACTION_ENGINE":
        (
            FACTORY
            / "runtime"
            / "durability_transaction.py"
        ).exists(),

    "ROUTING_POLICY":
        bool(
            routing_policy
        ),
}


# =============================================================================
# 10. QUARANTINE PLAN — PLAN ONLY
# =============================================================================

quarantine = {
    "schema":
        "raios.quarantine-plan.v1",

    "created_at":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "state_mutated":
        False,

    "invalidated_skill": {
        "hash":
            BAD_SKILL,

        "skill_id":
            (
                bad_skill.get(
                    "skill_id"
                )
                if bad_skill
                else None
            ),

        "stored_status":
            (
                bad_skill.get(
                    "status"
                )
                if bad_skill
                else None
            ),

        "effective_status":
            "QUARANTINED_PENDING_REVALIDATION",
    },

    "confidence_violations":
        confidence_violations,

    "required_before_repromotion": [
        "normalize all confidence values to [0,1]",
        "reject non-normalized numeric confidence",
        "invalidate legacy promotion result",
        "rerun repository-analysis replay set",
        "recompute aggregate metrics",
        "rerun promotion gate",
        "persist through durability transaction",
        "verify remote readback",
    ],
}


# =============================================================================
# 11. WRITE RECOVERABLE WORKING OUTPUTS
# =============================================================================

audit = {
    "schema":
        "raios.v8.6.2.greeny-preflight.v1",

    "generated_at":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "certified_baseline": {
        "repository_sha":
            source_truth.get("sha"),

        "branch":
            source_truth.get("branch"),

        "resume_point":
            recovery.get("resume_point"),

        "checkpoint":
            latest_checkpoint.get("hash"),

        "durability_status":
            durability_latest.get("status"),
    },

    "cognitive_counts": {
        name:
            len(families[name])
        for name
        in family_names
    },

    "journal_events":
        journal_events,

    "confidence": {
        "numeric_fields":
            len(confidence_fields),

        "violations":
            confidence_violations,
    },

    "invalidated_skill":
        quarantine[
            "invalidated_skill"
        ],

    "v9_foundations":
        v9_foundations,

    "greeny": {
        "branch":
            greeny["repository"]["branch"],

        "head":
            greeny["repository"]["head"],

        "capabilities":
            len(
                greeny_capabilities
            ),
    },

    "action_counts":
        dict(counts),

    "comparison":
        comparison,
}

audit_path = (
    WORK
    / "RAIOS-V8.6.2-GREENY-PREFLIGHT.json"
)

audit_path.write_text(
    json.dumps(
        audit,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

quarantine_path = (
    WORK
    / "RAIOS-V8.6.2-QUARANTINE-PLAN.json"
)

quarantine_path.write_text(
    json.dumps(
        quarantine,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


# =============================================================================
# 12. PRINT HIGH-VALUE GROUPS
# =============================================================================

print("\n" + "=" * 96)
print("V8.6.2 PREFLIGHT SUMMARY")
print("=" * 96)

print(
    "RAIOS_REPOSITORY_SHA            :",
    source_truth.get("sha")
)

print(
    "RAIOS_RESUME_POINT              :",
    recovery.get("resume_point")
)

print(
    "LATEST_CHECKPOINT               :",
    latest_checkpoint.get("hash")
)

print(
    "DURABILITY_STATUS               :",
    durability_latest.get("status")
)

print(
    "GREENY_CAPABILITIES             :",
    len(comparison)
)

print(
    "NUMERIC_CONFIDENCE_VIOLATIONS   :",
    len(confidence_violations)
)

print(
    "INVALIDATED_SKILL_EFFECTIVE     :",
    "QUARANTINED_PENDING_REVALIDATION"
)

print("\nCAPABILITY CLASSES")

for name in [
    "STRONG_REUSE_CANDIDATE",
    "REUSE_OR_COMPOSE_CANDIDATE",
    "EXTEND_COMPARE_CANDIDATE",
    "WEAK_OVERLAP_REVIEW",
    "NO_EQUIVALENT_FOUND_YET",
]:
    print(
        f"{name:32}: "
        f"{counts.get(name,0)}"
    )


print("\n" + "=" * 96)
print("REUSE_OR_COMPOSE_CANDIDATE")
print("=" * 96)

reuse_compose = [
    item
    for item in comparison
    if item["candidate_action"]
    == "REUSE_OR_COMPOSE_CANDIDATE"
]

for item in sorted(
    reuse_compose,
    key=lambda x:
        x["best_score"],
    reverse=True
):

    print(
        "\nGREENY:",
        item["greeny_path"]
    )

    print(
        "CATEGORY:",
        item["category"]
    )

    print(
        "SCORE:",
        item["best_score"]
    )

    for candidate in item[
        "raios_candidates"
    ][:4]:

        print(
            "  ->",
            candidate["family"],
            "|",
            candidate["file"],
            "|",
            candidate["score"]
        )


print("\n" + "=" * 96)
print("NO_EQUIVALENT_FOUND_YET")
print("=" * 96)

no_equivalent = [
    item
    for item in comparison
    if item["candidate_action"]
    == "NO_EQUIVALENT_FOUND_YET"
]

for item in no_equivalent:

    print(
        "\nGREENY:",
        item["greeny_path"]
    )

    print(
        "CATEGORY:",
        item["category"]
    )

    print(
        "SYMBOLS:",
        item["symbols"]
    )

    print(
        "BEST_SCORE:",
        item["best_score"]
    )


print("\n" + "=" * 96)
print("V9 FOUNDATIONS")
print("=" * 96)

for name, present in v9_foundations.items():

    print(
        f"{name:36} "
        f"{'YES' if present else 'NO'}"
    )


print("\nOUTPUTS")
print(audit_path)
print(quarantine_path)

print("""
GPU USED      : NO
MODEL LOADED  : NO
TRAINING      : NO
INPUT MUTATED : NO

NEXT GATE:
SEMANTIC_REUSE_VERIFICATION

STOP HERE.
""")

RAIOS V8.6.2 — PERSISTENT RECOVERY + GREENY ASSIMILATION PREFLIGHT
CPU ONLY | NO MODEL | NO TRAINING | NO INPUT STATE MUTATION

[PASS] Persistent inputs
RAIOS : /kaggle/input/datasets/greenylife/raios-cognitive-state
GREENY: /kaggle/input/datasets/greenylife/greeny-life
STATE : /kaggle/input/datasets/greenylife/raios-cognitive-state/RAIOS-STATE-LATEST/RAIOS/raios-cognitive-factory/state

CERTIFIED V8 BASELINE
Repository SHA : 3d9f58136d318ba07d743e127ab1e433605ce1ea
Branch         : raios/gl-005-convergence
Resume point   : V8.6.1
Checkpoint     : 0062a4f36a600d3ed2187f8e1c704e8e80dcfcc2db74723255e29df5f52a2a14
Durability     : DURABILITY_PASS
Remote readback: True
Exact tree     : True

COGNITIVE STATE COUNTS
skills                         6
experiences                   17
failures                      10
benchmarks                    15
training-candidates            4
decisions                      2
evidence                      25
performance                    3
model-profiles  

In [5]:
from __future__ import annotations

from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
import json
import re

print("=" * 100)
print("RAIOS V8.6.2 — STRUCTURAL REUSE VERIFICATION + CONTINUITY GAP ANALYSIS")
print("CPU ONLY | NO MODEL | NO TRAINING | NO INPUT MUTATION")
print("=" * 100)

DATASETS = Path("/kaggle/input/datasets/greenylife")

RAIOS_DATASET = DATASETS / "raios-cognitive-state"
GREENY_DATASET = DATASETS / "greeny-life"

RAIOS_ROOT = RAIOS_DATASET / "RAIOS-STATE-LATEST" / "RAIOS"
FACTORY = RAIOS_ROOT / "raios-cognitive-factory"
STATE = FACTORY / "state"

WORK = Path("/kaggle/working")

PREFLIGHT = WORK / "RAIOS-V8.6.2-GREENY-PREFLIGHT.json"

# ---------------------------------------------------------------------------
# 1. RECOVER PREFLIGHT IF /working WAS RESET
# ---------------------------------------------------------------------------

if PREFLIGHT.exists():
    audit = json.loads(PREFLIGHT.read_text(encoding="utf-8"))
else:
    raise RuntimeError(
        "RAIOS-V8.6.2-GREENY-PREFLIGHT.json is missing in this session. "
        "Run the previous V8.6.2 persistent recovery cell first."
    )

comparison = audit["comparison"]

print("\n[PASS] Preflight loaded")
print("       Capabilities:", len(comparison))


# ---------------------------------------------------------------------------
# 2. HELPERS
# ---------------------------------------------------------------------------

def read_json(path: Path, default=None):
    try:
        return json.loads(path.read_text(encoding="utf-8-sig"))
    except Exception:
        return default


def flatten_strings(value):
    values = []

    if isinstance(value, dict):
        for key, child in value.items():
            values.append(str(key))
            values.extend(flatten_strings(child))

    elif isinstance(value, list):
        for child in value:
            values.extend(flatten_strings(child))

    elif value is not None:
        values.append(str(value))

    return values


def tokens(text: str):
    stop = {
        "json", "state", "current", "active",
        "file", "data", "system", "raios",
        "true", "false", "none", "null",
        "version", "string", "number",
        "lib", "intelligence", "canonical",
    }

    return {
        x
        for x in re.findall(
            r"[a-z][a-z0-9_-]{2,}",
            text.lower()
        )
        if x not in stop
    }


# ---------------------------------------------------------------------------
# 3. BUILD STRUCTURAL SIGNATURES FOR RAIOS OBJECTS
# ---------------------------------------------------------------------------

family_dirs = {
    "SKILL": "skills",
    "EXPERIENCE": "experiences",
    "FAILURE": "failures",
    "BENCHMARK": "benchmarks",
    "TRAINING_CANDIDATE": "training-candidates",
    "DECISION": "decisions",
    "EVIDENCE": "evidence",
    "PERFORMANCE": "performance",
    "MODEL_PROFILE": "model-profiles",
    "ENVIRONMENT_PROFILE": "environment-profiles",
    "SEMANTIC_CACHE": "semantic-cache",
    "SEMANTIC_CORPUS": "semantic-corpus",
    "SEMANTIC_ATOM": "semantic-atoms",
}

raios_objects = []

for family, dirname in family_dirs.items():

    root = STATE / dirname

    if not root.exists():
        continue

    for path in root.rglob("*.json"):

        data = read_json(path, {})
        text = " ".join(flatten_strings(data))

        raios_objects.append({
            "family": family,
            "file": str(path.relative_to(STATE)),
            "tokens": tokens(text + " " + path.name),
            "keys": set(data.keys()) if isinstance(data, dict) else set(),
        })


# ---------------------------------------------------------------------------
# 4. ADD CONTROL-PLANE PRIMITIVES
# ---------------------------------------------------------------------------

control_files = {
    "REMOTE_CURRENT":
        RAIOS_DATASET / "CURRENT.json",

    "RECOVERY_CONTRACT":
        RAIOS_ROOT / "RECOVERY-CONTRACT.json",

    "SOURCE_OF_TRUTH":
        RAIOS_ROOT / "SOURCE-OF-TRUTH.json",

    "DURABLE_MANIFEST":
        RAIOS_ROOT / "DURABLE-MANIFEST.json",

    "LATEST_CHECKPOINT":
        STATE / "checkpoints" / "LATEST.json",

    "DURABILITY_LATEST":
        STATE / "checkpoints" / "DURABILITY-LATEST.json",

    "ROUTING_POLICY":
        STATE / "routing-policies" / "ACTIVE.json",

    "DURABILITY_POLICY":
        STATE / "durability-policy" / "ACTIVE.json",

    "LEARNING_LEDGER":
        STATE / "manifests" / "learning-ledger.json",

    "SEMANTIC_STATE_CONTRACT":
        STATE / "doctrine" / "SEMANTIC-STATE-CONTRACT.json",
}

for name, path in control_files.items():

    if not path.exists():
        continue

    data = read_json(path, {})
    text = " ".join(flatten_strings(data))

    raios_objects.append({
        "family": "CONTROL_PLANE",
        "file": name,
        "tokens": tokens(name + " " + text),
        "keys": set(data.keys()) if isinstance(data, dict) else set(),
    })


# ---------------------------------------------------------------------------
# 5. FUNCTIONAL CONCEPT MAP
#
# This is deliberately broader than lexical filename matching.
# ---------------------------------------------------------------------------

concepts = {
    "EVIDENCE_VALIDATION": {
        "evidence", "verify", "verification", "validation",
        "confidence", "decision", "unknown", "proof"
    },

    "GOVERNANCE_GATE": {
        "governance", "approval", "policy", "authority",
        "execution", "block", "prohibition", "canonical"
    },

    "ROUTING": {
        "route", "routing", "router", "decision",
        "fallback", "policy", "orchestration"
    },

    "ARCHITECTURE_MANIFEST": {
        "architecture", "manifest", "component",
        "dependency", "canonical", "registry"
    },

    "EXECUTION_CONTRACT": {
        "execution", "contract", "approval",
        "prohibit", "policy", "action", "authority"
    },

    "WORKFLOW_GOVERNANCE": {
        "workflow", "governance", "approval",
        "execution", "decision", "route"
    },

    "AUDIT_ENGINE": {
        "audit", "evidence", "verify",
        "validation", "finding", "trace"
    },

    "HANDOFF": {
        "handoff", "resume", "agent", "task",
        "continuation", "next", "context"
    },

    "PROJECT_CURRENT_STATE": {
        "current", "state", "resume", "task",
        "checkpoint", "status", "next"
    },

    "LOCK_COORDINATION": {
        "lock", "lease", "owner", "agent",
        "path", "collision", "task"
    },
}


def infer_concepts(path: str, category: str, symbols: list[str]):
    source = tokens(
        path + " " + category + " " + " ".join(symbols)
    )

    scored = []

    for concept, keywords in concepts.items():
        overlap = source & keywords

        if overlap:
            scored.append((
                concept,
                len(overlap) / len(keywords),
                sorted(overlap)
            ))

    scored.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return scored


def rank_raios_for_concept(concept: str):

    keywords = concepts[concept]
    results = []

    for obj in raios_objects:

        overlap = keywords & obj["tokens"]

        if not overlap:
            continue

        coverage = len(overlap) / len(keywords)

        results.append({
            "family": obj["family"],
            "file": obj["file"],
            "coverage": round(coverage, 4),
            "overlap": sorted(overlap),
        })

    results.sort(
        key=lambda x: x["coverage"],
        reverse=True
    )

    return results[:8]


# ---------------------------------------------------------------------------
# 6. VERIFY THE 7 REUSE/COMPOSE CANDIDATES
# ---------------------------------------------------------------------------

reuse_items = [
    x for x in comparison
    if x["candidate_action"] == "REUSE_OR_COMPOSE_CANDIDATE"
]

verified_reuse = []

print("\n" + "=" * 100)
print("STRUCTURAL VERIFICATION — REUSE / COMPOSE")
print("=" * 100)

for item in reuse_items:

    inferred = infer_concepts(
        item["greeny_path"],
        item["category"],
        item.get("symbols", [])
    )

    concept_evidence = []

    for concept, concept_score, trigger in inferred[:4]:

        matches = rank_raios_for_concept(concept)

        concept_evidence.append({
            "concept": concept,
            "greeny_trigger": trigger,
            "concept_score": round(concept_score, 4),
            "raios_matches": matches,
        })

    top_coverages = [
        m["coverage"]
        for c in concept_evidence
        for m in c["raios_matches"][:3]
    ]

    structural_strength = (
        max(top_coverages)
        if top_coverages
        else 0.0
    )

    if structural_strength >= 0.65:
        final_candidate = "REUSE_EXISTING_RAIOS_PRIMITIVES"

    elif structural_strength >= 0.40:
        final_candidate = "COMPOSE_EXISTING_RAIOS_PRIMITIVES"

    elif structural_strength >= 0.20:
        final_candidate = "EXTEND_EXISTING_RAIOS_PRIMITIVES"

    else:
        final_candidate = "SEMANTIC_MODEL_REVIEW_REQUIRED"

    record = {
        "greeny_path": item["greeny_path"],
        "original_score": item["best_score"],
        "structural_strength": structural_strength,
        "candidate_decision": final_candidate,
        "concept_evidence": concept_evidence,
    }

    verified_reuse.append(record)

    print("\nGREENY :", item["greeny_path"])
    print("DECISION:", final_candidate)
    print("STRENGTH:", round(structural_strength, 4))

    for c in concept_evidence[:3]:

        print(
            "  CONCEPT:",
            c["concept"],
            "| trigger:",
            ",".join(c["greeny_trigger"])
        )

        for match in c["raios_matches"][:3]:

            print(
                "    ->",
                match["family"],
                "|",
                match["file"],
                "| coverage=",
                match["coverage"]
            )


# ---------------------------------------------------------------------------
# 7. ANALYZE THE FOUR PREVIOUS "NO EQUIVALENT" ITEMS
# ---------------------------------------------------------------------------

gap_items = [
    x for x in comparison
    if x["candidate_action"] == "NO_EQUIVALENT_FOUND_YET"
]

continuity_analysis = []

print("\n" + "=" * 100)
print("CONTINUITY CAPABILITY ANALYSIS")
print("=" * 100)

for item in gap_items:

    path = item["greeny_path"].lower()

    if "handoff" in path:
        concept = "HANDOFF"

    elif "current-state" in path:
        concept = "PROJECT_CURRENT_STATE"

    elif "locks" in path:
        concept = "LOCK_COORDINATION"

    else:
        concept = "PROJECT_CURRENT_STATE"

    matches = rank_raios_for_concept(concept)

    best = (
        matches[0]["coverage"]
        if matches
        else 0.0
    )

    if concept == "HANDOFF":

        if best >= 0.50:
            decision = "COMPOSE_FROM_EXISTING_CONTINUITY"

        else:
            decision = "REAL_EXTENSION_CANDIDATE"

    elif concept == "PROJECT_CURRENT_STATE":

        if best >= 0.50:
            decision = "REUSE_EXISTING_CONTINUITY"

        else:
            decision = "EXTEND_PROJECT_LEVEL_STATE"

    elif concept == "LOCK_COORDINATION":

        if best >= 0.50:
            decision = "REUSE_EXISTING_LOCK_PRIMITIVE"

        else:
            decision = "REAL_EXTENSION_CANDIDATE"

    record = {
        "greeny_path": item["greeny_path"],
        "concept": concept,
        "best_existing_coverage": best,
        "decision": decision,
        "raios_matches": matches,
    }

    continuity_analysis.append(record)

    print("\nGREENY :", item["greeny_path"])
    print("CONCEPT:", concept)
    print("DECISION:", decision)
    print("BEST EXISTING COVERAGE:", best)

    for match in matches[:5]:

        print(
            "  ->",
            match["family"],
            "|",
            match["file"],
            "| coverage=",
            match["coverage"],
            "|",
            ",".join(match["overlap"])
        )


# ---------------------------------------------------------------------------
# 8. BUILD REAL GAP SUMMARY
# ---------------------------------------------------------------------------

decision_counts = Counter(
    x["candidate_decision"]
    for x in verified_reuse
)

continuity_counts = Counter(
    x["decision"]
    for x in continuity_analysis
)

real_extensions = [
    x for x in continuity_analysis
    if x["decision"] in {
        "REAL_EXTENSION_CANDIDATE",
        "EXTEND_PROJECT_LEVEL_STATE",
    }
]

semantic_model_needed = [
    x for x in verified_reuse
    if x["candidate_decision"] == "SEMANTIC_MODEL_REVIEW_REQUIRED"
]


# ---------------------------------------------------------------------------
# 9. OUTPUT
# ---------------------------------------------------------------------------

result = {
    "schema":
        "raios.v8.6.2.structural-reuse-verification.v1",

    "generated_at":
        datetime.now(timezone.utc).isoformat(),

    "mode": {
        "cpu_only": True,
        "model_loaded": False,
        "training": False,
        "state_mutated": False,
    },

    "reuse_verification":
        verified_reuse,

    "continuity_analysis":
        continuity_analysis,

    "semantic_model_review_needed":
        semantic_model_needed,

    "real_extension_candidates":
        real_extensions,

    "summary": {
        "reuse_decisions":
            dict(decision_counts),

        "continuity_decisions":
            dict(continuity_counts),

        "semantic_model_review_count":
            len(semantic_model_needed),

        "real_extension_candidate_count":
            len(real_extensions),
    },
}

out = WORK / "RAIOS-V8.6.2-STRUCTURAL-REUSE-VERIFICATION.json"

out.write_text(
    json.dumps(
        result,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


# ---------------------------------------------------------------------------
# 10. FINAL SUMMARY
# ---------------------------------------------------------------------------

print("\n" + "=" * 100)
print("STRUCTURAL REUSE VERIFICATION SUMMARY")
print("=" * 100)

print("\n7 REUSE/COMPOSE ITEMS")

for key, value in sorted(decision_counts.items()):
    print(f"{key:42} : {value}")

print("\n4 CONTINUITY ITEMS")

for key, value in sorted(continuity_counts.items()):
    print(f"{key:42} : {value}")

print()
print(
    "SEMANTIC MODEL REVIEW REQUIRED :",
    len(semantic_model_needed)
)

print(
    "REAL EXTENSION CANDIDATES      :",
    len(real_extensions)
)

print("\nREAL EXTENSION PATHS")

if real_extensions:

    for item in real_extensions:
        print(
            " -",
            item["greeny_path"],
            "=>",
            item["decision"]
        )

else:
    print(" NONE")

print("\nOUTPUT:")
print(out)

print("""
GPU USED      : NO
MODEL LOADED  : NO
TRAINING      : NO
STATE MUTATED : NO

DECISION:
If SEMANTIC MODEL REVIEW REQUIRED = 0,
do NOT enable T4 yet.

If > 0, review those exact cases first before deciding
whether a short T4 semantic pass is economically justified.
""")

RAIOS V8.6.2 — STRUCTURAL REUSE VERIFICATION + CONTINUITY GAP ANALYSIS
CPU ONLY | NO MODEL | NO TRAINING | NO INPUT MUTATION

[PASS] Preflight loaded
       Capabilities: 83

STRUCTURAL VERIFICATION — REUSE / COMPOSE

GREENY : canonical/intelligence/adapters/gl-dos-governance-gate.ts
DECISION: SEMANTIC_MODEL_REVIEW_REQUIRED
STRENGTH: 0.0

GREENY : canonical/intelligence/architecture.manifest.json
DECISION: EXTEND_EXISTING_RAIOS_PRIMITIVES
STRENGTH: 0.3333
  CONCEPT: ARCHITECTURE_MANIFEST | trigger: architecture,manifest
    -> EVIDENCE | evidence/e662b7f7c6416cee22681f02bd4859c3945019b0f391c6ace057ab2d4ca85439.json | coverage= 0.3333
    -> SKILL | skills/0435c33dbd2e2b2fd19133d9a65e44d9c0345b1546fa58b05ad670233cbca32e.json | coverage= 0.1667
    -> SKILL | skills/f629bd0b280be74e4a7638a1f165a8998b5dcaa273705058aef7de0fb74977e5.json | coverage= 0.1667

GREENY : canonical/intelligence/contracts/execution-contract.json
DECISION: SEMANTIC_MODEL_REVIEW_REQUIRED
STRENGTH: 0.0

GREENY : cano

In [6]:
canonical/intelligence/adapters/gl-dos-governance-gate.ts
canonical/intelligence/contracts/execution-contract.json
canonical/intelligence/intelligence/engines/audit-engine.ts
lib/intelligence/workflow-governance.ts

.ai-os/handoffs/20260815-171701-gemini-cli-GL-001.md
.ai-os/handoffs/20260815-173642-gemini-cli-GL-003.md
.ai-os/state/CURRENT-STATE.json
.ai-os/state/LOCKS.json

SyntaxError: invalid decimal literal (2287808694.py, line 6)

In [7]:
from pathlib import Path
import zipfile, hashlib

EXPECTED = "24ed4d85c4b5ae0265f7a8de115ce21dee586ba5191cbf931012ab21078243ae"

roots = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
]

candidates = []

for root in roots:
    if root.exists():
        candidates.extend(root.rglob("RAIOS-GREENY-DEEP-EVIDENCE.zip"))

if not candidates:
    raise FileNotFoundError("RAIOS-GREENY-DEEP-EVIDENCE.zip not found")

package = candidates[0]

h = hashlib.sha256()
with package.open("rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        h.update(chunk)

actual = h.hexdigest()

print("PACKAGE:", package)
print("SHA256 :", actual)

if actual != EXPECTED:
    raise RuntimeError("SHA256 mismatch")

out = Path("/kaggle/working/RAIOS-GREENY-DEEP-EVIDENCE")

if out.exists():
    import shutil
    shutil.rmtree(out)

out.mkdir(parents=True)

with zipfile.ZipFile(package, "r") as z:
    z.extractall(out)

files = [p for p in out.rglob("*") if p.is_file()]

print("\nFILES:", len(files))

for p in files:
    print(" -", p.relative_to(out), "|", p.stat().st_size, "bytes")

print("""
STATUS: DEEP_EVIDENCE_LOADED
GPU USED: NO
MODEL LOADED: NO
TRAINING: NO
""")

FileNotFoundError: RAIOS-GREENY-DEEP-EVIDENCE.zip not found

In [8]:
from pathlib import Path

print("=" * 80)
print("KAGGLE INPUT DISCOVERY")
print("=" * 80)

for p in Path("/kaggle/input").rglob("*"):
    if p.is_file():
        s = str(p).lower()
        if "evidence" in s or "raios" in s or "greeny" in s:
            print(p)

print("=" * 80)

KAGGLE INPUT DISCOVERY
/kaggle/input/datasets/greenylife/raios-version-evidence/RAIOS-VERSION-EVIDENCE.json
/kaggle/input/datasets/greenylife/raios-cognitive-state/CURRENT.json
/kaggle/input/datasets/greenylife/raios-cognitive-state/dataset-metadata.json
/kaggle/input/datasets/greenylife/greeny-life/MANIFEST.json
/kaggle/input/datasets/greenylife/greeny-life/greeny-capabilities.json
/kaggle/input/datasets/greenylife/greeny-life/mission.json
/kaggle/input/datasets/greenylife/greeny-life/MISSION.md
/kaggle/input/datasets/greenylife/greeny-life/greeny-capabilities.md
/kaggle/input/datasets/greenylife/raios-cognitive-state/RAIOS-STATE-LATEST/RAIOS/RECOVERY-CONTRACT.json
/kaggle/input/datasets/greenylife/raios-cognitive-state/RAIOS-STATE-LATEST/RAIOS/DURABLE-MANIFEST.json
/kaggle/input/datasets/greenylife/raios-cognitive-state/RAIOS-STATE-LATEST/RAIOS/SOURCE-OF-TRUTH.json
/kaggle/input/datasets/greenylife/raios-cognitive-state/RAIOS-STATE-LATEST/RAIOS/raios-cognitive-factory/runtime/durabil

In [9]:
from pathlib import Path

ROOT = Path("/kaggle/input/datasets/greenylife")

print("=" * 90)
print("KAGGLE DATASET MAP — RAIOS / GREENY")
print("CPU ONLY | READ ONLY")
print("=" * 90)

if not ROOT.exists():
    raise RuntimeError(f"Missing root: {ROOT}")

for dataset in sorted(p for p in ROOT.iterdir() if p.is_dir()):

    files = [p for p in dataset.rglob("*") if p.is_file()]

    total_bytes = 0
    for p in files:
        try:
            total_bytes += p.stat().st_size
        except OSError:
            pass

    print()
    print("DATASET :", dataset.name)
    print("FILES   :", len(files))
    print("BYTES   :", total_bytes)

    important = []

    for p in files:
        name = p.name.lower()
        rel = str(p.relative_to(dataset))

        if (
            "deep-evidence" in name
            or "manifest" in name
            or "collection" in name
            or "tool" in name
            or "agent" in name
            or "intelligence" in name
            or "handoff" in name
            or "locks" in name
            or "current-state" in name
            or name.endswith(".ts")
            or name.endswith(".md")
        ):
            important.append(rel)

    for rel in important[:100]:
        print("  -", rel)

    if len(important) > 100:
        print(f"  ... +{len(important)-100} important-looking files")

print()
print("=" * 90)
print("EXACT DEEP-EVIDENCE MANIFEST SEARCH")
print("=" * 90)

matches = list(ROOT.rglob("DEEP-EVIDENCE-MANIFEST.json"))

if matches:
    for p in matches:
        print("[FOUND]", p)
else:
    print("[NOT FOUND] DEEP-EVIDENCE-MANIFEST.json")

print()
print("GPU USED: NO")
print("MODEL LOADED: NO")
print("TRAINING: NO")

KAGGLE DATASET MAP — RAIOS / GREENY
CPU ONLY | READ ONLY

DATASET : collection-for-understand
FILES   : 117
BYTES   : 67454440
  - E3-SOURCE-TRACE-PACKAGE/CANONICAL__canonical__intelligence__eos-domain-activation-blueprint-v1.json.txt
  - E3-SOURCE-TRACE-PACKAGE/CANONICAL__canonical__intelligence__eos-gldos-architecture-alignment.json.txt
  - E3-SOURCE-TRACE-PACKAGE/CANONICAL__canonical__intelligence__architecture.manifest.json.txt
  - E3-SOURCE-TRACE-PACKAGE/CANONICAL__canonical__system_manifest.json.txt
  - E3-SOURCE-TRACE-PACKAGE/CANONICAL__canonical__intelligence__eos-real-execution-path-map.json.txt
  - E3-SOURCE-TRACE-PACKAGE/CANONICAL__canonical__intelligence__eos-gels-architecture-alignment.json.txt
  - E3-SOURCE-TRACE-PACKAGE/CANONICAL__canonical__intelligence__eos-enterprise-architecture-final.json.txt
  - E3-SOURCE-TRACE-PACKAGE/CANONICAL__canonical__intelligence__eos-execution-truth-map.json.txt
  - unified-intelligence-restore-5946c6bc/unified-intelligence/README.md
  - un

In [10]:
from __future__ import annotations

from pathlib import Path
from collections import defaultdict
import hashlib
import json
import re

print("=" * 104)
print("RAIOS V8.6.2 — COLLECTION INTELLIGENCE HARVEST")
print("CPU ONLY | READ ONLY | NO MODEL | NO TRAINING")
print("=" * 104)

COLLECTION = Path(
    "/kaggle/input/datasets/greenylife/collection-for-understand"
)

WORK = Path("/kaggle/working")

if not COLLECTION.exists():
    raise RuntimeError(f"Collection missing: {COLLECTION}")


# ============================================================================
# TARGETS WE PREVIOUSLY NEEDED
# ============================================================================

targets = [
    "gl-dos-governance-gate.ts",
    "execution-contract.json",
    "audit-engine.ts",
    "workflow-governance.ts",

    "20260815-171701-gemini-cli-GL-001.md",
    "20260815-173642-gemini-cli-GL-003.md",
    "CURRENT-STATE.json",
    "LOCKS.json",
]


# ============================================================================
# HIGH-VALUE CAPABILITY HINTS
# ============================================================================

high_value_patterns = {
    "AGENT_INFRASTRUCTURE": [
        "agent",
        "handoff",
        "routing",
        "router",
        "task",
        "lock",
        "orchestrator",
    ],

    "CODE_INTELLIGENCE": [
        "code-intelligence",
        "dependency-intelligence",
        "duplicate-intelligence",
        "source-trace",
        "static-analysis",
    ],

    "GOVERNANCE": [
        "governance",
        "approval",
        "policy",
        "execution-contract",
        "controlled-runtime",
    ],

    "ARCHITECTURE": [
        "architecture.manifest",
        "architecture-alignment",
        "system_manifest",
        "execution-truth",
        "execution-path",
    ],

    "LEARNING": [
        "learning",
        "training",
        "experience",
        "feedback",
        "evolution",
    ],

    "SEMANTIC_RETRIEVAL": [
        "semantic",
        "retrieval",
        "embedding",
        "taxonomy",
        "knowledge",
    ],

    "MAINTENANCE": [
        "maintenance",
        "repair",
        "audit",
        "duplicate",
        "cleanup",
        "assurance",
    ],
}


# ============================================================================
# HELPERS
# ============================================================================

def sha256(path: Path):
    h = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


def normalize(name: str):
    return re.sub(
        r"[^a-z0-9]+",
        "",
        name.lower()
    )


all_files = [
    p for p in COLLECTION.rglob("*")
    if p.is_file()
]

print("\nCOLLECTION FILES:", len(all_files))


# ============================================================================
# 1. EXACT / NEAR TARGET SEARCH
# ============================================================================

target_results = {}

print("\n" + "=" * 104)
print("TARGET FILE SEARCH")
print("=" * 104)

for target in targets:

    target_norm = normalize(target)

    exact = []
    near = []

    for p in all_files:

        name = p.name

        if name.lower() == target.lower():
            exact.append(p)
            continue

        name_norm = normalize(name)

        # Also catch .txt-wrapped repository exports
        if target_norm in name_norm or name_norm in target_norm:
            near.append(p)

    records = []

    for kind, matches in [
        ("EXACT", exact),
        ("NEAR", near),
    ]:

        for p in matches[:50]:

            try:
                size = p.stat().st_size
                digest = sha256(p)
            except Exception:
                size = -1
                digest = None

            records.append({
                "match_type": kind,
                "path": str(
                    p.relative_to(COLLECTION)
                ),
                "bytes": size,
                "sha256": digest,
            })

    target_results[target] = records

    print(f"\nTARGET: {target}")

    if not records:
        print("  [NOT FOUND]")
        continue

    for r in records[:15]:
        print(
            f"  [{r['match_type']}] "
            f"{r['path']} "
            f"| {r['bytes']} bytes"
        )


# ============================================================================
# 2. HIGH-VALUE INTELLIGENCE HARVEST
# ============================================================================

harvest = defaultdict(list)

for p in all_files:

    rel = str(
        p.relative_to(COLLECTION)
    )

    text = rel.lower()

    for category, patterns in high_value_patterns.items():

        if any(
            pattern in text
            for pattern in patterns
        ):

            harvest[category].append(rel)


print("\n" + "=" * 104)
print("HIGH-VALUE CAPABILITY SUPPLY POOL")
print("=" * 104)

for category in sorted(harvest):

    unique = sorted(
        set(harvest[category])
    )

    print(
        f"\n{category}: "
        f"{len(unique)} candidate files"
    )

    for rel in unique[:30]:
        print("  -", rel)

    if len(unique) > 30:
        print(
            f"  ... +{len(unique)-30} more"
        )


# ============================================================================
# 3. DUPLICATE BASENAME DETECTION
#
# Useful because collection contains restored/legacy/current-ish copies.
# ============================================================================

by_name = defaultdict(list)

for p in all_files:
    by_name[p.name.lower()].append(p)

duplicate_names = {
    name: paths
    for name, paths in by_name.items()
    if len(paths) > 1
}


print("\n" + "=" * 104)
print("REPEATED FILENAMES — POSSIBLE VERSION / DUPLICATE SOURCES")
print("=" * 104)

interesting_duplicates = []

for name, paths in duplicate_names.items():

    if any(
        hint in name
        for hints in high_value_patterns.values()
        for hint in hints
    ):

        interesting_duplicates.append(
            (name, paths)
        )

for name, paths in sorted(
    interesting_duplicates
)[:100]:

    print(f"\n{name} ({len(paths)} copies)")

    for p in paths[:10]:
        print(
            "  -",
            p.relative_to(COLLECTION)
        )


# ============================================================================
# 4. BUILD MACHINE REPORT
# ============================================================================

report = {
    "schema":
        "raios.collection-harvest.v1",

    "collection":
        str(COLLECTION),

    "file_count":
        len(all_files),

    "target_search":
        target_results,

    "high_value_pool": {
        category:
            sorted(set(paths))
        for category, paths
        in harvest.items()
    },

    "interesting_duplicate_names": {
        name: [
            str(p.relative_to(COLLECTION))
            for p in paths
        ]
        for name, paths
        in interesting_duplicates
    },

    "mode": {
        "cpu_only": True,
        "model_loaded": False,
        "training": False,
        "input_mutated": False,
    }
}


OUT = (
    WORK
    / "RAIOS-V8.6.2-COLLECTION-HARVEST.json"
)

OUT.write_text(
    json.dumps(
        report,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


# ============================================================================
# 5. FINAL SUMMARY
# ============================================================================

print("\n" + "=" * 104)
print("COLLECTION HARVEST SUMMARY")
print("=" * 104)

found_exact = 0
found_any = 0

for target, records in target_results.items():

    if records:
        found_any += 1

    if any(
        x["match_type"] == "EXACT"
        for x in records
    ):
        found_exact += 1


print(
    "TARGETS                 :",
    len(targets)
)

print(
    "TARGETS WITH ANY MATCH  :",
    found_any
)

print(
    "TARGETS WITH EXACT MATCH:",
    found_exact
)

print()
print("SUPPLY POOL COUNTS:")

for category in sorted(harvest):
    print(
        f"{category:26} "
        f"{len(set(harvest[category]))}"
    )

print()
print("OUTPUT:")
print(OUT)

print("""
GPU USED      : NO
MODEL LOADED  : NO
TRAINING      : NO
INPUT MUTATED : NO

NEXT:
Use the actual Collection contents before uploading
or rebuilding any equivalent capability.
""")

RAIOS V8.6.2 — COLLECTION INTELLIGENCE HARVEST
CPU ONLY | READ ONLY | NO MODEL | NO TRAINING

COLLECTION FILES: 117

TARGET FILE SEARCH

TARGET: gl-dos-governance-gate.ts
  [EXACT] unified-intelligence-restore-5946c6bc/unified-intelligence/adapters/gl-dos-governance-gate.ts | 1816 bytes

TARGET: execution-contract.json
  [EXACT] unified-intelligence-restore-5946c6bc/unified-intelligence/contracts/execution-contract.json | 1023 bytes

TARGET: audit-engine.ts
  [NOT FOUND]

TARGET: workflow-governance.ts
  [NOT FOUND]

TARGET: 20260815-171701-gemini-cli-GL-001.md
  [NOT FOUND]

TARGET: 20260815-173642-gemini-cli-GL-003.md
  [NOT FOUND]

TARGET: CURRENT-STATE.json
  [NOT FOUND]

TARGET: LOCKS.json
  [NOT FOUND]

HIGH-VALUE CAPABILITY SUPPLY POOL

AGENT_INFRASTRUCTURE: 1 candidate files
  - unified-intelligence-restore-5946c6bc/unified-intelligence/runtime/controlled-runtime-orchestrator.ts

ARCHITECTURE: 7 candidate files
  - E3-SOURCE-TRACE-PACKAGE/CANONICAL__canonical__intelligence__arc

In [11]:
from __future__ import annotations

from pathlib import Path
from collections import Counter
import json
import re
import math
from datetime import datetime, timezone

print("=" * 108)
print("RAIOS V8.6.2 — COLLECTION DEEP CAPABILITY CONVERGENCE")
print("CPU ONLY | CONTENT-LEVEL SEARCH | NO MODEL | NO TRAINING | READ ONLY")
print("=" * 108)

ROOT = Path("/kaggle/input/datasets/greenylife")

COLLECTION = ROOT / "collection-for-understand"
GREENY     = ROOT / "greeny-life"
RAIOS_DS   = ROOT / "raios-cognitive-state"
VERSION_EV = ROOT / "raios-version-evidence"

RAIOS_ROOT = RAIOS_DS / "RAIOS-STATE-LATEST" / "RAIOS"
FACTORY    = RAIOS_ROOT / "raios-cognitive-factory"
STATE      = FACTORY / "state"

WORK = Path("/kaggle/working")

for required in [
    COLLECTION,
    GREENY,
    RAIOS_DS,
    STATE,
]:
    if not required.exists():
        raise RuntimeError(f"Missing persistent input: {required}")


# ============================================================================
# HELPERS
# ============================================================================

TEXT_EXTS = {
    ".ts", ".tsx", ".js", ".jsx",
    ".json", ".jsonl",
    ".md", ".txt",
    ".py", ".yml", ".yaml",
    ".prisma",
}

STOP = {
    "the", "and", "for", "with", "from",
    "this", "that", "true", "false",
    "null", "string", "number", "const",
    "export", "import", "function", "class",
    "interface", "json", "file", "data",
    "canonical", "intelligence", "lib",
    "state", "system",
}


def read_text(path: Path, max_bytes=3_000_000):
    try:
        if path.stat().st_size > max_bytes:
            return ""
        return path.read_text(
            encoding="utf-8",
            errors="ignore"
        )
    except Exception:
        return ""


def read_json(path: Path, default=None):
    try:
        return json.loads(
            path.read_text(
                encoding="utf-8-sig"
            )
        )
    except Exception:
        return default


def tokenize(text: str):
    return [
        x
        for x in re.findall(
            r"[a-z][a-z0-9_-]{2,}",
            text.lower()
        )
        if x not in STOP
    ]


def unique_tokens(text: str):
    return set(tokenize(text))


def compact(text: str, limit=1200):
    text = re.sub(r"\s+", " ", text).strip()
    return text[:limit]


# ============================================================================
# 1. LOAD GREENY BRIDGE
# ============================================================================

bridge = read_json(
    GREENY / "greeny-capabilities.json"
)

if not bridge:
    raise RuntimeError("Unable to load Greeny capability bridge.")

caps = bridge.get("capabilities", [])

if len(caps) != 83:
    raise RuntimeError(
        f"Expected 83 capabilities, found {len(caps)}"
    )

print("\n[PASS] GREENY capability bridge:", len(caps))


# ============================================================================
# 2. BUILD COLLECTION CONTENT CORPUS
# ============================================================================

collection_docs = []

for p in COLLECTION.rglob("*"):

    if not p.is_file():
        continue

    suffix = p.suffix.lower()

    # Many E3 exported sources end in .txt.
    if suffix not in TEXT_EXTS:
        continue

    content = read_text(p)

    if not content:
        continue

    rel = str(p.relative_to(COLLECTION))

    combined = (
        rel
        + "\n"
        + content
    )

    collection_docs.append({
        "path": rel,
        "tokens": unique_tokens(combined),
        "content": content,
        "preview": compact(content),
    })

print(
    "[PASS] COLLECTION textual documents:",
    len(collection_docs)
)


# ============================================================================
# 3. BUILD RAIOS COGNITIVE CORPUS
# ============================================================================

raios_docs = []

for p in STATE.rglob("*"):

    if not p.is_file():
        continue

    if p.suffix.lower() not in {
        ".json", ".jsonl", ".py"
    }:
        continue

    content = read_text(p)

    if not content:
        continue

    rel = str(p.relative_to(STATE))

    raios_docs.append({
        "path": rel,
        "tokens": unique_tokens(
            rel + "\n" + content
        ),
        "content": content,
    })


important_runtime = [
    FACTORY / "runtime" / "durability_transaction.py",
    RAIOS_ROOT / "RECOVERY-CONTRACT.json",
    RAIOS_ROOT / "SOURCE-OF-TRUTH.json",
    RAIOS_ROOT / "DURABLE-MANIFEST.json",
]

for p in important_runtime:

    if not p.exists():
        continue

    content = read_text(p)

    raios_docs.append({
        "path": str(p.relative_to(RAIOS_ROOT)),
        "tokens": unique_tokens(
            p.name + "\n" + content
        ),
        "content": content,
    })

print(
    "[PASS] RAIOS cognitive documents:",
    len(raios_docs)
)


# ============================================================================
# 4. TARGET UNRESOLVED / HIGH-VALUE GREENY CAPABILITIES
# ============================================================================

target_names = {
    "canonical/intelligence/adapters/gl-dos-governance-gate.ts",
    "canonical/intelligence/contracts/execution-contract.json",
    "canonical/intelligence/intelligence/engines/audit-engine.ts",
    "lib/intelligence/workflow-governance.ts",

    ".ai-os/handoffs/20260815-171701-gemini-cli-GL-001.md",
    ".ai-os/handoffs/20260815-173642-gemini-cli-GL-003.md",
    ".ai-os/state/CURRENT-STATE.json",
    ".ai-os/state/LOCKS.json",
}

targets = [
    c for c in caps
    if c.get("path") in target_names
]

print(
    "\n[PASS] High-value unresolved targets:",
    len(targets)
)


# ============================================================================
# 5. ENRICH QUERIES
#
# Important: filenames alone are weak evidence.
# Add domain concepts inferred from each capability type.
# ============================================================================

CONCEPT_EXPANSION = {

    "gl-dos-governance-gate.ts": [
        "governance", "approval", "execution",
        "authorization", "policy", "blocker",
        "human approval", "controlled write",
        "prohibited action",
    ],

    "execution-contract.json": [
        "execution contract", "approval",
        "authorization", "controlled execution",
        "human approval", "action",
        "prohibited", "authority",
    ],

    "audit-engine.ts": [
        "audit", "verification", "validation",
        "evidence", "finding", "trace",
        "integrity", "report",
    ],

    "workflow-governance.ts": [
        "workflow", "governance",
        "orchestration", "approval",
        "controlled runtime", "execution",
        "decision",
    ],

    "GL-001.md": [
        "handoff", "agent", "task",
        "continuation", "context",
        "completed", "next action",
        "resume",
    ],

    "GL-003.md": [
        "handoff", "agent", "task",
        "continuation", "context",
        "completed", "next action",
        "resume",
    ],

    "CURRENT-STATE.json": [
        "current state", "active task",
        "current phase", "resume",
        "next action", "status",
        "project state",
    ],

    "LOCKS.json": [
        "lock", "lease", "owner",
        "agent", "path", "collision",
        "task coordination",
        "concurrency",
    ],
}


def expansion_for(path: str):

    for key, values in CONCEPT_EXPANSION.items():
        if key.lower() in path.lower():
            return values

    return []


# ============================================================================
# 6. SIMPLE IDF WEIGHTS
#
# This is still deterministic CPU search.
# No embedding model.
# ============================================================================

all_docs = collection_docs + raios_docs

document_frequency = Counter()

for doc in all_docs:
    for token in doc["tokens"]:
        document_frequency[token] += 1

N = max(1, len(all_docs))


def idf(token):
    return math.log(
        (N + 1)
        / (document_frequency[token] + 1)
    ) + 1.0


def rank(query_tokens, docs, top_n=12):

    query = set(query_tokens)

    scored = []

    if not query:
        return []

    query_weight = sum(
        idf(t)
        for t in query
    )

    for doc in docs:

        overlap = query & doc["tokens"]

        if not overlap:
            continue

        overlap_weight = sum(
            idf(t)
            for t in overlap
        )

        coverage = (
            overlap_weight
            / max(query_weight, 1e-9)
        )

        specificity = (
            len(overlap)
            / max(
                1,
                math.sqrt(
                    len(doc["tokens"])
                )
            )
        )

        score = (
            0.88 * coverage
            + 0.12 * min(
                specificity,
                1.0
            )
        )

        scored.append({
            "path": doc["path"],
            "score": round(score, 4),
            "coverage": round(
                coverage,
                4
            ),
            "overlap": sorted(overlap),
        })

    scored.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    return scored[:top_n]


# ============================================================================
# 7. DEEP CONVERGENCE
# ============================================================================

results = []

print("\n" + "=" * 108)
print("CONTENT-LEVEL CONVERGENCE")
print("=" * 108)

for cap in targets:

    path = cap.get("path", "")
    symbols = cap.get("symbols", [])
    category = cap.get("category", "OTHER")

    expanded = expansion_for(path)

    query_text = (
        path
        + " "
        + category
        + " "
        + " ".join(symbols)
        + " "
        + " ".join(expanded)
    )

    query_tokens = unique_tokens(
        query_text
    )

    collection_matches = rank(
        query_tokens,
        collection_docs,
        10
    )

    raios_matches = rank(
        query_tokens,
        raios_docs,
        10
    )

    collection_best = (
        collection_matches[0]["score"]
        if collection_matches
        else 0.0
    )

    raios_best = (
        raios_matches[0]["score"]
        if raios_matches
        else 0.0
    )

    # ------------------------------------------------------------
    # Decision is still pre-integration evidence,
    # not automatic promotion.
    # ------------------------------------------------------------

    exact_collection = any(
        Path(x["path"]).name.lower()
        == Path(path).name.lower()
        for x in collection_matches
    )

    if exact_collection and raios_best >= 0.50:

        decision = (
            "RAIOS_HAS_PRIMITIVES_AND_GREENY_HAS_REFERENCE"
        )

    elif exact_collection:

        decision = (
            "IMPORT_BEHAVIOR_FROM_GREENY_REFERENCE"
        )

    elif raios_best >= 0.65:

        decision = (
            "REUSE_EXISTING_RAIOS"
        )

    elif raios_best >= 0.40:

        decision = (
            "COMPOSE_OR_EXTEND_RAIOS"
        )

    elif collection_best >= 0.45:

        decision = (
            "RECOVER_CAPABILITY_FROM_COLLECTION"
        )

    else:

        decision = (
            "REAL_GAP_CANDIDATE"
        )

    result = {
        "greeny_path": path,
        "category": category,
        "symbols": symbols,
        "query_expansion": expanded,

        "decision": decision,

        "collection_best":
            collection_best,

        "raios_best":
            raios_best,

        "collection_matches":
            collection_matches,

        "raios_matches":
            raios_matches,
    }

    results.append(result)

    print("\n" + "-" * 108)
    print("GREENY   :", path)
    print("CATEGORY :", category)
    print("DECISION :", decision)
    print(
        "COLLECTION BEST:",
        collection_best
    )
    print(
        "RAIOS BEST     :",
        raios_best
    )

    print("\nTOP COLLECTION")

    for x in collection_matches[:5]:
        print(
            "  ->",
            x["path"],
            "| score=",
            x["score"],
            "|",
            ",".join(
                x["overlap"][:15]
            )
        )

    print("\nTOP RAIOS")

    for x in raios_matches[:5]:
        print(
            "  ->",
            x["path"],
            "| score=",
            x["score"],
            "|",
            ",".join(
                x["overlap"][:15]
            )
        )


# ============================================================================
# 8. COLLECTION SUPPLY POOL — NOT JUST THE 8 TARGETS
#
# Find particularly valuable actual executable/source assets.
# ============================================================================

source_candidates = []

for doc in collection_docs:

    path = doc["path"].lower()

    source_signal = any([
        path.endswith(".ts"),
        path.endswith(".tsx"),
        path.endswith(".py"),
        "orchestrator" in path,
        "adapter" in path,
        "engine" in path,
        "governance" in path,
        "architecture.manifest" in path,
    ])

    if not source_signal:
        continue

    source_candidates.append(
        doc["path"]
    )

source_candidates = sorted(
    set(source_candidates)
)


# ============================================================================
# 9. VERSION EVIDENCE PRESENCE
# ============================================================================

version_evidence_files = []

if VERSION_EV.exists():
    version_evidence_files = [
        str(p.relative_to(VERSION_EV))
        for p in VERSION_EV.rglob("*")
        if p.is_file()
    ]


# ============================================================================
# 10. OUTPUT
# ============================================================================

decision_counts = Counter(
    r["decision"]
    for r in results
)

real_gaps = [
    r
    for r in results
    if r["decision"]
    == "REAL_GAP_CANDIDATE"
]

report = {
    "schema":
        "raios.v8.6.2.collection-convergence.v1",

    "generated_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "mode": {
        "cpu_only": True,
        "model_loaded": False,
        "training": False,
        "input_mutated": False,
    },

    "collection_documents":
        len(collection_docs),

    "raios_documents":
        len(raios_docs),

    "greenylife_capabilities":
        len(caps),

    "deep_targets":
        len(results),

    "decision_counts":
        dict(decision_counts),

    "results":
        results,

    "real_gap_candidates":
        real_gaps,

    "collection_source_candidates":
        source_candidates,

    "version_evidence_files":
        version_evidence_files,
}


OUT = (
    WORK
    / "RAIOS-V8.6.2-COLLECTION-CONVERGENCE.json"
)

OUT.write_text(
    json.dumps(
        report,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


# ============================================================================
# 11. FINAL SUMMARY
# ============================================================================

print("\n" + "=" * 108)
print("COLLECTION CONVERGENCE SUMMARY")
print("=" * 108)

print(
    "COLLECTION DOCS       :",
    len(collection_docs)
)

print(
    "RAIOS DOCS            :",
    len(raios_docs)
)

print(
    "GREENY CAPABILITIES   :",
    len(caps)
)

print(
    "DEEP TARGETS          :",
    len(results)
)

print()
print("DECISIONS")

for decision, count in sorted(
    decision_counts.items()
):
    print(
        f"{decision:48} "
        f"{count}"
    )

print()
print(
    "REAL GAP CANDIDATES   :",
    len(real_gaps)
)

for r in real_gaps:
    print(
        " -",
        r["greeny_path"]
    )

print()
print(
    "COLLECTION SOURCE ASSETS:",
    len(source_candidates)
)

for p in source_candidates[:30]:
    print(" -", p)

if len(source_candidates) > 30:
    print(
        f" ... +{len(source_candidates)-30} more"
    )

print()
print("OUTPUT:")
print(OUT)

print("""
GPU USED      : NO
MODEL LOADED  : NO
TRAINING      : NO
INPUT MUTATED : NO

NEXT DECISION:
- If REAL GAP CANDIDATES = 0:
  keep Accelerator=None and start V8.6.2 implementation planning.

- If only a small number remain:
  inspect them individually before spending T4.

- T4 is NOT automatically justified merely because
  deterministic matching is uncertain.
""")

RAIOS V8.6.2 — COLLECTION DEEP CAPABILITY CONVERGENCE
CPU ONLY | CONTENT-LEVEL SEARCH | NO MODEL | NO TRAINING | READ ONLY

[PASS] GREENY capability bridge: 83
[PASS] COLLECTION textual documents: 112
[PASS] RAIOS cognitive documents: 122

[PASS] High-value unresolved targets: 8

CONTENT-LEVEL CONVERGENCE

------------------------------------------------------------------------------------------------------------
GREENY   : canonical/intelligence/adapters/gl-dos-governance-gate.ts
CATEGORY : CANONICAL_GOVERNANCE
DECISION : IMPORT_BEHAVIOR_FROM_GREENY_REFERENCE
COLLECTION BEST: 0.5419
RAIOS BEST     : 0.417

TOP COLLECTION
  -> unified-intelligence-restore-5946c6bc/unified-intelligence/adapters/gl-dos-governance-gate.ts | score= 0.5419 | adapters,authorization,controlled,execution,gl-dos-governance-gate,gldosgovernancegate,governance,human
  -> unified-intelligence-restore-5946c6bc/unified-intelligence/runtime/controlled-runtime-orchestrator.ts | score= 0.398 | adapters,controlled,execu

In [12]:
$ErrorActionPreference = "Stop"

$Root = (Get-Location).Path

if (-not (Test-Path ".git")) {
    throw "Run from Greeny-Life-Repair repository root."
}

$Branch = (git branch --show-current).Trim()
$Head   = (git rev-parse HEAD).Trim()

$OutDir = Join-Path $Root "RAIOS-CONTINUITY-EVIDENCE"
$ZipOut = Join-Path $Root "RAIOS-GREENY-CONTINUITY-EVIDENCE.zip"

if (Test-Path $OutDir) {
    Remove-Item $OutDir -Recurse -Force
}

New-Item -ItemType Directory -Force -Path $OutDir | Out-Null

$Candidates = @(
    ".ai-os\CORE-CONTRACT.md",
    ".ai-os\ROUTING.md",
    ".ai-os\state\CURRENT-STATE.json",
    ".ai-os\state\LOCKS.json",
    ".ai-os\state\TASKS.json",
    ".ai-os\handoffs\20260815-171701-gemini-cli-GL-001.md",
    ".ai-os\handoffs\20260815-173642-gemini-cli-GL-003.md"
)

$Manifest = @()

foreach ($Relative in $Candidates) {

    $Source = Join-Path $Root $Relative

    if (-not (Test-Path -LiteralPath $Source -PathType Leaf)) {
        Write-Host "[WARN] Missing: $Relative"
        continue
    }

    $SafeName = $Relative.Replace("\", "__").Replace("/", "__")
    $Destination = Join-Path $OutDir $SafeName

    Copy-Item `
        -LiteralPath $Source `
        -Destination $Destination `
        -Force

    $Item = Get-Item -LiteralPath $Source
    $Hash = (Get-FileHash -LiteralPath $Source -Algorithm SHA256).Hash.ToLowerInvariant()

    $Manifest += [pscustomobject]@{
        sourcePath = $Relative.Replace("\", "/")
        packagedAs = $SafeName
        bytes = $Item.Length
        sha256 = $Hash
        modifiedUtc = $Item.LastWriteTimeUtc.ToString("o")
    }

    Write-Host "[PACK] $Relative"
}

if ($Manifest.Count -lt 4) {
    throw "Too few continuity files found. STOP."
}

$PackageManifest = [ordered]@{
    schema = "raios.greeny.project-continuity-evidence.v1"
    generatedAtUtc = (Get-Date).ToUniversalTime().ToString("o")

    repository = [ordered]@{
        branch = $Branch
        head = $Head
    }

    purpose = @(
        "Teach RAIOS how Greeny-Life coordinates persistent AI developers.",
        "Extract project-level current-state, handoff, task and lock behavior.",
        "Reuse existing Greeny-Life coordination instead of rebuilding it.",
        "Prepare RAIOS to operate continuously inside the VS Code project."
    )

    requiredCapabilityOutcome = @(
        "PROJECT_SESSION_REHYDRATION",
        "CURRENT_TASK_POINTER",
        "AGENT_HANDOFF",
        "TASK_COORDINATION",
        "PATH_LOCK_OR_LEASE",
        "RESUME_POINT",
        "CONFLICT_AVOIDANCE"
    )

    rules = @(
        "DO_NOT_COPY_BLINDLY",
        "REUSE_RAIOS_COGNITIVE_CONTINUITY",
        "ADD_ONLY_PROJECT_COORDINATION_GAPS",
        "NO_DUPLICATE_MEMORY_SYSTEM",
        "NO_DUPLICATE_DURABILITY_SYSTEM",
        "NO_DUPLICATE_ROUTING_SYSTEM"
    )

    files = $Manifest
}

$ManifestPath = Join-Path $OutDir "CONTINUITY-EVIDENCE-MANIFEST.json"

$PackageManifest |
    ConvertTo-Json -Depth 12 |
    Set-Content $ManifestPath -Encoding UTF8

if (Test-Path $ZipOut) {
    Remove-Item $ZipOut -Force
}

Compress-Archive `
    -Path "$OutDir\*" `
    -DestinationPath $ZipOut `
    -CompressionLevel Optimal `
    -Force

$ZipHash = (Get-FileHash $ZipOut -Algorithm SHA256).Hash.ToLowerInvariant()
$ZipSize = (Get-Item $ZipOut).Length

Write-Host ""
Write-Host "============================================================"
Write-Host " RAIOS GREENY CONTINUITY EVIDENCE READY"
Write-Host "============================================================"
Write-Host "Branch : $Branch"
Write-Host "HEAD   : $Head"
Write-Host "Files  : $($Manifest.Count)"
Write-Host "ZIP    : $ZipOut"
Write-Host "Bytes  : $ZipSize"
Write-Host "SHA256 : $ZipHash"
Write-Host ""
Write-Host "GPU USED: 0"
Write-Host "STATUS: READY_FOR_CONTINUITY_ASSIMILATION"
Write-Host "============================================================"

SyntaxError: unterminated string literal (detected at line 42) (447715360.py, line 42)

In [13]:
from pathlib import Path
import json

print("=" * 88)
print("RAIOS CONTINUITY EVIDENCE DISCOVERY")
print("CPU ONLY | READ ONLY")
print("=" * 88)

root = Path("/kaggle/input")

manifests = list(
    root.rglob("CONTINUITY-EVIDENCE-MANIFEST.json")
)

if not manifests:
    print("MANIFEST NOT FOUND")
else:
    for manifest in manifests:
        print("\nFOUND MANIFEST:")
        print(manifest)

        try:
            data = json.loads(
                manifest.read_text(
                    encoding="utf-8-sig"
                )
            )

            print("\nREPOSITORY:")
            print(
                "  branch:",
                data.get("repository", {}).get("branch")
            )
            print(
                "  head  :",
                data.get("repository", {}).get("head")
            )

            files = data.get("files", [])

            print("\nFILES:", len(files))

            for item in files:
                print(
                    " -",
                    item.get("sourcePath"),
                    "|",
                    item.get("bytes"),
                    "bytes"
                )

        except Exception as exc:
            print("MANIFEST READ ERROR:", exc)

print("\n" + "=" * 88)
print("CONTINUITY FILE SEARCH")
print("=" * 88)

keywords = [
    "core-contract",
    "routing",
    "current-state",
    "locks",
    "tasks",
    "handoff",
]

found = []

for p in root.rglob("*"):
    if not p.is_file():
        continue

    name = p.name.lower()

    if any(k in name for k in keywords):
        found.append(p)

for p in sorted(found):
    print(p)

print()
print("GPU USED: NO")
print("MODEL LOADED: NO")
print("TRAINING: NO")
print("STATE MUTATED: NO")

RAIOS CONTINUITY EVIDENCE DISCOVERY
CPU ONLY | READ ONLY

FOUND MANIFEST:
/kaggle/input/datasets/greenylife/raios-greeny-continuity-evidence/CONTINUITY-EVIDENCE-MANIFEST.json

REPOSITORY:
  branch: codex-clean
  head  : fb130e232246a00b531835f529f25e4edfae2854

FILES: 7
 - .ai-os/CORE-CONTRACT.md | 666 bytes
 - .ai-os/ROUTING.md | 429 bytes
 - .ai-os/state/CURRENT-STATE.json | 1561 bytes
 - .ai-os/state/LOCKS.json | 2746 bytes
 - .ai-os/state/TASKS.json | 5864 bytes
 - .ai-os/handoffs/20260815-171701-gemini-cli-GL-001.md | 551 bytes
 - .ai-os/handoffs/20260815-173642-gemini-cli-GL-003.md | 453 bytes

CONTINUITY FILE SEARCH
/kaggle/input/datasets/greenylife/raios-greeny-continuity-evidence/.ai-os__CORE-CONTRACT.md
/kaggle/input/datasets/greenylife/raios-greeny-continuity-evidence/.ai-os__ROUTING.md
/kaggle/input/datasets/greenylife/raios-greeny-continuity-evidence/.ai-os__handoffs__20260815-171701-gemini-cli-GL-001.md
/kaggle/input/datasets/greenylife/raios-greeny-continuity-evidence/.a

In [14]:
from __future__ import annotations

from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
import json
import re

print("=" * 112)
print("RAIOS V8.6.2 — PROJECT CONTINUITY BEHAVIOR ASSIMILATION")
print("CPU ONLY | READ ONLY | NO MODEL | NO TRAINING | NO STATE MUTATION")
print("=" * 112)

ROOT = Path("/kaggle/input/datasets/greenylife")

CONTINUITY_DS = ROOT / "raios-greeny-continuity-evidence"
RAIOS_DS      = ROOT / "raios-cognitive-state"

RAIOS_ROOT = RAIOS_DS / "RAIOS-STATE-LATEST" / "RAIOS"
FACTORY    = RAIOS_ROOT / "raios-cognitive-factory"
STATE      = FACTORY / "state"

WORK = Path("/kaggle/working")

for required in [
    CONTINUITY_DS,
    RAIOS_DS,
    RAIOS_ROOT,
    FACTORY,
    STATE,
]:
    if not required.exists():
        raise RuntimeError(f"Missing required input: {required}")

# =============================================================================
# 1. LOAD CONTINUITY MANIFEST
# =============================================================================

manifest_path = CONTINUITY_DS / "CONTINUITY-EVIDENCE-MANIFEST.json"

if not manifest_path.exists():
    raise RuntimeError("Continuity manifest missing.")

manifest = json.loads(
    manifest_path.read_text(
        encoding="utf-8-sig"
    )
)

print("\n[PASS] Continuity package")
print("Branch :", manifest["repository"]["branch"])
print("HEAD   :", manifest["repository"]["head"])
print("Files  :", len(manifest["files"]))

# =============================================================================
# 2. HELPERS
# =============================================================================

def read_text(path: Path) -> str:
    return path.read_text(
        encoding="utf-8",
        errors="ignore"
    )

def read_json(path: Path, default=None):
    try:
        return json.loads(
            path.read_text(
                encoding="utf-8-sig"
            )
        )
    except Exception:
        return default

def norm(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()

def tokens(text: str) -> set[str]:
    stop = {
        "the", "and", "for", "with", "from",
        "this", "that", "true", "false",
        "null", "none", "json", "state",
        "file", "system", "current", "active",
    }

    return {
        x
        for x in re.findall(
            r"[a-z][a-z0-9_-]{2,}",
            text.lower()
        )
        if x not in stop
    }

# =============================================================================
# 3. MAP PACKAGED FILES BACK TO THEIR ORIGINAL PATHS
# =============================================================================

source_files = {}

for item in manifest["files"]:

    source = item["sourcePath"]
    packaged = item["packagedAs"]

    path = CONTINUITY_DS / packaged

    if not path.exists():
        raise RuntimeError(
            f"Packaged evidence missing: {packaged}"
        )

    source_files[source] = path

print("\n[PASS] Packaged evidence resolved:", len(source_files))

# =============================================================================
# 4. PARSE GREENY CONTINUITY LAYER
# =============================================================================

core_contract = read_text(
    source_files[".ai-os/CORE-CONTRACT.md"]
)

routing_md = read_text(
    source_files[".ai-os/ROUTING.md"]
)

current_state = read_json(
    source_files[".ai-os/state/CURRENT-STATE.json"],
    {}
)

locks = read_json(
    source_files[".ai-os/state/LOCKS.json"],
    {}
)

tasks = read_json(
    source_files[".ai-os/state/TASKS.json"],
    {}
)

handoffs = []

for source, path in source_files.items():
    if "/handoffs/" in source:
        handoffs.append({
            "source": source,
            "content": read_text(path),
        })

# =============================================================================
# 5. EXTRACT BEHAVIORAL PRIMITIVES
# =============================================================================

behavior = {
    "agent_contract": {
        "evidence": norm(core_contract),
        "signals": sorted(tokens(core_contract)),
    },

    "routing": {
        "evidence": norm(routing_md),
        "signals": sorted(tokens(routing_md)),
    },

    "current_state": {
        "keys": sorted(current_state.keys()),
        "record": current_state,
    },

    "locks": {
        "keys": sorted(locks.keys()) if isinstance(locks, dict) else [],
        "record": locks,
    },

    "tasks": {
        "keys": sorted(tasks.keys()) if isinstance(tasks, dict) else [],
        "record": tasks,
    },

    "handoffs": handoffs,
}

# =============================================================================
# 6. FUNCTIONAL CAPABILITY DETECTION
# =============================================================================

capability_rules = {
    "SESSION_REHYDRATION": [
        "resume",
        "continuation",
        "current-state",
        "context",
        "handoff",
    ],

    "CURRENT_TASK_POINTER": [
        "task",
        "current",
        "next",
        "status",
        "owner",
    ],

    "AGENT_HANDOFF": [
        "handoff",
        "completed",
        "next",
        "context",
        "agent",
    ],

    "TASK_REGISTRY": [
        "task",
        "status",
        "owner",
        "dependencies",
        "scope",
    ],

    "PATH_LOCKING": [
        "lock",
        "path",
        "owner",
        "lease",
        "agent",
    ],

    "COLLISION_AVOIDANCE": [
        "lock",
        "collision",
        "conflict",
        "owner",
        "task",
    ],

    "AGENT_IDENTITY": [
        "agent",
        "owner",
        "requestedby",
        "identity",
    ],

    "ROUTING_COORDINATION": [
        "routing",
        "route",
        "task",
        "agent",
        "handoff",
    ],

    "AUDITABLE_PROJECT_STATE": [
        "state",
        "status",
        "task",
        "owner",
        "timestamp",
        "history",
    ],
}

all_greeny_text = "\n".join([
    core_contract,
    routing_md,
    json.dumps(current_state, ensure_ascii=False),
    json.dumps(locks, ensure_ascii=False),
    json.dumps(tasks, ensure_ascii=False),
    *[x["content"] for x in handoffs],
])

green_tokens = tokens(all_greeny_text)

green_capabilities = {}

for capability, signals in capability_rules.items():

    present = [
        s
        for s in signals
        if s.lower() in green_tokens
        or s.lower() in all_greeny_text.lower()
    ]

    green_capabilities[capability] = {
        "present": len(present) >= max(1, len(signals) // 3),
        "signals_found": present,
        "signals_total": signals,
    }

# =============================================================================
# 7. BUILD RAIOS PRIMITIVE MAP
# =============================================================================

raios_control_paths = {
    "REMOTE_CURRENT":
        RAIOS_DS / "CURRENT.json",

    "RECOVERY_CONTRACT":
        RAIOS_ROOT / "RECOVERY-CONTRACT.json",

    "SOURCE_OF_TRUTH":
        RAIOS_ROOT / "SOURCE-OF-TRUTH.json",

    "DURABLE_MANIFEST":
        RAIOS_ROOT / "DURABLE-MANIFEST.json",

    "LATEST_CHECKPOINT":
        STATE / "checkpoints" / "LATEST.json",

    "DURABILITY_LATEST":
        STATE / "checkpoints" / "DURABILITY-LATEST.json",

    "ROUTING_POLICY":
        STATE / "routing-policies" / "ACTIVE.json",

    "LEARNING_LEDGER":
        STATE / "manifests" / "learning-ledger.json",

    "COGNITIVE_DOCTRINE":
        STATE / "doctrine" / "RAIOS-COGNITIVE-DOCTRINE.json",

    "SEMANTIC_STATE_CONTRACT":
        STATE / "doctrine" / "SEMANTIC-STATE-CONTRACT.json",
}

raios_control = {}

raios_text_parts = []

for name, path in raios_control_paths.items():

    if not path.exists():
        raios_control[name] = None
        continue

    data = read_json(path, {})

    raios_control[name] = data

    raios_text_parts.append(
        name + " " +
        json.dumps(
            data,
            ensure_ascii=False
        )
    )

# Include journal vocabulary but not mutate it.

journal_path = STATE / "journal" / "events.jsonl"

journal_lines = []

if journal_path.exists():
    journal_lines = [
        line
        for line in read_text(journal_path).splitlines()
        if line.strip()
    ]

raios_text_parts.extend(journal_lines)

raios_text = "\n".join(raios_text_parts)
raios_tokens = tokens(raios_text)

# =============================================================================
# 8. RAIOS CAPABILITY COVERAGE
# =============================================================================

coverage = {}

for capability, signals in capability_rules.items():

    found = [
        s
        for s in signals
        if s.lower() in raios_tokens
        or s.lower() in raios_text.lower()
    ]

    ratio = (
        len(found) / len(signals)
        if signals
        else 0
    )

    if ratio >= 0.65:
        classification = "EXISTING"

    elif ratio >= 0.35:
        classification = "PARTIAL"

    elif ratio > 0:
        classification = "TRACE_ONLY"

    else:
        classification = "ABSENT"

    coverage[capability] = {
        "classification": classification,
        "coverage": round(ratio, 4),
        "signals_found": found,
        "signals_total": signals,
    }

# =============================================================================
# 9. CONVERGENCE DECISION
# =============================================================================

decisions = {}

for capability in capability_rules:

    greeny_has = green_capabilities[capability]["present"]
    raios_class = coverage[capability]["classification"]

    if not greeny_has:
        decision = "NO_GREENY_CAPABILITY"

    elif raios_class == "EXISTING":
        decision = "REUSE_RAIOS"

    elif raios_class == "PARTIAL":
        decision = "EXTEND_RAIOS_WITH_GREENY_BEHAVIOR"

    elif raios_class == "TRACE_ONLY":
        decision = "ADD_THIN_PROJECT_COORDINATION_ADAPTER"

    else:
        decision = "REAL_PROJECT_LEVEL_GAP"

    decisions[capability] = decision

# =============================================================================
# 10. CHECK FOR DUPLICATE ARCHITECTURE RISK
# =============================================================================

duplicate_risk = []

for capability, decision in decisions.items():

    if decision == "REUSE_RAIOS":
        duplicate_risk.append({
            "capability": capability,
            "risk": "HIGH_IF_REIMPLEMENTED",
            "rule": "Reuse existing RAIOS primitive; do not build parallel subsystem."
        })

    elif decision == "EXTEND_RAIOS_WITH_GREENY_BEHAVIOR":
        duplicate_risk.append({
            "capability": capability,
            "risk": "MEDIUM_IF_NEW_SUBSYSTEM_CREATED",
            "rule": "Extend current primitive only."
        })

# =============================================================================
# 11. DERIVE MINIMAL PROJECT COORDINATION LAYER
# =============================================================================

minimal_adapter = []

for capability, decision in decisions.items():

    if decision in {
        "ADD_THIN_PROJECT_COORDINATION_ADAPTER",
        "REAL_PROJECT_LEVEL_GAP",
        "EXTEND_RAIOS_WITH_GREENY_BEHAVIOR",
    }:
        minimal_adapter.append(capability)

# =============================================================================
# 12. OUTPUT
# =============================================================================

report = {
    "schema":
        "raios.v8.6.2.project-continuity-assimilation.v1",

    "generated_at_utc":
        datetime.now(timezone.utc).isoformat(),

    "greenylife_repository": manifest["repository"],

    "greenylife_capabilities":
        green_capabilities,

    "raios_coverage":
        coverage,

    "convergence_decisions":
        decisions,

    "minimal_project_coordination_adapter":
        minimal_adapter,

    "duplicate_risk":
        duplicate_risk,

    "raios_control_sources": {
        name: str(path)
        for name, path in raios_control_paths.items()
    },

    "journal_events":
        len(journal_lines),

    "mode": {
        "cpu_only": True,
        "model_loaded": False,
        "training": False,
        "state_mutated": False,
    }
}

out = (
    WORK
    / "RAIOS-V8.6.2-PROJECT-CONTINUITY-ASSIMILATION.json"
)

out.write_text(
    json.dumps(
        report,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

# =============================================================================
# 13. SUMMARY
# =============================================================================

decision_counts = Counter(decisions.values())

print("\n" + "=" * 112)
print("PROJECT CONTINUITY ASSIMILATION SUMMARY")
print("=" * 112)

print("\nGREENY PROJECT CAPABILITIES")

for capability, info in green_capabilities.items():
    print(
        f"{capability:34} "
        f"{'YES' if info['present'] else 'NO'} "
        f"| {','.join(info['signals_found'])}"
    )

print("\nRAIOS COVERAGE")

for capability, info in coverage.items():
    print(
        f"{capability:34} "
        f"{info['classification']:12} "
        f"| coverage={info['coverage']}"
    )

print("\nCONVERGENCE DECISIONS")

for capability, decision in decisions.items():
    print(
        f"{capability:34} "
        f"=> {decision}"
    )

print("\nDECISION COUNTS")

for decision, count in sorted(decision_counts.items()):
    print(
        f"{decision:45} {count}"
    )

print("\nMINIMAL PROJECT COORDINATION ADAPTER")

if minimal_adapter:
    for capability in minimal_adapter:
        print(" -", capability)
else:
    print(" NONE")

print("\nDUPLICATE-RISK GUARDS")

for item in duplicate_risk:
    print(
        " -",
        item["capability"],
        "=>",
        item["risk"]
    )

print("\nOUTPUT:")
print(out)

print("""
GPU USED      : NO
MODEL LOADED  : NO
TRAINING      : NO
STATE MUTATED : NO

NEXT:
Use this result to define the smallest V8.6.2 code change.
Do not create a second memory, routing, checkpoint,
durability, or learning subsystem.
""")

RAIOS V8.6.2 — PROJECT CONTINUITY BEHAVIOR ASSIMILATION
CPU ONLY | READ ONLY | NO MODEL | NO TRAINING | NO STATE MUTATION

[PASS] Continuity package
Branch : codex-clean
HEAD   : fb130e232246a00b531835f529f25e4edfae2854
Files  : 7

[PASS] Packaged evidence resolved: 7

PROJECT CONTINUITY ASSIMILATION SUMMARY

GREENY PROJECT CAPABILITIES
SESSION_REHYDRATION                YES | current-state,handoff
CURRENT_TASK_POINTER               YES | task,current,next,status
AGENT_HANDOFF                      YES | handoff,completed,next,agent
TASK_REGISTRY                      YES | task,status,dependencies,scope
PATH_LOCKING                       YES | lock,path,lease,agent
COLLISION_AVOIDANCE                YES | lock,task
AGENT_IDENTITY                     YES | agent
ROUTING_COORDINATION               YES | routing,route,task,agent,handoff
AUDITABLE_PROJECT_STATE            YES | state,status,task,history

RAIOS COVERAGE
SESSION_REHYDRATION                PARTIAL      | coverage=0.4
CURRENT_T

In [16]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime, timezone
import json
import re

print("=" * 108)
print("RAIOS V8.6.2 — MINIMAL PATCH PLANNER")
print("CPU ONLY | NO MODEL | NO TRAINING | NO STATE MUTATION")
print("=" * 108)

ROOT = Path("/kaggle/input/datasets/greenylife")

RAIOS_DS = ROOT / "raios-cognitive-state"
RAIOS_ROOT = RAIOS_DS / "RAIOS-STATE-LATEST" / "RAIOS"
FACTORY = RAIOS_ROOT / "raios-cognitive-factory"
STATE = FACTORY / "state"

WORK = Path("/kaggle/working")

for p in [RAIOS_DS, RAIOS_ROOT, FACTORY, STATE]:
    if not p.exists():
        raise RuntimeError(f"Missing required path: {p}")


def read_text(path: Path) -> str:
    return path.read_text(encoding="utf-8", errors="ignore")


def read_json(path: Path, default=None):
    try:
        return json.loads(path.read_text(encoding="utf-8-sig"))
    except Exception:
        return default


def contains_any(text: str, patterns: list[str]) -> bool:
    low = text.lower()
    return any(p.lower() in low for p in patterns)


# ============================================================================
# 1. FIND EXISTING RAIOS IMPLEMENTATION BOUNDARIES
# ============================================================================

candidates = []

for p in FACTORY.rglob("*"):
    if not p.is_file():
        continue

    if p.suffix.lower() not in {".py", ".json", ".jsonl", ".md"}:
        continue

    text = read_text(p)

    score = 0
    reasons = []

    checks = {
        "routing": ["routing", "route", "fallback"],
        "state": ["current", "checkpoint", "journal", "state"],
        "learning": ["experience", "skill", "promotion", "training"],
        "durability": ["durability", "transaction", "remote_ack", "remote acknowledged"],
        "confidence": ["confidence"],
        "agent": ["agent", "task", "handoff", "lock", "lease"],
    }

    for label, terms in checks.items():
        if contains_any(text, terms):
            score += 1
            reasons.append(label)

    if score:
        candidates.append({
            "path": str(p.relative_to(RAIOS_ROOT)),
            "score": score,
            "reasons": reasons,
            "bytes": p.stat().st_size,
        })

candidates.sort(key=lambda x: (-x["score"], x["bytes"]))


# ============================================================================
# 2. KNOWN REQUIRED CHANGES
# ============================================================================

bad_skill = (
    "683ca22e97a2cb919ab078321076bd06"
    "b146b1bf69e937a2062d68755e993b65"
)

required_changes = [
    {
        "id": "CONFIDENCE_BOUNDARY",
        "goal": "Enforce runtime confidence domain [0,1] before aggregation, storage, replay, and promotion.",
        "must_not": [
            "silently accept 70 as 0.70",
            "repair historical evidence in place",
            "change semantic contract",
        ],
    },
    {
        "id": "INVALID_SKILL_QUARANTINE",
        "goal": "Prevent the known invalidated repository-analysis skill from being selected or promoted until replay validation passes.",
        "skill_hash": bad_skill,
        "effective_status": "QUARANTINED_PENDING_REVALIDATION",
    },
    {
        "id": "PROJECT_COORDINATION_ADAPTER",
        "goal": "Extend existing RAIOS state with thin project-level coordination only.",
        "capabilities": [
            "SESSION_REHYDRATION",
            "CURRENT_TASK_POINTER",
            "AGENT_HANDOFF",
            "TASK_REGISTRY",
            "PATH_LOCKING",
            "COLLISION_AVOIDANCE",
            "AGENT_IDENTITY",
        ],
        "reuse_existing": [
            "ROUTING_COORDINATION",
            "AUDITABLE_PROJECT_STATE",
            "CHECKPOINTS",
            "JOURNAL",
            "DURABILITY",
            "LEARNING",
        ],
    },
]


# ============================================================================
# 3. PROPOSE MINIMAL FILE SET
# ============================================================================

top = candidates[:30]

likely_runtime = [
    x for x in top
    if any(r in x["reasons"] for r in ["routing", "state", "learning", "durability", "confidence"])
]

selected = []

seen = set()

for item in likely_runtime:
    if item["path"] in seen:
        continue

    selected.append(item)
    seen.add(item["path"])

    if len(selected) >= 8:
        break


# ============================================================================
# 4. PATCH PLAN
# ============================================================================

plan = {
    "schema": "raios.v8.6.2.minimal-patch-plan.v1",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),

    "principles": [
        "No new memory subsystem.",
        "No new routing subsystem.",
        "No new durability subsystem.",
        "No new checkpoint subsystem.",
        "No duplicate audit/state fabric.",
        "Extend existing primitives only.",
        "Fail closed on invalid confidence.",
        "Quarantined skills cannot route or promote.",
    ],

    "required_changes": required_changes,

    "candidate_existing_files": candidates[:50],

    "preferred_minimal_file_set": selected,

    "implementation_order": [
        "1. confidence boundary enforcement",
        "2. invalid skill quarantine enforcement",
        "3. thin project coordination adapter",
        "4. focused tests",
        "5. replay validation",
        "6. durability transaction only after tests pass",
    ],

    "gpu_required": False,
    "state_mutated": False,
}

out = WORK / "RAIOS-V8.6.2-MINIMAL-PATCH-PLAN.json"

out.write_text(
    json.dumps(plan, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("\n" + "=" * 108)
print("MINIMAL PATCH PLAN SUMMARY")
print("=" * 108)

print("\nPREFERRED EXISTING FILES TO EXTEND")

for item in selected:
    print(
        " -",
        item["path"],
        "| score=",
        item["score"],
        "|",
        ",".join(item["reasons"])
    )

print("\nREQUIRED CHANGES")
for change in required_changes:
    print(" -", change["id"], "=>", change["goal"])

print("\nOUTPUT:")
print(out)

print("""
GPU USED      : NO
MODEL LOADED  : NO
TRAINING      : NO
STATE MUTATED : NO

NEXT:
Use the preferred existing file set to generate the actual V8.6.2 code patch.
""")

RAIOS V8.6.2 — MINIMAL PATCH PLANNER
CPU ONLY | NO MODEL | NO TRAINING | NO STATE MUTATION

MINIMAL PATCH PLAN SUMMARY

PREFERRED EXISTING FILES TO EXTEND
 - raios-cognitive-factory/state/doctrine/RAIOS-COGNITIVE-DOCTRINE.json | score= 6 | routing,state,learning,durability,confidence,agent
 - raios-cognitive-factory/state/doctrine/SEMANTIC-STATE-CONTRACT.json | score= 5 | routing,state,learning,confidence,agent
 - raios-cognitive-factory/state/journal/events.jsonl | score= 5 | routing,state,learning,confidence,agent
 - raios-cognitive-factory/state/failures/5af917a4c7f31227d076745700ea99303780b7aa8aaad725d8a620e4e8a78238.json | score= 4 | routing,state,learning,confidence
 - raios-cognitive-factory/state/training-candidates/ea7446b1dbc07ee4fb6d8994a8129ad85aea9375378c745bf450f3d3d1247af3.json | score= 4 | routing,state,learning,confidence
 - raios-cognitive-factory/state/checkpoints/20260817T043037Z-2f4c15ca-f195f8fc3c11.json | score= 4 | state,learning,confidence,agent
 - raios-cognit

In [17]:
from __future__ import annotations

from pathlib import Path
from collections import defaultdict
import json
import re

print("=" * 112)
print("RAIOS V8.6.2 — EXECUTABLE RUNTIME BOUNDARY RESOLVER")
print("CPU ONLY | READ ONLY | NO MODEL | NO TRAINING | NO STATE MUTATION")
print("=" * 112)

INPUT_ROOT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

# -----------------------------------------------------------------------------
# 1. What we are actually looking for
# -----------------------------------------------------------------------------

CAPABILITIES = {
    "CONFIDENCE_BOUNDARY": [
        "confidence",
        "normalize",
        "validation",
        "semantic",
        "promotion",
        "replay",
    ],

    "SKILL_ROUTING": [
        "skill",
        "routing",
        "route",
        "compiled_entries",
        "fallback",
        "promotion",
    ],

    "LEARNING_RUNTIME": [
        "experience",
        "learning",
        "failure",
        "training_candidate",
        "skill",
        "journal",
    ],

    "CHECKPOINT_RUNTIME": [
        "checkpoint",
        "restore",
        "resume",
        "state",
    ],

    "DURABILITY_RUNTIME": [
        "durability",
        "transaction",
        "remote",
        "ack",
        "commit",
        "precommit",
    ],

    "PROJECT_COORDINATION": [
        "task",
        "handoff",
        "lock",
        "lease",
        "agent",
        "resume",
    ],
}

SOURCE_EXTENSIONS = {
    ".py",
    ".ts",
    ".tsx",
    ".js",
    ".mjs",
    ".cjs",
}

REFERENCE_EXTENSIONS = {
    ".json",
    ".jsonl",
    ".md",
    ".txt",
    ".yaml",
    ".yml",
}

# -----------------------------------------------------------------------------
# 2. Helpers
# -----------------------------------------------------------------------------

def read_text(path: Path, max_bytes=5_000_000):
    try:
        if path.stat().st_size > max_bytes:
            return ""
        return path.read_text(
            encoding="utf-8",
            errors="ignore"
        )
    except Exception:
        return ""


def classify_location(path: Path):
    s = str(path).lower().replace("\\", "/")

    # Cognitive state is evidence/runtime state, not implementation source.
    if "/state/" in s:
        return "STATE_ARTIFACT"

    # Collection restore trees are useful references but not current RAIOS truth.
    if "collection-for-understand" in s:
        return "HISTORICAL_REFERENCE"

    # Version-evidence is forensic evidence.
    if "raios-version-evidence" in s:
        return "FORENSIC_EVIDENCE"

    # Source inside certified RAIOS factory but outside /state/.
    if "raios-cognitive-factory" in s:
        return "RAIOS_RUNTIME_SOURCE"

    return "OTHER_SOURCE"


def feature_hits(text: str):
    low = text.lower()

    result = {}

    for capability, terms in CAPABILITIES.items():
        found = sorted({
            term
            for term in terms
            if term.lower() in low
        })

        if found:
            result[capability] = found

    return result


# -----------------------------------------------------------------------------
# 3. Search all mounted Kaggle inputs
# -----------------------------------------------------------------------------

records = []

for p in INPUT_ROOT.rglob("*"):

    if not p.is_file():
        continue

    suffix = p.suffix.lower()

    if suffix not in SOURCE_EXTENSIONS | REFERENCE_EXTENSIONS:
        continue

    text = read_text(p)

    if not text:
        continue

    hits = feature_hits(
        p.name + "\n" + text
    )

    if not hits:
        continue

    classification = classify_location(p)

    score = sum(
        len(v)
        for v in hits.values()
    )

    records.append({
        "path": str(p),
        "relative": str(p.relative_to(INPUT_ROOT)),
        "classification": classification,
        "extension": suffix,
        "bytes": p.stat().st_size,
        "score": score,
        "hits": hits,
    })

records.sort(
    key=lambda x: (
        x["classification"] != "RAIOS_RUNTIME_SOURCE",
        -x["score"],
        x["bytes"],
    )
)

# -----------------------------------------------------------------------------
# 4. Separate actual executable source from state/history
# -----------------------------------------------------------------------------

runtime_source = [
    r for r in records
    if r["classification"] == "RAIOS_RUNTIME_SOURCE"
    and r["extension"] in SOURCE_EXTENSIONS
]

other_source = [
    r for r in records
    if r["classification"] == "OTHER_SOURCE"
    and r["extension"] in SOURCE_EXTENSIONS
]

historical_source = [
    r for r in records
    if r["classification"] == "HISTORICAL_REFERENCE"
    and r["extension"] in SOURCE_EXTENSIONS
]

state_artifacts = [
    r for r in records
    if r["classification"] == "STATE_ARTIFACT"
]

# -----------------------------------------------------------------------------
# 5. Print actual RAIOS runtime candidates
# -----------------------------------------------------------------------------

print("\n" + "=" * 112)
print("CURRENT RAIOS EXECUTABLE SOURCE")
print("=" * 112)

if not runtime_source:
    print("NONE FOUND IN MOUNTED INPUTS")
else:
    for r in runtime_source[:100]:

        print("\nFILE :", r["relative"])
        print("SCORE:", r["score"])

        for capability, hits in r["hits"].items():
            print(
                "  ",
                capability,
                "=>",
                ", ".join(hits)
            )

# -----------------------------------------------------------------------------
# 6. Other executable code — useful but not automatically authoritative
# -----------------------------------------------------------------------------

print("\n" + "=" * 112)
print("OTHER EXECUTABLE SOURCE")
print("=" * 112)

for r in other_source[:50]:

    print("\nFILE :", r["relative"])
    print("SCORE:", r["score"])

    for capability, hits in r["hits"].items():
        print(
            "  ",
            capability,
            "=>",
            ", ".join(hits)
        )

# -----------------------------------------------------------------------------
# 7. Historical implementation supply pool
# -----------------------------------------------------------------------------

print("\n" + "=" * 112)
print("HISTORICAL / GREENY EXECUTABLE REFERENCES")
print("=" * 112)

for r in historical_source[:50]:

    print("\nFILE :", r["relative"])
    print("SCORE:", r["score"])

    for capability, hits in r["hits"].items():
        print(
            "  ",
            capability,
            "=>",
            ", ".join(hits)
        )

# -----------------------------------------------------------------------------
# 8. Explicitly identify state files we must NOT patch as implementation
# -----------------------------------------------------------------------------

print("\n" + "=" * 112)
print("STATE ARTIFACTS — DO NOT PATCH AS IMPLEMENTATION")
print("=" * 112)

for r in state_artifacts[:30]:
    print(" -", r["relative"])

if len(state_artifacts) > 30:
    print(
        f" ... +{len(state_artifacts)-30} more"
    )

# -----------------------------------------------------------------------------
# 9. Check certified source-of-truth identity
# -----------------------------------------------------------------------------

source_truth_candidates = list(
    INPUT_ROOT.rglob("SOURCE-OF-TRUTH.json")
)

source_truth = None

for p in source_truth_candidates:
    try:
        obj = json.loads(
            p.read_text(
                encoding="utf-8-sig"
            )
        )

        if "repository" in obj and "sha" in obj:
            source_truth = obj
            break

    except Exception:
        pass

print("\n" + "=" * 112)
print("CERTIFIED SOURCE OF TRUTH")
print("=" * 112)

if source_truth:
    print(
        "Repository:",
        source_truth.get("repository")
    )
    print(
        "Branch    :",
        source_truth.get("branch")
    )
    print(
        "SHA       :",
        source_truth.get("sha")
    )
else:
    print("SOURCE-OF-TRUTH.json could not be resolved.")

# -----------------------------------------------------------------------------
# 10. Determine whether actual implementation source is available
# -----------------------------------------------------------------------------

critical_runtime = {
    "confidence": [],
    "routing": [],
    "learning": [],
    "durability": [],
    "coordination": [],
}

for r in runtime_source:

    h = r["hits"]

    if "CONFIDENCE_BOUNDARY" in h:
        critical_runtime["confidence"].append(r["relative"])

    if "SKILL_ROUTING" in h:
        critical_runtime["routing"].append(r["relative"])

    if "LEARNING_RUNTIME" in h:
        critical_runtime["learning"].append(r["relative"])

    if "DURABILITY_RUNTIME" in h:
        critical_runtime["durability"].append(r["relative"])

    if "PROJECT_COORDINATION" in h:
        critical_runtime["coordination"].append(r["relative"])


# -----------------------------------------------------------------------------
# 11. Machine report
# -----------------------------------------------------------------------------

report = {
    "schema":
        "raios.v8.6.2.runtime-boundary-resolution.v1",

    "source_of_truth":
        source_truth,

    "current_raios_runtime_source":
        runtime_source,

    "other_current_source":
        other_source,

    "historical_executable_references":
        historical_source,

    "state_artifacts_not_for_direct_patch":
        state_artifacts,

    "critical_runtime_map":
        critical_runtime,

    "decision": (
        "SOURCE_PATCH_READY"
        if runtime_source
        else "CERTIFIED_SOURCE_CODE_REQUIRED"
    ),

    "rules": [
        "Do not patch checkpoints as implementation.",
        "Do not patch historical failure records as implementation.",
        "Do not patch journal history in place.",
        "Do not repair the invalid skill record by rewriting historical evidence.",
        "Implement runtime guards in source, then create new state through normal runtime/durability paths.",
    ],
}

OUT = WORK / "RAIOS-V8.6.2-RUNTIME-BOUNDARY.json"

OUT.write_text(
    json.dumps(
        report,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

# -----------------------------------------------------------------------------
# 12. Final summary
# -----------------------------------------------------------------------------

print("\n" + "=" * 112)
print("RUNTIME BOUNDARY SUMMARY")
print("=" * 112)

print(
    "CURRENT RAIOS SOURCE FILES :",
    len(runtime_source)
)

print(
    "OTHER SOURCE FILES         :",
    len(other_source)
)

print(
    "HISTORICAL SOURCE FILES    :",
    len(historical_source)
)

print(
    "STATE ARTIFACT MATCHES     :",
    len(state_artifacts)
)

print("\nCRITICAL RUNTIME MAP")

for capability, paths in critical_runtime.items():

    print(
        f"{capability:16}: "
        f"{len(paths)}"
    )

    for p in paths[:10]:
        print("   -", p)

print("\nDECISION:")

if runtime_source:
    print("SOURCE_PATCH_READY")
    print(
        "The next step may generate the real V8.6.2 implementation patch."
    )
else:
    print("CERTIFIED_SOURCE_CODE_REQUIRED")
    print(
        "Mounted Kaggle state is not enough to safely implement V8.6.2."
    )
    print(
        "Use the certified Git repository at the exact SOURCE-OF-TRUTH SHA."
    )

print("\nOUTPUT:")
print(OUT)

print("""
GPU USED      : NO
MODEL LOADED  : NO
TRAINING      : NO
STATE MUTATED : NO
""")

RAIOS V8.6.2 — EXECUTABLE RUNTIME BOUNDARY RESOLVER
CPU ONLY | READ ONLY | NO MODEL | NO TRAINING | NO STATE MUTATION

CURRENT RAIOS EXECUTABLE SOURCE

FILE : datasets/greenylife/raios-cognitive-state/RAIOS-STATE-LATEST/RAIOS/raios-cognitive-factory/runtime/durability_transaction.py
SCORE: 13
   CONFIDENCE_BOUNDARY => promotion, semantic
   SKILL_ROUTING => promotion, skill
   LEARNING_RUNTIME => experience, skill, training_candidate
   CHECKPOINT_RUNTIME => state
   DURABILITY_RUNTIME => ack, commit, durability, remote, transaction

OTHER EXECUTABLE SOURCE

HISTORICAL / GREENY EXECUTABLE REFERENCES

STATE ARTIFACTS — DO NOT PATCH AS IMPLEMENTATION
 - datasets/greenylife/raios-cognitive-state/RAIOS-STATE-LATEST/RAIOS/raios-cognitive-factory/state/doctrine/RAIOS-COGNITIVE-DOCTRINE.json
 - datasets/greenylife/raios-cognitive-state/RAIOS-STATE-LATEST/RAIOS/raios-cognitive-factory/state/journal/events.jsonl
 - datasets/greenylife/raios-cognitive-state/RAIOS-STATE-LATEST/RAIOS/raios-cogniti

In [18]:
from __future__ import annotations

from pathlib import Path
from collections import defaultdict
import subprocess
import shutil
import json
import re
import os

print("=" * 116)
print("RAIOS V8.6.2 — CERTIFIED SOURCE RESTORE + EXECUTION TRACE")
print("CPU ONLY | EXACT GIT SHA | NO MODEL | NO TRAINING | NO COGNITIVE STATE MUTATION")
print("=" * 116)

REPO_URL = "https://github.com/greenylifeonline-beep/Greeny-Life.git"
BRANCH = "raios/gl-005-convergence"
CERTIFIED_SHA = "3d9f58136d318ba07d743e127ab1e433605ce1ea"

WORK = Path("/kaggle/working")
REPO = WORK / "RAIOS-CERTIFIED-SOURCE"

# =============================================================================
# HELPERS
# =============================================================================

def run(cmd, cwd=None, check=True):
    result = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

    if check and result.returncode != 0:
        print(result.stdout)
        raise RuntimeError(
            f"Command failed ({result.returncode}): {' '.join(cmd)}"
        )

    return result.stdout.strip()


def read_text(path: Path, max_bytes=2_000_000):
    try:
        if path.stat().st_size > max_bytes:
            return ""
        return path.read_text(
            encoding="utf-8",
            errors="ignore"
        )
    except Exception:
        return ""


# =============================================================================
# 1. CLEAN ONLY OUR EPHEMERAL WORKING COPY
# =============================================================================

if REPO.exists():
    shutil.rmtree(REPO)

print("\n[RUN ] Restoring certified repository source...")
print("URL   :", REPO_URL)
print("BRANCH:", BRANCH)
print("SHA   :", CERTIFIED_SHA)

# =============================================================================
# 2. CLONE THE CERTIFIED BRANCH
#
# Start narrow to save time/bandwidth.
# =============================================================================

clone = subprocess.run(
    [
        "git",
        "clone",
        "--filter=blob:none",
        "--no-checkout",
        "--single-branch",
        "--branch",
        BRANCH,
        REPO_URL,
        str(REPO),
    ],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

if clone.returncode != 0:
    print(clone.stdout)
    raise RuntimeError(
        "Git clone failed. "
        "Do not enable GPU. Check Kaggle Internet/GitHub access."
    )

print("[PASS] Repository cloned")

# =============================================================================
# 3. CHECKOUT EXACT CERTIFIED SHA — NOT LATEST BRANCH HEAD
# =============================================================================

checkout = subprocess.run(
    ["git", "checkout", "--detach", CERTIFIED_SHA],
    cwd=str(REPO),
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

if checkout.returncode != 0:

    print("[INFO] Certified SHA not yet present locally; fetching exact object...")

    fetch = subprocess.run(
        ["git", "fetch", "origin", CERTIFIED_SHA],
        cwd=str(REPO),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

    if fetch.returncode != 0:
        print(fetch.stdout)
        raise RuntimeError(
            "Could not fetch the certified SHA. STOP."
        )

    run(
        ["git", "checkout", "--detach", CERTIFIED_SHA],
        cwd=REPO,
    )

HEAD = run(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO,
)

if HEAD != CERTIFIED_SHA:
    raise RuntimeError(
        f"Certified SHA mismatch: expected {CERTIFIED_SHA}, got {HEAD}"
    )

print("[PASS] Exact certified SHA restored:", HEAD)

# =============================================================================
# 4. REPOSITORY IDENTITY
# =============================================================================

branch_info = run(
    ["git", "branch", "--show-current"],
    cwd=REPO,
    check=False,
)

status = run(
    ["git", "status", "--short"],
    cwd=REPO,
    check=False,
)

print("\nCERTIFIED SOURCE")
print("Root   :", REPO)
print("HEAD   :", HEAD)
print("Mode   : DETACHED_CERTIFIED_SHA")
print("Dirty  :", bool(status.strip()))

if status.strip():
    print(status)

# =============================================================================
# 5. SOURCE INVENTORY
#
# Exclude generated/dependency/archive areas.
# =============================================================================

SOURCE_EXTS = {
    ".py",
    ".ts",
    ".tsx",
    ".js",
    ".jsx",
    ".mjs",
    ".cjs",
}

EXCLUDE_PARTS = {
    ".git",
    "node_modules",
    ".next",
    "dist",
    "build",
    "coverage",
    "__pycache__",
    "archive",
}

source_files = []

for p in REPO.rglob("*"):

    if not p.is_file():
        continue

    rel = p.relative_to(REPO)

    if any(part in EXCLUDE_PARTS for part in rel.parts):
        continue

    if p.suffix.lower() not in SOURCE_EXTS:
        continue

    source_files.append(p)

print("\n[PASS] Executable source files:", len(source_files))

# =============================================================================
# 6. CAPABILITY PROBES
#
# These are search probes, not proof by themselves.
# =============================================================================

PROBES = {

    "CONFIDENCE_BOUNDARY": [
        r"\bconfidence\b",
        r"avg_confidence",
        r"confidence_raw",
        r"minimumConfidence",
        r"normalize",
        r"semantic-state",
        r"0\s*,\s*1",
    ],

    "PROMOTION": [
        r"\bpromotion\b",
        r"\bpromote\b",
        r"PROMOTED",
        r"promotion_gate",
        r"skill.*status",
    ],

    "SKILL_ROUTING": [
        r"routing",
        r"compiled_entries",
        r"fallback_entries",
        r"ZERO_LLM",
        r"MICRO_DECISION",
        r"reusable_skill",
        r"skill_ref",
    ],

    "LEARNING": [
        r"experience",
        r"training.?candidate",
        r"failure.?memory",
        r"learning.?ledger",
        r"skill.?candidate",
        r"replay",
    ],

    "CHECKPOINT": [
        r"checkpoint",
        r"restore",
        r"resume",
        r"LATEST\.json",
    ],

    "DURABILITY": [
        r"durability",
        r"REMOTE_ACK",
        r"REMOTE_ACKNOWLEDGED",
        r"PRECOMMIT",
        r"transaction",
    ],

    "PROJECT_COORDINATION": [
        r"handoff",
        r"CURRENT-STATE",
        r"LOCKS",
        r"TASKS",
        r"\blease\b",
        r"agent.?identity",
        r"task.?registry",
        r"collision",
    ],
}

compiled = {
    category: [
        re.compile(pattern, re.I)
        for pattern in patterns
    ]
    for category, patterns in PROBES.items()
}

records = []

for path in source_files:

    text = read_text(path)

    if not text:
        continue

    hits = {}

    for category, patterns in compiled.items():

        found = []

        for rx in patterns:
            if rx.search(text):
                found.append(rx.pattern)

        if found:
            hits[category] = found

    if not hits:
        continue

    rel = str(
        path.relative_to(REPO)
    ).replace("\\", "/")

    records.append({
        "path": rel,
        "hits": hits,
        "category_count": len(hits),
        "signal_count": sum(
            len(v)
            for v in hits.values()
        ),
        "bytes": path.stat().st_size,
    })

records.sort(
    key=lambda x: (
        -x["category_count"],
        -x["signal_count"],
        x["bytes"],
    )
)

# =============================================================================
# 7. PRINT TOP SOURCE CANDIDATES
# =============================================================================

print("\n" + "=" * 116)
print("CERTIFIED EXECUTION CANDIDATES")
print("=" * 116)

for record in records[:80]:

    print("\nFILE:", record["path"])

    print(
        "CATEGORIES:",
        record["category_count"],
        "| SIGNALS:",
        record["signal_count"],
    )

    for category, hits in record["hits"].items():
        print(
            "  ",
            category,
            "=>",
            ", ".join(hits),
        )

# =============================================================================
# 8. BUILD CAPABILITY → FILE MAP
# =============================================================================

capability_map = defaultdict(list)

for record in records:

    for category in record["hits"]:
        capability_map[category].append(
            record["path"]
        )

print("\n" + "=" * 116)
print("CAPABILITY → CERTIFIED SOURCE MAP")
print("=" * 116)

for category in PROBES:

    paths = capability_map.get(
        category,
        []
    )

    print(
        f"\n{category}: "
        f"{len(paths)} source candidate(s)"
    )

    for path in paths[:20]:
        print("  -", path)

# =============================================================================
# 9. SPECIFIC FILE DISCOVERY
#
# Look for obvious RAIOS factory/runtime paths and relevant names.
# =============================================================================

keywords = [
    "raios",
    "cognitive",
    "learning",
    "skill",
    "routing",
    "semantic",
    "checkpoint",
    "durability",
    "experience",
    "replay",
    "promotion",
    "agent",
    "coordination",
]

named_candidates = []

for p in source_files:

    rel = str(
        p.relative_to(REPO)
    ).replace("\\", "/")

    low = rel.lower()

    if any(k in low for k in keywords):
        named_candidates.append(rel)

print("\n" + "=" * 116)
print("RAIOS / COGNITIVE NAMED SOURCE")
print("=" * 116)

for rel in sorted(named_candidates)[:150]:
    print(" -", rel)

if len(named_candidates) > 150:
    print(
        f" ... +{len(named_candidates)-150} more"
    )

# =============================================================================
# 10. VERIFY DURABILITY SNAPSHOT FILE AGAINST REPOSITORY IF PRESENT
# =============================================================================

repo_durability = [
    p for p in source_files
    if p.name == "durability_transaction.py"
]

print("\n" + "=" * 116)
print("DURABILITY SOURCE CHECK")
print("=" * 116)

if repo_durability:
    for p in repo_durability:
        print(
            "FOUND:",
            p.relative_to(REPO)
        )
else:
    print(
        "No durability_transaction.py found "
        "inside the certified Git checkout."
    )

# =============================================================================
# 11. DECISION
# =============================================================================

confidence_files = capability_map.get(
    "CONFIDENCE_BOUNDARY",
    []
)

promotion_files = capability_map.get(
    "PROMOTION",
    []
)

routing_files = capability_map.get(
    "SKILL_ROUTING",
    []
)

coordination_files = capability_map.get(
    "PROJECT_COORDINATION",
    []
)

core_ready = (
    bool(confidence_files)
    and bool(promotion_files)
    and bool(routing_files)
)

if core_ready:
    decision = "V8_6_2_CORE_PATCH_TRACE_READY"
else:
    decision = "SOURCE_TRACE_INCOMPLETE"

# Coordination may legitimately be absent:
# that is the thin extension we already identified.

report = {
    "schema":
        "raios.v8.6.2.certified-source-trace.v1",

    "repository": {
        "url": REPO_URL,
        "certified_branch": BRANCH,
        "certified_sha": CERTIFIED_SHA,
        "restored_head": HEAD,
    },

    "source_file_count":
        len(source_files),

    "candidate_records":
        records,

    "capability_map":
        dict(capability_map),

    "core_patch_trace_ready":
        core_ready,

    "coordination_existing_source_count":
        len(coordination_files),

    "decision":
        decision,

    "rules": [
        "Do not modify historical checkpoint records in place.",
        "Do not rewrite invalid historical evidence.",
        "Fix confidence at ingestion/validation boundary.",
        "Block quarantined skill at routing/promotion boundary.",
        "Use Greeny coordination behavior only as a thin extension.",
        "Do not duplicate routing, journal, checkpoint, durability, or memory systems.",
    ],
}

OUT = (
    WORK
    / "RAIOS-V8.6.2-CERTIFIED-SOURCE-TRACE.json"
)

OUT.write_text(
    json.dumps(
        report,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

# =============================================================================
# 12. FINAL
# =============================================================================

print("\n" + "=" * 116)
print("CERTIFIED SOURCE TRACE SUMMARY")
print("=" * 116)

print("HEAD                 :", HEAD)
print("SOURCE FILES         :", len(source_files))
print("CONFIDENCE FILES     :", len(confidence_files))
print("PROMOTION FILES      :", len(promotion_files))
print("ROUTING FILES        :", len(routing_files))
print("COORDINATION FILES   :", len(coordination_files))
print("CORE TRACE READY     :", core_ready)
print("DECISION             :", decision)

print("\nOUTPUT:")
print(OUT)

print("""
GPU USED        : NO
MODEL LOADED    : NO
TRAINING        : NO
COGNITIVE STATE : UNCHANGED

STOP HERE.
""")

RAIOS V8.6.2 — CERTIFIED SOURCE RESTORE + EXECUTION TRACE
CPU ONLY | EXACT GIT SHA | NO MODEL | NO TRAINING | NO COGNITIVE STATE MUTATION

[RUN ] Restoring certified repository source...
URL   : https://github.com/greenylifeonline-beep/Greeny-Life.git
BRANCH: raios/gl-005-convergence
SHA   : 3d9f58136d318ba07d743e127ab1e433605ce1ea
[PASS] Repository cloned
[PASS] Exact certified SHA restored: 3d9f58136d318ba07d743e127ab1e433605ce1ea

CERTIFIED SOURCE
Root   : /kaggle/working/RAIOS-CERTIFIED-SOURCE
HEAD   : 3d9f58136d318ba07d743e127ab1e433605ce1ea
Mode   : DETACHED_CERTIFIED_SHA
Dirty  : False

[PASS] Executable source files: 248

CERTIFIED EXECUTION CANDIDATES

FILE: lib/intelligence/task-orchestration.ts
CATEGORIES: 3 | SIGNALS: 4
   CONFIDENCE_BOUNDARY => normalize
   SKILL_ROUTING => routing
   PROJECT_COORDINATION => TASKS, collision

FILE: brain.py
CATEGORIES: 3 | SIGNALS: 4
   CONFIDENCE_BOUNDARY => normalize, 0\s*,\s*1
   PROMOTION => \bpromote\b
   DURABILITY => transaction

FI

In [20]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime, timezone
from decimal import Decimal, InvalidOperation
import json
import math
import hashlib
import shutil

print("=" * 118)
print("RAIOS V8.6.2 — RUNTIME SAFETY PATCH / BUILD + SELF-TEST")
print("CPU ONLY | WORKING COPY ONLY | NO TRAINING | NO INPUT MUTATION")
print("=" * 118)

INPUT = Path("/kaggle/input/datasets/greenylife")
RAIOS_DS = INPUT / "raios-cognitive-state"

RAIOS_ROOT = (
    RAIOS_DS
    / "RAIOS-STATE-LATEST"
    / "RAIOS"
)

FACTORY = (
    RAIOS_ROOT
    / "raios-cognitive-factory"
)

STATE = FACTORY / "state"

WORK_ROOT = Path("/kaggle/working/RAIOS-V8.6.2")
RUNTIME = WORK_ROOT / "runtime"
REPORTS = WORK_ROOT / "reports"

for p in [RUNTIME, REPORTS]:
    p.mkdir(parents=True, exist_ok=True)

INVALID_SKILL_HASH = (
    "683ca22e97a2cb919ab078321076bd06"
    "b146b1bf69e937a2062d68755e993b65"
)

INVALID_SKILL_ID = "RAIOS.REPOSITORY_ANALYSIS.MICRO.v1"

# -------------------------------------------------------------------------
# A. CONFIDENCE CONTRACT
# -------------------------------------------------------------------------

confidence_runtime = r'''
from __future__ import annotations

from dataclasses import dataclass
from decimal import Decimal, InvalidOperation
import math
from typing import Any


class ConfidenceContractError(ValueError):
    pass


@dataclass(frozen=True)
class ConfidenceResult:
    value: float
    original: Any
    normalized: bool
    source_type: str


def validate_confidence(value: Any) -> ConfidenceResult:
    """
    RAIOS canonical confidence domain: [0.0, 1.0].

    IMPORTANT:
    - We DO NOT silently convert 70 -> 0.70.
    - Numeric percentage-scale values are rejected.
    - Safe string representation like "0.70" may be parsed.
    - bool is rejected even though bool subclasses int in Python.
    """

    if isinstance(value, bool):
        raise ConfidenceContractError(
            "Boolean is not a valid confidence value."
        )

    original = value
    normalized = False

    if isinstance(value, str):
        stripped = value.strip()

        if not stripped:
            raise ConfidenceContractError(
                "Empty confidence string."
            )

        try:
            value = Decimal(stripped)
        except InvalidOperation as exc:
            raise ConfidenceContractError(
                f"Confidence is not numeric: {original!r}"
            ) from exc

        normalized = True

    elif isinstance(value, Decimal):
        pass

    elif isinstance(value, (int, float)):
        if isinstance(value, float) and not math.isfinite(value):
            raise ConfidenceContractError(
                f"Confidence must be finite: {value!r}"
            )

        value = Decimal(str(value))

    else:
        raise ConfidenceContractError(
            f"Unsupported confidence type: {type(original).__name__}"
        )

    if not value.is_finite():
        raise ConfidenceContractError(
            f"Confidence must be finite: {original!r}"
        )

    if value < Decimal("0") or value > Decimal("1"):
        raise ConfidenceContractError(
            "Confidence violates canonical [0,1] domain: "
            f"{original!r}"
        )

    result = float(value)

    return ConfidenceResult(
        value=result,
        original=original,
        normalized=normalized,
        source_type=type(original).__name__,
    )


def require_confidence(value: Any) -> float:
    return validate_confidence(value).value
'''

(RUNTIME / "confidence_contract.py").write_text(
    confidence_runtime,
    encoding="utf-8",
)

# -------------------------------------------------------------------------
# B. EFFECTIVE QUARANTINE
#
# Historical evidence is immutable.
# Runtime overlay determines whether an asset is usable now.
# -------------------------------------------------------------------------

quarantine_runtime = rf'''
from __future__ import annotations

from dataclasses import dataclass
from typing import Any


INVALIDATED_SKILLS = {{
    "{INVALID_SKILL_HASH}": {{
        "skill_id": "{INVALID_SKILL_ID}",
        "effective_status": "QUARANTINED_PENDING_REVALIDATION",
        "reason": "Historical promotion depended on invalid confidence scale.",
        "required_gate": "INDEPENDENT_REPLAY_REVALIDATION",
    }}
}}


@dataclass(frozen=True)
class EffectiveSkillState:
    content_hash: str
    stored_status: str | None
    effective_status: str
    usable: bool
    reason: str | None


def effective_skill_state(
    content_hash: str,
    stored_status: str | None,
) -> EffectiveSkillState:

    quarantine = INVALIDATED_SKILLS.get(content_hash)

    if quarantine:
        return EffectiveSkillState(
            content_hash=content_hash,
            stored_status=stored_status,
            effective_status=quarantine["effective_status"],
            usable=False,
            reason=quarantine["reason"],
        )

    return EffectiveSkillState(
        content_hash=content_hash,
        stored_status=stored_status,
        effective_status=stored_status or "UNKNOWN",
        usable=(stored_status == "PROMOTED"),
        reason=None,
    )
'''

(RUNTIME / "skill_quarantine.py").write_text(
    quarantine_runtime,
    encoding="utf-8",
)

# -------------------------------------------------------------------------
# C. ROUTING GUARD
# -------------------------------------------------------------------------

routing_guard = r'''
from __future__ import annotations

from dataclasses import dataclass
from typing import Any

from skill_quarantine import effective_skill_state


@dataclass(frozen=True)
class RoutingDecision:
    requested_route: str
    effective_route: str
    allowed: bool
    reason: str
    skill_hash: str | None


SAFE_FALLBACK = "REASONING"


def guard_skill_route(
    *,
    requested_route: str,
    skill_hash: str | None,
    stored_skill_status: str | None,
) -> RoutingDecision:

    route = requested_route.strip().upper()

    if skill_hash is None:
        return RoutingDecision(
            requested_route=route,
            effective_route=route,
            allowed=True,
            reason="NO_SKILL_BOUND_ROUTE",
            skill_hash=None,
        )

    state = effective_skill_state(
        content_hash=skill_hash,
        stored_status=stored_skill_status,
    )

    if not state.usable:
        return RoutingDecision(
            requested_route=route,
            effective_route=SAFE_FALLBACK,
            allowed=False,
            reason=(
                "SKILL_NOT_RUNTIME_USABLE:"
                + state.effective_status
            ),
            skill_hash=skill_hash,
        )

    return RoutingDecision(
        requested_route=route,
        effective_route=route,
        allowed=True,
        reason="SKILL_RUNTIME_USABLE",
        skill_hash=skill_hash,
    )
'''

(RUNTIME / "routing_guard.py").write_text(
    routing_guard,
    encoding="utf-8",
)

# -------------------------------------------------------------------------
# D. SELF TEST
# -------------------------------------------------------------------------

import sys

sys.path.insert(0, str(RUNTIME))

from confidence_contract import (
    validate_confidence,
    ConfidenceContractError,
)

from skill_quarantine import effective_skill_state
from routing_guard import guard_skill_route

tests = []

def record(name, passed, detail):
    tests.append({
        "name": name,
        "passed": bool(passed),
        "detail": str(detail),
    })

# Valid confidence
for value, expected in [
    (0, 0.0),
    (0.0, 0.0),
    (0.70, 0.70),
    (1, 1.0),
    ("0.70", 0.70),
]:
    try:
        got = validate_confidence(value).value
        record(
            f"confidence_accept_{value!r}",
            got == expected,
            got,
        )
    except Exception as exc:
        record(
            f"confidence_accept_{value!r}",
            False,
            exc,
        )

# Invalid confidence
for value in [
    70,
    14.79,
    -0.01,
    1.01,
    float("nan"),
    float("inf"),
    "",
    "70",
    True,
    None,
]:
    try:
        validate_confidence(value)

        record(
            f"confidence_reject_{value!r}",
            False,
            "unexpectedly accepted",
        )

    except ConfidenceContractError as exc:
        record(
            f"confidence_reject_{value!r}",
            True,
            exc,
        )

    except Exception as exc:
        record(
            f"confidence_reject_{value!r}",
            False,
            f"wrong exception: {exc}",
        )

# Historical invalid skill
skill_file = (
    STATE
    / "skills"
    / f"{INVALID_SKILL_HASH}.json"
)

if not skill_file.exists():
    raise RuntimeError(
        f"Known invalidated skill missing: {skill_file}"
    )

historical_skill = json.loads(
    skill_file.read_text(
        encoding="utf-8-sig"
    )
)

state = effective_skill_state(
    INVALID_SKILL_HASH,
    historical_skill.get("status"),
)

record(
    "historical_status_preserved",
    historical_skill.get("status") == "PROMOTED",
    historical_skill.get("status"),
)

record(
    "effective_status_quarantined",
    (
        state.effective_status
        == "QUARANTINED_PENDING_REVALIDATION"
        and state.usable is False
    ),
    state,
)

# Routing must fail closed.
guarded = guard_skill_route(
    requested_route="ZERO_LLM",
    skill_hash=INVALID_SKILL_HASH,
    stored_skill_status=historical_skill.get("status"),
)

record(
    "quarantined_zero_llm_blocked",
    (
        guarded.allowed is False
        and guarded.effective_route == "REASONING"
    ),
    guarded,
)

guarded_micro = guard_skill_route(
    requested_route="MICRO_DECISION",
    skill_hash=INVALID_SKILL_HASH,
    stored_skill_status=historical_skill.get("status"),
)

record(
    "quarantined_micro_decision_blocked",
    (
        guarded_micro.allowed is False
        and guarded_micro.effective_route == "REASONING"
    ),
    guarded_micro,
)

# -------------------------------------------------------------------------
# E. VERIFY KNOWN BAD EVIDENCE IS DETECTED
# -------------------------------------------------------------------------

known_bad_values = []

def walk(obj, path="$"):
    if isinstance(obj, dict):
        for key, value in obj.items():
            yield from walk(
                value,
                f"{path}.{key}"
            )

    elif isinstance(obj, list):
        for i, value in enumerate(obj):
            yield from walk(
                value,
                f"{path}[{i}]"
            )

    else:
        yield path, obj


for rel in [
    Path("skills") / f"{INVALID_SKILL_HASH}.json",
    Path("experiences") / "fc5089847ca28e90915d9f4ef45b84c0a29baf6f004ad6b48173a3b9a267d47f.json",
    Path("benchmarks") / "059e8506518a5eb4663c202509556e852ee17433d6e1e7d269b13dd3abcfc4d0.json",
]:

    p = STATE / rel

    if not p.exists():
        continue

    obj = json.loads(
        p.read_text(
            encoding="utf-8-sig"
        )
    )

    for json_path, value in walk(obj):

        if (
            "confidence" in json_path.lower()
            and isinstance(value, (int, float))
            and not isinstance(value, bool)
        ):
            try:
                validate_confidence(value)
            except ConfidenceContractError:
                known_bad_values.append({
                    "file": str(rel),
                    "path": json_path,
                    "value": value,
                })

record(
    "known_confidence_corruption_detected",
    len(known_bad_values) >= 3,
    known_bad_values,
)

# -------------------------------------------------------------------------
# F. RESULT
# -------------------------------------------------------------------------

failed = [
    test
    for test in tests
    if not test["passed"]
]

report = {
    "schema":
        "raios.v8.6.2.runtime-safety-patch-test.v1",

    "generated_at_utc":
        datetime.now(timezone.utc).isoformat(),

    "historical_state_mutated":
        False,

    "runtime_files": [
        "runtime/confidence_contract.py",
        "runtime/skill_quarantine.py",
        "runtime/routing_guard.py",
    ],

    "known_invalid_skill": {
        "content_hash": INVALID_SKILL_HASH,
        "skill_id": INVALID_SKILL_ID,
        "stored_status":
            historical_skill.get("status"),
        "effective_status":
            state.effective_status,
    },

    "known_bad_confidence_values":
        known_bad_values,

    "tests":
        tests,

    "summary": {
        "total": len(tests),
        "passed": len(tests) - len(failed),
        "failed": len(failed),
    },
}

REPORT_PATH = REPORTS / "runtime-safety-self-test.json"

REPORT_PATH.write_text(
    json.dumps(
        report,
        ensure_ascii=False,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

# Hash generated runtime source.
hashes = {}

for p in sorted(RUNTIME.glob("*.py")):
    digest = hashlib.sha256(
        p.read_bytes()
    ).hexdigest()

    hashes[p.name] = digest

print("\n" + "=" * 118)
print("V8.6.2 RUNTIME SAFETY PATCH RESULT")
print("=" * 118)

print("Tests total :", len(tests))
print("Passed      :", len(tests) - len(failed))
print("Failed      :", len(failed))

print("\nRuntime files:")

for name, digest in hashes.items():
    print(
        f" - {name:28} {digest}"
    )

print("\nHistorical skill:")
print(
    " Stored status   :",
    historical_skill.get("status")
)
print(
    " Effective status:",
    state.effective_status
)
print(
    " Runtime usable  :",
    state.usable
)

print("\nKnown bad confidence values detected:")
for item in known_bad_values:
    print(
        " -",
        item["file"],
        "|",
        item["path"],
        "=",
        item["value"],
    )

if failed:
    print("\nFAILED TESTS:")
    for t in failed:
        print(
            " -",
            t["name"],
            "=>",
            t["detail"],
        )

    print("\nSTATUS: PATCH_SELF_TEST_FAILED")

else:
    print("\nSTATUS: PATCH_SELF_TEST_PASS")

print("\nOUTPUT:")
print(REPORT_PATH)

print("""
GPU USED        : NO
MODEL LOADED    : NO
TRAINING        : NO
INPUT MUTATED   : NO

IMPORTANT:
This is an isolated runtime candidate only.
Nothing has been promoted or committed yet.
""")

RAIOS V8.6.2 — RUNTIME SAFETY PATCH / BUILD + SELF-TEST
CPU ONLY | WORKING COPY ONLY | NO TRAINING | NO INPUT MUTATION

V8.6.2 RUNTIME SAFETY PATCH RESULT
Tests total : 20
Passed      : 20
Failed      : 0

Runtime files:
 - confidence_contract.py       d5306df6c42a52cb2937b458584575174ddfc52ee488b121a437a240ac346f85
 - routing_guard.py             fbda164c233b39ec25b60d59f5b8b45cc2c5f60604149e5aa646993d4d130c20
 - skill_quarantine.py          883f438b7aa832c051ae83222bdd3ceae736d2b8e05abc9067e92b4cecb2514a

Historical skill:
 Stored status   : PROMOTED
 Effective status: QUARANTINED_PENDING_REVALIDATION
 Runtime usable  : False

Known bad confidence values detected:
 - skills/683ca22e97a2cb919ab078321076bd06b146b1bf69e937a2062d68755e993b65.json | $.metrics.avg_confidence = 14.790000000000001
 - experiences/fc5089847ca28e90915d9f4ef45b84c0a29baf6f004ad6b48173a3b9a267d47f.json | $.avg_confidence = 14.790000000000001
 - benchmarks/059e8506518a5eb4663c202509556e852ee17433d6e1e7d269b13dd3ab

In [21]:
from __future__ import annotations

from pathlib import Path
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from typing import Any
import hashlib
import importlib.util
import json
import re
import sys

print("=" * 120)
print("RAIOS V8.6.2-R2 — GREENY AI-OS COORDINATION CONVERGENCE")
print("CPU ONLY | WORKING COPY ONLY | NO MODEL | NO TRAINING | NO INPUT MUTATION")
print("=" * 120)

# =============================================================================
# PATHS
# =============================================================================

INPUT_ROOT = Path("/kaggle/input/datasets/greenylife")

CONTINUITY_DS = (
    INPUT_ROOT
    / "raios-greeny-continuity-evidence"
)

RAIOS_DS = (
    INPUT_ROOT
    / "raios-cognitive-state"
)

RAIOS_ROOT = (
    RAIOS_DS
    / "RAIOS-STATE-LATEST"
    / "RAIOS"
)

FACTORY = (
    RAIOS_ROOT
    / "raios-cognitive-factory"
)

STATE = FACTORY / "state"

CERTIFIED_SOURCE = Path(
    "/kaggle/working/RAIOS-CERTIFIED-SOURCE"
)

V862 = Path(
    "/kaggle/working/RAIOS-V8.6.2"
)

RUNTIME = V862 / "runtime"
REPORTS = V862 / "reports"

RUNTIME.mkdir(
    parents=True,
    exist_ok=True
)

REPORTS.mkdir(
    parents=True,
    exist_ok=True
)

required = [
    CONTINUITY_DS,
    STATE,
    CERTIFIED_SOURCE,
    RUNTIME / "confidence_contract.py",
    RUNTIME / "skill_quarantine.py",
    RUNTIME / "routing_guard.py",
]

for p in required:
    if not p.exists():
        raise RuntimeError(
            f"Required dependency missing: {p}"
        )


# =============================================================================
# LOAD PREVIOUS RUNTIME CANDIDATES
# =============================================================================

sys.path.insert(
    0,
    str(RUNTIME)
)

from confidence_contract import (
    validate_confidence,
    ConfidenceContractError,
)

from skill_quarantine import (
    effective_skill_state,
)

from routing_guard import (
    guard_skill_route,
)


INVALID_SKILL_HASH = (
    "683ca22e97a2cb919ab078321076bd06"
    "b146b1bf69e937a2062d68755e993b65"
)


# =============================================================================
# CONTINUITY MANIFEST
# =============================================================================

manifest_path = (
    CONTINUITY_DS
    / "CONTINUITY-EVIDENCE-MANIFEST.json"
)

manifest = json.loads(
    manifest_path.read_text(
        encoding="utf-8-sig"
    )
)

package_by_source = {
    item["sourcePath"]: (
        CONTINUITY_DS
        / item["packagedAs"]
    )
    for item in manifest["files"]
}


def required_source(source_path: str) -> Path:
    p = package_by_source.get(
        source_path
    )

    if p is None or not p.exists():
        raise RuntimeError(
            f"Continuity source missing: {source_path}"
        )

    return p


CURRENT_STATE_FILE = required_source(
    ".ai-os/state/CURRENT-STATE.json"
)

TASKS_FILE = required_source(
    ".ai-os/state/TASKS.json"
)

LOCKS_FILE = required_source(
    ".ai-os/state/LOCKS.json"
)

CORE_CONTRACT_FILE = required_source(
    ".ai-os/CORE-CONTRACT.md"
)

ROUTING_FILE = required_source(
    ".ai-os/ROUTING.md"
)

HANDOFF_FILES = [
    p
    for source, p in package_by_source.items()
    if "/handoffs/" in source
]


# =============================================================================
# CERTIFIED AI-OS IMPLEMENTATION EVIDENCE
# =============================================================================

AIOS_SOURCE = (
    CERTIFIED_SOURCE
    / "scripts"
    / "ai-os"
    / "aios.py"
)

LOCAL_AGENT_SOURCE = (
    CERTIFIED_SOURCE
    / "scripts"
    / "ai-os"
    / "local-agent.py"
)

if not AIOS_SOURCE.exists():
    raise RuntimeError(
        "Certified scripts/ai-os/aios.py missing."
    )

if not LOCAL_AGENT_SOURCE.exists():
    raise RuntimeError(
        "Certified scripts/ai-os/local-agent.py missing."
    )


# =============================================================================
# BUILD THIN PROJECT COORDINATION RUNTIME
# =============================================================================

coordination_runtime = r'''
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any
import json
import re


@dataclass(frozen=True)
class ProjectAgentIdentity:
    agent_id: str
    source: str


@dataclass(frozen=True)
class ProjectTaskView:
    task_id: str | None
    status: str | None
    owner: str | None
    scope: Any
    raw: Any


@dataclass(frozen=True)
class ProjectLockView:
    path: str | None
    owner: str | None
    task_id: str | None
    lease: Any
    raw: Any


@dataclass(frozen=True)
class ProjectHandoffView:
    source: str
    summary: str
    next_action: str | None
    raw_text: str


@dataclass(frozen=True)
class ProjectResumeContext:
    repository_branch: str | None
    repository_head: str | None
    current_state: Any
    current_task: ProjectTaskView | None
    active_locks: tuple[ProjectLockView, ...]
    handoffs: tuple[ProjectHandoffView, ...]
    collision_paths: tuple[str, ...]
    agent_identity: ProjectAgentIdentity
    safe_to_resume: bool
    blockers: tuple[str, ...]


def _load_json(path: Path) -> Any:
    return json.loads(
        path.read_text(
            encoding="utf-8-sig"
        )
    )


def _dict_items(value: Any):
    if isinstance(value, dict):
        return value.items()
    return []


def _records(value: Any):
    """
    Accepts several existing JSON shapes without imposing
    a new Greeny/RAIOS schema.
    """
    if isinstance(value, list):
        return value

    if isinstance(value, dict):
        for key in (
            "tasks",
            "items",
            "records",
            "locks",
            "active",
        ):
            candidate = value.get(key)
            if isinstance(candidate, list):
                return candidate

        # Fall back to dictionary values when they look record-like.
        dict_values = [
            v
            for v in value.values()
            if isinstance(v, dict)
        ]

        if dict_values:
            return dict_values

    return []


def _pick(record: dict, *names):
    for name in names:
        if name in record:
            return record[name]
    return None


def _normalize_status(value: Any) -> str | None:
    if value is None:
        return None

    return str(value).strip().upper()


def load_tasks(path: Path) -> list[ProjectTaskView]:
    data = _load_json(path)
    output = []

    for record in _records(data):
        if not isinstance(record, dict):
            continue

        output.append(
            ProjectTaskView(
                task_id=(
                    _pick(
                        record,
                        "task_id",
                        "taskId",
                        "id",
                        "key",
                    )
                ),
                status=_normalize_status(
                    _pick(
                        record,
                        "status",
                        "state",
                    )
                ),
                owner=(
                    _pick(
                        record,
                        "owner",
                        "agent",
                        "assignedTo",
                        "assigned_to",
                    )
                ),
                scope=_pick(
                    record,
                    "scope",
                    "paths",
                    "path",
                ),
                raw=record,
            )
        )

    return output


def load_locks(path: Path) -> list[ProjectLockView]:
    data = _load_json(path)
    output = []

    for record in _records(data):
        if not isinstance(record, dict):
            continue

        path_value = _pick(
            record,
            "path",
            "resource",
            "target",
            "scope",
        )

        output.append(
            ProjectLockView(
                path=(
                    str(path_value)
                    if path_value is not None
                    else None
                ),
                owner=_pick(
                    record,
                    "owner",
                    "agent",
                    "holder",
                ),
                task_id=_pick(
                    record,
                    "task_id",
                    "taskId",
                    "task",
                ),
                lease=_pick(
                    record,
                    "lease",
                    "expires",
                    "expiresAt",
                    "expires_at",
                ),
                raw=record,
            )
        )

    return output


def parse_handoff(
    source: str,
    text: str,
) -> ProjectHandoffView:

    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    summary = (
        lines[0]
        if lines
        else "EMPTY_HANDOFF"
    )

    next_action = None

    patterns = [
        r"next(?:\s+action|\s+step)?\s*[:=-]\s*(.+)",
        r"resume\s*[:=-]\s*(.+)",
        r"continue\s*[:=-]\s*(.+)",
    ]

    for line in lines:
        for pattern in patterns:
            match = re.search(
                pattern,
                line,
                flags=re.I,
            )

            if match:
                next_action = (
                    match.group(1).strip()
                )
                break

        if next_action:
            break

    return ProjectHandoffView(
        source=source,
        summary=summary,
        next_action=next_action,
        raw_text=text,
    )


def select_current_task(
    tasks: list[ProjectTaskView],
) -> ProjectTaskView | None:

    active_statuses = {
        "ACTIVE",
        "IN_PROGRESS",
        "RUNNING",
        "STARTED",
        "ASSIGNED",
        "OPEN",
    }

    for task in tasks:
        if task.status in active_statuses:
            return task

    # No invention.
    return None


def detect_collisions(
    locks: list[ProjectLockView],
) -> list[str]:

    owners_by_path: dict[str, set[str]] = {}

    for lock in locks:
        if not lock.path:
            continue

        owner = (
            str(lock.owner)
            if lock.owner is not None
            else "UNKNOWN_OWNER"
        )

        owners_by_path.setdefault(
            lock.path,
            set(),
        ).add(owner)

    return sorted(
        path
        for path, owners
        in owners_by_path.items()
        if len(owners) > 1
    )


def build_resume_context(
    *,
    current_state_path: Path,
    tasks_path: Path,
    locks_path: Path,
    handoffs: list[tuple[str, Path]],
    repository_branch: str | None,
    repository_head: str | None,
    agent_id: str,
) -> ProjectResumeContext:

    current_state = _load_json(
        current_state_path
    )

    tasks = load_tasks(
        tasks_path
    )

    locks = load_locks(
        locks_path
    )

    parsed_handoffs = []

    for source, path in handoffs:
        parsed_handoffs.append(
            parse_handoff(
                source,
                path.read_text(
                    encoding="utf-8",
                    errors="ignore",
                ),
            )
        )

    current_task = select_current_task(
        tasks
    )

    collisions = detect_collisions(
        locks
    )

    blockers = []

    if collisions:
        blockers.append(
            "LOCK_COLLISION_DETECTED"
        )

    if current_task is None:
        blockers.append(
            "NO_ACTIVE_TASK_POINTER"
        )

    if not repository_head:
        blockers.append(
            "REPOSITORY_IDENTITY_MISSING"
        )

    return ProjectResumeContext(
        repository_branch=repository_branch,
        repository_head=repository_head,
        current_state=current_state,
        current_task=current_task,
        active_locks=tuple(locks),
        handoffs=tuple(parsed_handoffs),
        collision_paths=tuple(collisions),
        agent_identity=ProjectAgentIdentity(
            agent_id=agent_id,
            source="GREENY_AI_OS",
        ),
        safe_to_resume=(
            len(collisions) == 0
            and repository_head is not None
        ),
        blockers=tuple(blockers),
    )
'''

(
    RUNTIME
    / "project_coordination.py"
).write_text(
    coordination_runtime,
    encoding="utf-8",
)


# =============================================================================
# BUILD UNIFIED RAIOS ↔ GREENY BRIDGE
# =============================================================================

bridge_runtime = r'''
from __future__ import annotations

from dataclasses import dataclass
from typing import Any

from confidence_contract import (
    validate_confidence,
)

from routing_guard import (
    guard_skill_route,
)

from project_coordination import (
    ProjectResumeContext,
)


@dataclass(frozen=True)
class RaiosProjectDecision:
    confidence: float
    requested_route: str
    effective_route: str
    route_allowed: bool
    route_reason: str
    project_safe_to_resume: bool
    project_blockers: tuple[str, ...]


def bind_project_decision(
    *,
    confidence: Any,
    requested_route: str,
    skill_hash: str | None,
    stored_skill_status: str | None,
    project_context: ProjectResumeContext,
) -> RaiosProjectDecision:

    confidence_result = (
        validate_confidence(
            confidence
        )
    )

    route = guard_skill_route(
        requested_route=requested_route,
        skill_hash=skill_hash,
        stored_skill_status=stored_skill_status,
    )

    blockers = list(
        project_context.blockers
    )

    if not project_context.safe_to_resume:
        blockers.append(
            "PROJECT_CONTEXT_NOT_SAFE"
        )

    return RaiosProjectDecision(
        confidence=confidence_result.value,
        requested_route=route.requested_route,
        effective_route=route.effective_route,
        route_allowed=(
            route.allowed
            and project_context.safe_to_resume
        ),
        route_reason=route.reason,
        project_safe_to_resume=(
            project_context.safe_to_resume
        ),
        project_blockers=tuple(
            dict.fromkeys(blockers)
        ),
    )
'''

(
    RUNTIME
    / "greeny_raios_bridge.py"
).write_text(
    bridge_runtime,
    encoding="utf-8",
)


# =============================================================================
# IMPORT GENERATED RUNTIME
# =============================================================================

from project_coordination import (
    build_resume_context,
    load_tasks,
    load_locks,
)

from greeny_raios_bridge import (
    bind_project_decision,
)


# =============================================================================
# BUILD PROJECT CONTEXT FROM REAL GREENY EVIDENCE
# =============================================================================

repo_branch = (
    manifest
    .get("repository", {})
    .get("branch")
)

repo_head = (
    manifest
    .get("repository", {})
    .get("head")
)

handoff_pairs = []

for source, path in package_by_source.items():
    if "/handoffs/" in source:
        handoff_pairs.append(
            (source, path)
        )

context = build_resume_context(
    current_state_path=CURRENT_STATE_FILE,
    tasks_path=TASKS_FILE,
    locks_path=LOCKS_FILE,
    handoffs=handoff_pairs,
    repository_branch=repo_branch,
    repository_head=repo_head,
    agent_id="RAIOS",
)


# =============================================================================
# LOAD HISTORICAL BAD SKILL
# =============================================================================

skill_path = (
    STATE
    / "skills"
    / f"{INVALID_SKILL_HASH}.json"
)

historical_skill = json.loads(
    skill_path.read_text(
        encoding="utf-8-sig"
    )
)


# =============================================================================
# TEST HARNESS
# =============================================================================

tests = []


def add_test(
    name: str,
    passed: bool,
    detail: Any,
):
    tests.append({
        "name": name,
        "passed": bool(passed),
        "detail": str(detail),
    })


# -----------------------------------------------------------------------------
# AI-OS source reuse proof
# -----------------------------------------------------------------------------

aios_text = AIOS_SOURCE.read_text(
    encoding="utf-8",
    errors="ignore",
)

local_agent_text = (
    LOCAL_AGENT_SOURCE
    .read_text(
        encoding="utf-8",
        errors="ignore",
    )
)

for signal in [
    "CURRENT-STATE",
    "LOCKS",
    "TASKS",
]:
    add_test(
        f"aios_existing_{signal}",
        (
            signal.lower()
            in (
                aios_text
                + "\n"
                + local_agent_text
            ).lower()
        ),
        signal,
    )


# -----------------------------------------------------------------------------
# Real Greeny package resolution
# -----------------------------------------------------------------------------

add_test(
    "greeny_current_state_loaded",
    isinstance(
        context.current_state,
        (dict, list),
    ),
    type(context.current_state).__name__,
)

add_test(
    "greeny_handoffs_loaded",
    len(context.handoffs) >= 2,
    len(context.handoffs),
)

add_test(
    "repository_identity_preserved",
    (
        context.repository_branch
        == "codex-clean"
        and
        context.repository_head
        == "fb130e232246a00b531835f529f25e4edfae2854"
    ),
    (
        context.repository_branch,
        context.repository_head,
    ),
)


# -----------------------------------------------------------------------------
# Coordination parser behavior
# -----------------------------------------------------------------------------

tasks = load_tasks(
    TASKS_FILE
)

locks = load_locks(
    LOCKS_FILE
)

add_test(
    "tasks_parse_without_schema_rewrite",
    isinstance(tasks, list),
    len(tasks),
)

add_test(
    "locks_parse_without_schema_rewrite",
    isinstance(locks, list),
    len(locks),
)

add_test(
    "agent_identity_bound",
    context.agent_identity.agent_id == "RAIOS",
    context.agent_identity,
)


# -----------------------------------------------------------------------------
# Confidence gate remains enforced through bridge
# -----------------------------------------------------------------------------

try:
    bind_project_decision(
        confidence=70,
        requested_route="REASONING",
        skill_hash=None,
        stored_skill_status=None,
        project_context=context,
    )

    add_test(
        "bridge_rejects_confidence_70",
        False,
        "70 unexpectedly accepted",
    )

except ConfidenceContractError as exc:
    add_test(
        "bridge_rejects_confidence_70",
        True,
        exc,
    )


# -----------------------------------------------------------------------------
# Quarantined skill remains blocked through project bridge
# -----------------------------------------------------------------------------

decision = bind_project_decision(
    confidence=0.70,
    requested_route="ZERO_LLM",
    skill_hash=INVALID_SKILL_HASH,
    stored_skill_status=(
        historical_skill.get("status")
    ),
    project_context=context,
)

add_test(
    "bridge_blocks_invalid_skill_zero_llm",
    (
        decision.effective_route
        == "REASONING"
        and decision.route_allowed is False
    ),
    decision,
)


decision_micro = bind_project_decision(
    confidence="0.70",
    requested_route="MICRO_DECISION",
    skill_hash=INVALID_SKILL_HASH,
    stored_skill_status=(
        historical_skill.get("status")
    ),
    project_context=context,
)

add_test(
    "bridge_blocks_invalid_skill_micro",
    (
        decision_micro.effective_route
        == "REASONING"
        and decision_micro.route_allowed is False
    ),
    decision_micro,
)


# -----------------------------------------------------------------------------
# Valid unbound reasoning path
# -----------------------------------------------------------------------------

normal_decision = bind_project_decision(
    confidence=0.82,
    requested_route="REASONING",
    skill_hash=None,
    stored_skill_status=None,
    project_context=context,
)

add_test(
    "normal_confidence_preserved",
    normal_decision.confidence == 0.82,
    normal_decision.confidence,
)

add_test(
    "no_new_routing_system",
    normal_decision.requested_route == "REASONING",
    normal_decision,
)


# =============================================================================
# COLLISION TEST WITH SYNTHETIC TEMP FILES
#
# Tests behavior only. Does not mutate Greeny or RAIOS inputs.
# =============================================================================

synthetic_dir = (
    V862
    / "synthetic-coordination-test"
)

synthetic_dir.mkdir(
    parents=True,
    exist_ok=True
)

synthetic_current = (
    synthetic_dir
    / "CURRENT.json"
)

synthetic_tasks = (
    synthetic_dir
    / "TASKS.json"
)

synthetic_locks = (
    synthetic_dir
    / "LOCKS.json"
)

synthetic_handoff = (
    synthetic_dir
    / "HANDOFF.md"
)

synthetic_current.write_text(
    json.dumps({
        "status": "ACTIVE",
        "phase": "TEST",
    }),
    encoding="utf-8",
)

synthetic_tasks.write_text(
    json.dumps({
        "tasks": [
            {
                "id": "T-1",
                "status": "IN_PROGRESS",
                "owner": "RAIOS",
                "scope": ["lib/a.ts"],
            }
        ]
    }),
    encoding="utf-8",
)

synthetic_locks.write_text(
    json.dumps({
        "locks": [
            {
                "path": "lib/a.ts",
                "owner": "RAIOS",
                "task": "T-1",
            },
            {
                "path": "lib/a.ts",
                "owner": "OTHER_AGENT",
                "task": "T-2",
            },
        ]
    }),
    encoding="utf-8",
)

synthetic_handoff.write_text(
    """
# Handoff
Completed: inspection
Next action: continue T-1
""".strip(),
    encoding="utf-8",
)

collision_context = build_resume_context(
    current_state_path=synthetic_current,
    tasks_path=synthetic_tasks,
    locks_path=synthetic_locks,
    handoffs=[
        (
            "synthetic/HANDOFF.md",
            synthetic_handoff,
        )
    ],
    repository_branch="test",
    repository_head="abc",
    agent_id="RAIOS",
)

add_test(
    "collision_detected",
    "lib/a.ts"
    in collision_context.collision_paths,
    collision_context.collision_paths,
)

add_test(
    "collision_blocks_safe_resume",
    collision_context.safe_to_resume is False,
    collision_context.blockers,
)


# =============================================================================
# NO INPUT MUTATION PROOF
# =============================================================================

def digest_file(path: Path):
    return hashlib.sha256(
        path.read_bytes()
    ).hexdigest()


evidence_digests = {}

for source, path in package_by_source.items():
    evidence_digests[source] = (
        digest_file(path)
    )

skill_digest_after = digest_file(
    skill_path
)

# We did not record "before" in this cell,
# so compare against known content hash identity logic:
# file itself must still represent same historical skill.

add_test(
    "historical_skill_still_promoted",
    historical_skill.get("status")
    == "PROMOTED",
    historical_skill.get("status"),
)


# =============================================================================
# REPORT
# =============================================================================

failed = [
    t
    for t in tests
    if not t["passed"]
]

runtime_hashes = {}

for path in sorted(
    RUNTIME.glob("*.py")
):
    runtime_hashes[path.name] = (
        digest_file(path)
    )


def safe_task(task):
    if task is None:
        return None

    return {
        "task_id": task.task_id,
        "status": task.status,
        "owner": task.owner,
        "scope": task.scope,
    }


report = {
    "schema":
        "raios.v8.6.2-r2.greeny-coordination-convergence.v1",

    "generated_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "greenylife_repository": {
        "branch": repo_branch,
        "head": repo_head,
    },

    "certified_ai_os_sources": [
        "scripts/ai-os/aios.py",
        "scripts/ai-os/local-agent.py",
    ],

    "runtime_components": {
        "existing_v862": [
            "confidence_contract.py",
            "skill_quarantine.py",
            "routing_guard.py",
        ],

        "r2_added": [
            "project_coordination.py",
            "greeny_raios_bridge.py",
        ],
    },

    "reuse_policy": {
        "new_routing_system": False,
        "new_memory_system": False,
        "new_checkpoint_system": False,
        "new_durability_system": False,
        "new_project_state_schema": False,
        "greenylife_ai_os_behavior_reused": True,
    },

    "project_context": {
        "repository_branch":
            context.repository_branch,

        "repository_head":
            context.repository_head,

        "current_task":
            safe_task(
                context.current_task
            ),

        "lock_count":
            len(context.active_locks),

        "handoff_count":
            len(context.handoffs),

        "collision_paths":
            list(
                context.collision_paths
            ),

        "safe_to_resume":
            context.safe_to_resume,

        "blockers":
            list(context.blockers),
    },

    "invalid_skill": {
        "hash":
            INVALID_SKILL_HASH,

        "stored_status":
            historical_skill.get(
                "status"
            ),

        "effective_status":
            effective_skill_state(
                INVALID_SKILL_HASH,
                historical_skill.get(
                    "status"
                ),
            ).effective_status,
    },

    "tests":
        tests,

    "summary": {
        "total":
            len(tests),

        "passed":
            len(tests) - len(failed),

        "failed":
            len(failed),
    },

    "runtime_hashes":
        runtime_hashes,

    "input_mutated":
        False,
}

REPORT_PATH = (
    REPORTS
    / "v8.6.2-r2-coordination-self-test.json"
)

REPORT_PATH.write_text(
    json.dumps(
        report,
        ensure_ascii=False,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)


# =============================================================================
# FINAL OUTPUT
# =============================================================================

print("\n" + "=" * 120)
print("V8.6.2-R2 COORDINATION CONVERGENCE RESULT")
print("=" * 120)

print(
    "Tests total      :",
    len(tests)
)

print(
    "Passed           :",
    len(tests) - len(failed)
)

print(
    "Failed           :",
    len(failed)
)

print()
print("REAL GREENY PROJECT CONTEXT")

print(
    "Branch           :",
    context.repository_branch
)

print(
    "HEAD             :",
    context.repository_head
)

print(
    "Tasks parsed     :",
    len(tasks)
)

print(
    "Locks parsed     :",
    len(locks)
)

print(
    "Handoffs parsed  :",
    len(context.handoffs)
)

print(
    "Current task     :",
    safe_task(context.current_task)
)

print(
    "Collisions       :",
    list(context.collision_paths)
)

print(
    "Safe to resume   :",
    context.safe_to_resume
)

print(
    "Blockers         :",
    list(context.blockers)
)

print()
print("RAIOS SAFETY")

print(
    "Invalid skill stored status   :",
    historical_skill.get("status")
)

effective = effective_skill_state(
    INVALID_SKILL_HASH,
    historical_skill.get("status"),
)

print(
    "Invalid skill effective status:",
    effective.effective_status
)

print(
    "Invalid skill runtime usable  :",
    effective.usable
)

print()
print("RUNTIME FILES")

for name, digest in sorted(
    runtime_hashes.items()
):
    print(
        f" - {name:32} {digest}"
    )

if failed:

    print("\nFAILED TESTS")

    for t in failed:
        print(
            " -",
            t["name"],
            "=>",
            t["detail"],
        )

    print(
        "\nSTATUS: "
        "V8.6.2-R2_SELF_TEST_FAILED"
    )

else:

    print(
        "\nSTATUS: "
        "V8.6.2-R2_SELF_TEST_PASS"
    )

print()
print("OUTPUT:")
print(REPORT_PATH)

print("""
GPU USED        : NO
MODEL LOADED    : NO
TRAINING        : NO
INPUT MUTATED   : NO
PROMOTED        : NO

NEXT GATE:
INDEPENDENT REPLAY REVALIDATION
""")

RAIOS V8.6.2-R2 — GREENY AI-OS COORDINATION CONVERGENCE
CPU ONLY | WORKING COPY ONLY | NO MODEL | NO TRAINING | NO INPUT MUTATION

V8.6.2-R2 COORDINATION CONVERGENCE RESULT
Tests total      : 17
Passed           : 17
Failed           : 0

REAL GREENY PROJECT CONTEXT
Branch           : codex-clean
HEAD             : fb130e232246a00b531835f529f25e4edfae2854
Tasks parsed     : 5
Locks parsed     : 8
Handoffs parsed  : 2
Current task     : {'task_id': 'GL-003', 'status': 'IN_PROGRESS', 'owner': None, 'scope': ['projects', 'brains', 'canonical', 'intelligence']}
Collisions       : ['governance', 'intelligence/main']
Safe to resume   : False
Blockers         : ['LOCK_COLLISION_DETECTED']

RAIOS SAFETY
Invalid skill stored status   : PROMOTED
Invalid skill effective status: QUARANTINED_PENDING_REVALIDATION
Invalid skill runtime usable  : False

RUNTIME FILES
 - confidence_contract.py           d5306df6c42a52cb2937b458584575174ddfc52ee488b121a437a240ac346f85
 - greeny_raios_bridge.py          

In [1]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime, timezone
import json
import os
import re
import sys
import time
import hashlib
import subprocess

print("=" * 122)
print("RAIOS V8.6.2-R3 — INDEPENDENT REPLAY REVALIDATION")
print("T4 GATE | NO TRAINING | NO PROMOTION | FAIL CLOSED")
print("=" * 122)

# =============================================================================
# PATHS
# =============================================================================

INPUT = Path("/kaggle/input/datasets/greenylife")

RAIOS_DS = INPUT / "raios-cognitive-state"

RAIOS_ROOT = (
    RAIOS_DS
    / "RAIOS-STATE-LATEST"
    / "RAIOS"
)

FACTORY = (
    RAIOS_ROOT
    / "raios-cognitive-factory"
)

STATE = FACTORY / "state"

V862 = Path("/kaggle/working/RAIOS-V8.6.2")
RUNTIME = V862 / "runtime"
REPORTS = V862 / "reports"
R3 = V862 / "r3-replay"

REPORTS.mkdir(parents=True, exist_ok=True)
R3.mkdir(parents=True, exist_ok=True)

INVALID_SKILL_HASH = (
    "683ca22e97a2cb919ab078321076bd06"
    "b146b1bf69e937a2062d68755e993b65"
)

INVALID_SKILL_ID = "RAIOS.REPOSITORY_ANALYSIS.MICRO.v1"

# =============================================================================
# 1. VERIFY GPU
# =============================================================================

print("\n" + "=" * 122)
print("GPU GATE")
print("=" * 122)

gpu_info = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total,memory.free",
        "--format=csv,noheader",
    ],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

if gpu_info.returncode != 0:
    print(gpu_info.stdout)
    raise RuntimeError(
        "No NVIDIA GPU detected. "
        "Set Kaggle Accelerator to T4 before continuing."
    )

print(gpu_info.stdout.strip())

# =============================================================================
# 2. LOAD V8.6.2 SAFETY RUNTIME
# =============================================================================

required_runtime = [
    RUNTIME / "confidence_contract.py",
    RUNTIME / "skill_quarantine.py",
    RUNTIME / "routing_guard.py",
]

for p in required_runtime:
    if not p.exists():
        raise RuntimeError(
            f"Missing V8.6.2 safety runtime: {p}"
        )

sys.path.insert(0, str(RUNTIME))

from confidence_contract import (
    validate_confidence,
    ConfidenceContractError,
)

from skill_quarantine import (
    effective_skill_state,
)

from routing_guard import (
    guard_skill_route,
)

# =============================================================================
# 3. VERIFY THE TARGET IS STILL QUARANTINED
# =============================================================================

skill_path = (
    STATE
    / "skills"
    / f"{INVALID_SKILL_HASH}.json"
)

if not skill_path.exists():
    raise RuntimeError(
        f"Invalidated skill record not found: {skill_path}"
    )

historical_skill = json.loads(
    skill_path.read_text(
        encoding="utf-8-sig"
    )
)

effective_before = effective_skill_state(
    INVALID_SKILL_HASH,
    historical_skill.get("status"),
)

print("\n" + "=" * 122)
print("TARGET SKILL")
print("=" * 122)

print("Skill ID        :", INVALID_SKILL_ID)
print("Stored status   :", historical_skill.get("status"))
print("Effective status:", effective_before.effective_status)
print("Runtime usable  :", effective_before.usable)

if effective_before.usable:
    raise RuntimeError(
        "Safety failure: invalidated skill is currently usable."
    )

# =============================================================================
# 4. READ MODEL PROFILE
# =============================================================================

profile_dir = STATE / "model-profiles"

profiles = sorted(
    profile_dir.glob("*.json")
) if profile_dir.exists() else []

model_profiles = []

for p in profiles:
    try:
        obj = json.loads(
            p.read_text(
                encoding="utf-8-sig"
            )
        )
        model_profiles.append({
            "file": str(p),
            "data": obj,
        })
    except Exception:
        pass

print("\n" + "=" * 122)
print("RAIOS MODEL PROFILE")
print("=" * 122)

if not model_profiles:
    print("No model profile found.")
else:
    for profile in model_profiles:
        print("\nPROFILE FILE:")
        print(profile["file"])
        print(
            json.dumps(
                profile["data"],
                ensure_ascii=False,
                indent=2,
            )[:5000]
        )

# =============================================================================
# 5. DISCOVER LOCAL MODEL ASSETS
#
# IMPORTANT:
# No download is performed.
# We first reuse an existing Kaggle model/dataset asset.
# =============================================================================

print("\n" + "=" * 122)
print("LOCAL MODEL ASSET DISCOVERY")
print("=" * 122)

SEARCH_ROOTS = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
]

config_candidates = []

for root in SEARCH_ROOTS:
    if not root.exists():
        continue

    for p in root.rglob("config.json"):

        low = str(p).lower()

        if any(
            blocked in low
            for blocked in [
                "node_modules",
                ".git",
                ".next",
            ]
        ):
            continue

        parent = p.parent

        has_tokenizer = any([
            (parent / "tokenizer.json").exists(),
            (parent / "tokenizer_config.json").exists(),
            (parent / "tokenizer.model").exists(),
        ])

        has_weights = (
            any(parent.glob("*.safetensors"))
            or any(parent.glob("pytorch_model*.bin"))
            or any(parent.glob("model*.safetensors"))
        )

        if has_tokenizer and has_weights:
            weight_bytes = sum(
                x.stat().st_size
                for pattern in [
                    "*.safetensors",
                    "pytorch_model*.bin",
                ]
                for x in parent.glob(pattern)
                if x.is_file()
            )

            config_candidates.append({
                "path": parent,
                "weight_bytes": weight_bytes,
            })

# Deduplicate
seen = set()
unique_candidates = []

for item in config_candidates:
    key = str(item["path"])

    if key in seen:
        continue

    seen.add(key)
    unique_candidates.append(item)

unique_candidates.sort(
    key=lambda x: x["weight_bytes"],
    reverse=True,
)

for i, item in enumerate(
    unique_candidates[:20],
    start=1,
):
    print(
        f"{i:02d}.",
        item["path"],
        "|",
        round(
            item["weight_bytes"]
            / (1024**3),
            3,
        ),
        "GiB",
    )

# =============================================================================
# 6. FAIL FAST IF MODEL ASSET IS ABSENT
# =============================================================================

if not unique_candidates:

    stop_report = {
        "schema":
            "raios.v8.6.2-r3.model-discovery.v1",

        "status":
            "LOCAL_MODEL_ASSET_NOT_FOUND",

        "gpu_detected":
            gpu_info.stdout.strip(),

        "model_profiles":
            model_profiles,

        "training":
            False,

        "promotion":
            False,

        "input_mutated":
            False,
    }

    out = REPORTS / "v8.6.2-r3-model-discovery.json"

    out.write_text(
        json.dumps(
            stop_report,
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

    print("\nSTATUS: LOCAL_MODEL_ASSET_NOT_FOUND")
    print("No model was downloaded.")
    print("No replay was started.")
    print("No GPU-heavy work was performed.")
    print("\nOUTPUT:")
    print(out)

    raise SystemExit(0)

# =============================================================================
# 7. SELECT LOCAL MODEL
#
# Prefer the largest complete local model asset.
# No network download.
# =============================================================================

MODEL_PATH = unique_candidates[0]["path"]

print("\nSELECTED LOCAL MODEL:")
print(MODEL_PATH)

# =============================================================================
# 8. LOAD TRANSFORMERS
# =============================================================================

print("\n" + "=" * 122)
print("MODEL LOAD")
print("=" * 122)

try:
    import torch
    from transformers import (
        AutoTokenizer,
        AutoModelForCausalLM,
    )
except Exception as exc:
    raise RuntimeError(
        "Required local transformers runtime unavailable."
    ) from exc

load_started = time.perf_counter()

tokenizer = AutoTokenizer.from_pretrained(
    str(MODEL_PATH),
    local_files_only=True,
    trust_remote_code=True,
)

load_kwargs = {
    "local_files_only": True,
    "trust_remote_code": True,
    "device_map": "auto",
}

# Prefer economical precision on T4.
try:
    model = AutoModelForCausalLM.from_pretrained(
        str(MODEL_PATH),
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
        **load_kwargs,
    )
except Exception as first_exc:

    print(
        "[WARN] FP16 load failed; attempting 4-bit local load."
    )
    print(
        str(first_exc)[:1000]
    )

    try:
        from transformers import BitsAndBytesConfig

        quant = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
        )

        model = AutoModelForCausalLM.from_pretrained(
            str(MODEL_PATH),
            quantization_config=quant,
            low_cpu_mem_usage=True,
            **load_kwargs,
        )

    except Exception as second_exc:
        raise RuntimeError(
            "Local model exists but could not be loaded safely "
            "on the available T4."
        ) from second_exc

model.eval()

load_seconds = (
    time.perf_counter()
    - load_started
)

print(
    "Model load seconds:",
    round(load_seconds, 3)
)

# =============================================================================
# 9. BUILD INDEPENDENT REPLAY SET
#
# We reuse historical benchmark/evidence instead of inventing new claims.
# =============================================================================

benchmark_dir = STATE / "benchmarks"

benchmark_files = [
    benchmark_dir
    / "71a1c1740fb813104059011559dbc1ad40ca7e764180a679a28f7e419bfa4e36.json",

    benchmark_dir
    / "059e8506518a5eb4663c202509556e852ee17433d6e1e7d269b13dd3abcfc4d0.json",
]

benchmark_objects = []

for p in benchmark_files:
    if p.exists():
        try:
            benchmark_objects.append({
                "path": str(p),
                "data": json.loads(
                    p.read_text(
                        encoding="utf-8-sig"
                    )
                ),
            })
        except Exception:
            pass

# Also use semantic repository role corpus.
corpus_files = sorted(
    (STATE / "semantic-corpus").glob("*.json")
)

corpus_objects = []

for p in corpus_files:
    try:
        corpus_objects.append({
            "path": str(p),
            "data": json.loads(
                p.read_text(
                    encoding="utf-8-sig"
                )
            ),
        })
    except Exception:
        pass

# =============================================================================
# 10. EXTRACT COMPACT TEST CASES
# =============================================================================

def collect_strings(
    obj,
    prefix="$",
    depth=0,
    max_depth=8,
):
    output = []

    if depth > max_depth:
        return output

    if isinstance(obj, dict):
        for k, v in obj.items():
            output.extend(
                collect_strings(
                    v,
                    f"{prefix}.{k}",
                    depth + 1,
                    max_depth,
                )
            )

    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            output.extend(
                collect_strings(
                    v,
                    f"{prefix}[{i}]",
                    depth + 1,
                    max_depth,
                )
            )

    elif isinstance(obj, str):
        text = obj.strip()

        if 15 <= len(text) <= 4000:
            output.append(
                (prefix, text)
            )

    return output


source_strings = []

for obj in benchmark_objects + corpus_objects:
    for path, text in collect_strings(
        obj["data"]
    ):
        source_strings.append({
            "source": obj["path"],
            "json_path": path,
            "text": text,
        })

# Prefer text that looks like repository-analysis evidence.
priority_terms = [
    "repository",
    "component",
    "file",
    "intelligence",
    "canonical",
    "runtime",
    "routing",
    "evidence",
    "role",
    "agent",
]

for item in source_strings:
    low = item["text"].lower()

    item["priority"] = sum(
        term in low
        for term in priority_terms
    )

source_strings.sort(
    key=lambda x: (
        -x["priority"],
        -len(x["text"]),
    )
)

# Deduplicate texts.
seen_text = set()
selected_evidence = []

for item in source_strings:

    normalized = re.sub(
        r"\s+",
        " ",
        item["text"],
    ).strip()

    key = normalized.lower()

    if key in seen_text:
        continue

    seen_text.add(key)

    selected_evidence.append(item)

    if len(selected_evidence) >= 5:
        break

if len(selected_evidence) < 5:
    raise RuntimeError(
        "Could not derive 5 independent replay evidence cases."
    )

# =============================================================================
# 11. REPLAY PROMPT
#
# Critical:
# Model must emit canonical confidence [0,1].
# =============================================================================

SYSTEM_RULE = """
You are performing a bounded repository-analysis replay for RAIOS.

Rules:
1. Use only the supplied evidence.
2. Do not invent repository facts.
3. Return strict JSON only.
4. confidence MUST be a numeric value in the inclusive range [0,1].
5. If evidence is insufficient, use decision="ABSTAIN".
6. unresolved_flags must preserve uncertainty.
7. evidence_refs must name the supplied evidence reference.
"""

def make_prompt(
    case_id,
    evidence,
):
    return f"""
{SYSTEM_RULE}

CASE_ID:
{case_id}

EVIDENCE_REF:
{evidence['source']}::{evidence['json_path']}

EVIDENCE:
{evidence['text']}

TASK:
Classify the repository evidence conservatively.

Return exactly:

{{
  "decision": "SUPPORTED|PARTIAL|ABSTAIN",
  "confidence": 0.0,
  "evidence_refs": ["..."],
  "unresolved_flags": ["..."],
  "summary": "..."
}}
""".strip()

# =============================================================================
# 12. GENERATION + STRICT PARSING
# =============================================================================

def extract_json(text: str):
    text = text.strip()

    # Remove fenced wrappers if model ignored instruction.
    text = re.sub(
        r"^```(?:json)?\s*",
        "",
        text,
        flags=re.I,
    )

    text = re.sub(
        r"\s*```$",
        "",
        text,
    )

    try:
        return json.loads(text)
    except Exception:
        pass

    start = text.find("{")
    end = text.rfind("}")

    if start >= 0 and end > start:
        return json.loads(
            text[start:end + 1]
        )

    raise ValueError(
        "No parseable JSON object."
    )


replays = []

for index, evidence in enumerate(
    selected_evidence,
    start=1,
):

    case_id = f"V862-R3-{index:02d}"

    prompt = make_prompt(
        case_id,
        evidence,
    )

    encoded = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096,
    )

    # Move tensors to model's first device when possible.
    try:
        device = next(
            model.parameters()
        ).device

        encoded = {
            k: v.to(device)
            for k, v in encoded.items()
        }
    except Exception:
        pass

    started = time.perf_counter()

    with torch.inference_mode():
        output = model.generate(
            **encoded,
            max_new_tokens=256,
            do_sample=False,
            temperature=None,
            top_p=None,
            use_cache=True,
        )

    latency = (
        time.perf_counter()
        - started
    )

    generated = output[
        0,
        encoded["input_ids"].shape[1]:
    ]

    raw = tokenizer.decode(
        generated,
        skip_special_tokens=True,
    ).strip()

    replay = {
        "case_id": case_id,
        "evidence_ref":
            f"{evidence['source']}::{evidence['json_path']}",

        "latency_seconds":
            round(latency, 4),

        "raw_output":
            raw,

        "parse_pass":
            False,

        "confidence_pass":
            False,

        "schema_pass":
            False,

        "overall_pass":
            False,
    }

    try:
        parsed = extract_json(raw)

        replay["parsed"] = parsed
        replay["parse_pass"] = True

        required_fields = {
            "decision",
            "confidence",
            "evidence_refs",
            "unresolved_flags",
            "summary",
        }

        schema_ok = (
            isinstance(parsed, dict)
            and required_fields.issubset(parsed.keys())
            and parsed.get("decision")
                in {
                    "SUPPORTED",
                    "PARTIAL",
                    "ABSTAIN",
                }
            and isinstance(
                parsed.get("evidence_refs"),
                list,
            )
            and isinstance(
                parsed.get("unresolved_flags"),
                list,
            )
        )

        replay["schema_pass"] = schema_ok

        try:
            confidence_result = (
                validate_confidence(
                    parsed.get("confidence")
                )
            )

            replay["confidence"] = (
                confidence_result.value
            )

            replay["confidence_pass"] = True

        except ConfidenceContractError as exc:
            replay[
                "confidence_error"
            ] = str(exc)

        expected_ref = (
            replay["evidence_ref"]
        )

        evidence_bound = (
            isinstance(
                parsed.get("evidence_refs"),
                list,
            )
            and len(
                parsed["evidence_refs"]
            ) > 0
        )

        replay["evidence_bound"] = (
            evidence_bound
        )

        replay["overall_pass"] = all([
            replay["parse_pass"],
            replay["schema_pass"],
            replay["confidence_pass"],
            evidence_bound,
        ])

    except Exception as exc:
        replay["parse_error"] = str(exc)

    replays.append(replay)

    print(
        f"{case_id} | "
        f"pass={replay['overall_pass']} | "
        f"confidence={replay.get('confidence')} | "
        f"latency={replay['latency_seconds']}s"
    )

# =============================================================================
# 13. EVALUATE REVALIDATION
# =============================================================================

passed = [
    r
    for r in replays
    if r["overall_pass"]
]

confidence_values = [
    r["confidence"]
    for r in replays
    if r.get("confidence_pass")
]

avg_confidence = (
    sum(confidence_values)
    / len(confidence_values)
    if confidence_values
    else None
)

avg_latency = (
    sum(
        r["latency_seconds"]
        for r in replays
    )
    / len(replays)
)

# Conservative gate:
# 5/5 required for reinstatement candidate.
all_pass = (
    len(replays) == 5
    and len(passed) == 5
)

if all_pass:
    effective_result = (
        "REVALIDATION_CANDIDATE_PASS"
    )
else:
    effective_result = (
        "REVALIDATION_FAIL_KEEP_QUARANTINED"
    )

# =============================================================================
# 14. IMPORTANT — NO AUTOMATIC PROMOTION
# =============================================================================

promotion_allowed = False

report = {
    "schema":
        "raios.v8.6.2-r3.independent-replay.v1",

    "generated_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "model": {
        "path":
            str(MODEL_PATH),

        "load_seconds":
            round(
                load_seconds,
                3,
            ),

        "gpu":
            gpu_info.stdout.strip(),
    },

    "target_skill": {
        "content_hash":
            INVALID_SKILL_HASH,

        "skill_id":
            INVALID_SKILL_ID,

        "historical_status":
            historical_skill.get(
                "status"
            ),

        "effective_status_before":
            effective_before.effective_status,
    },

    "replays":
        replays,

    "metrics": {
        "replays":
            len(replays),

        "passed":
            len(passed),

        "success_rate":
            (
                len(passed)
                / len(replays)
            ),

        "avg_confidence":
            avg_confidence,

        "avg_latency_seconds":
            avg_latency,
    },

    "gate": {
        "required_passes":
            "5/5",

        "confidence_domain":
            "[0,1]",

        "result":
            effective_result,

        "automatic_promotion":
            False,
    },

    "input_mutated":
        False,

    "training":
        False,

    "promotion":
        False,
}

REPORT_PATH = (
    REPORTS
    / "v8.6.2-r3-independent-replay.json"
)

REPORT_PATH.write_text(
    json.dumps(
        report,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

# =============================================================================
# 15. FINAL
# =============================================================================

print("\n" + "=" * 122)
print("R3 INDEPENDENT REPLAY RESULT")
print("=" * 122)

print("Replays             :", len(replays))
print("Passed              :", len(passed))
print(
    "Success rate        :",
    round(
        len(passed)
        / len(replays),
        4,
    )
)

print(
    "Average confidence  :",
    (
        round(avg_confidence, 6)
        if avg_confidence is not None
        else None
    )
)

print(
    "Average latency     :",
    round(
        avg_latency,
        4,
    ),
    "seconds"
)

print("Gate result          :", effective_result)
print("Automatic promotion  :", promotion_allowed)

print("\nOUTPUT:")
print(REPORT_PATH)

print("""
TRAINING        : NO
INPUT MUTATED   : NO
PROMOTED        : NO

IMPORTANT:
Even if the replay passes 5/5,
the skill remains quarantined until the next
explicit promotion/durability gate.
""")

RAIOS V8.6.2-R3 — INDEPENDENT REPLAY REVALIDATION
T4 GATE | NO TRAINING | NO PROMOTION | FAIL CLOSED

GPU GATE
Tesla T4, 15360 MiB, 14912 MiB
Tesla T4, 15360 MiB, 14912 MiB


RuntimeError: Missing V8.6.2 safety runtime: /kaggle/working/RAIOS-V8.6.2/runtime/confidence_contract.py

In [1]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime, timezone
import json
import math
import random
import time

print("=" * 120)
print("RAIOS V8.6.2-R4 — REVALIDATION STRESS + BOUNDARY TEST")
print("GPU REUSE | NO TRAINING | NO PROMOTION")
print("=" * 120)

ROOT = Path("/kaggle/working/RAIOS-V8.6.2")
REPORTS = ROOT / "reports"

R3_REPORT = REPORTS / "v8.6.2-r3-independent-replay.json"

if not R3_REPORT.exists():
    raise RuntimeError("R3 replay report missing.")

r3 = json.loads(
    R3_REPORT.read_text(encoding="utf-8")
)

if r3["gate"]["result"] != "REVALIDATION_CANDIDATE_PASS":
    raise RuntimeError(
        "R4 PASS path blocked: R3 did not pass 5/5."
    )

# model/tokenizer are intentionally reused from the current notebook session.
if "model" not in globals() or "tokenizer" not in globals():
    raise RuntimeError(
        "Loaded model/tokenizer not present in notebook memory."
    )

if "validate_confidence" not in globals():
    raise RuntimeError(
        "Confidence runtime is not loaded."
    )

base_cases = [
    x for x in r3["replays"]
    if x.get("overall_pass")
]

if len(base_cases) != 5:
    raise RuntimeError("Expected exactly 5 valid R3 cases.")

def mutate_case(case, mode):
    raw = case.get("parsed", {})
    evidence = case.get("evidence_ref", "")

    summary = str(raw.get("summary", ""))

    if mode == "NEGATION":
        payload = (
            "The following statement may be false or contradicted. "
            "Do not assume it is true:\n"
            + summary
        )

    elif mode == "MISSING":
        payload = (
            "Evidence is incomplete. Important context is missing:\n"
            + summary[: max(30, len(summary)//2)]
        )

    elif mode == "CONTRADICTION":
        payload = (
            "Two claims conflict:\n"
            f"A: {summary}\n"
            f"B: The opposite interpretation may also be supported.\n"
            "Preserve the conflict."
        )

    elif mode == "NOISE":
        payload = (
            "Irrelevant context: package version, UI wording, timestamp.\n"
            "Relevant claim:\n"
            + summary
        )

    elif mode == "HIGH_RISK":
        payload = (
            "Treat this as a high-risk repository decision. "
            "Do not increase confidence merely because the claim sounds precise.\n"
            + summary
        )

    else:
        raise ValueError(mode)

    return {
        "source": evidence,
        "text": payload,
        "mode": mode,
    }


modes = [
    "NEGATION",
    "MISSING",
    "CONTRADICTION",
    "NOISE",
    "HIGH_RISK",
]

stress_cases = []

for i, base in enumerate(base_cases):
    stress_cases.append(
        mutate_case(base, modes[i])
    )

SYSTEM = """
You are RAIOS performing adversarial repository-analysis validation.

Rules:
- Use only supplied evidence.
- Never invent missing facts.
- confidence MUST be numeric and inside [0,1].
- Contradiction must remain visible.
- Missing evidence should reduce certainty.
- Use ABSTAIN where evidence is insufficient.
- Return JSON only.

Schema:
{
  "decision": "SUPPORTED|PARTIAL|ABSTAIN",
  "confidence": 0.0,
  "evidence_refs": ["..."],
  "unresolved_flags": ["..."],
  "summary": "..."
}
"""

def extract_json(text):
    text = text.strip()

    if text.startswith("```"):
        text = text.replace("```json", "").replace("```", "").strip()

    try:
        return json.loads(text)
    except Exception:
        start = text.find("{")
        end = text.rfind("}")

        if start >= 0 and end > start:
            return json.loads(text[start:end+1])

        raise


results = []

for idx, case in enumerate(stress_cases, 1):

    prompt = f"""
{SYSTEM}

CASE_ID: R4-{idx:02d}
MODE: {case['mode']}
EVIDENCE_REF: {case['source']}

EVIDENCE:
{case['text']}
""".strip()

    encoded = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096,
    )

    try:
        device = next(model.parameters()).device
        encoded = {
            k: v.to(device)
            for k, v in encoded.items()
        }
    except Exception:
        pass

    started = time.perf_counter()

    with torch.inference_mode():
        output = model.generate(
            **encoded,
            max_new_tokens=220,
            do_sample=False,
            use_cache=True,
        )

    latency = time.perf_counter() - started

    generated = output[
        0,
        encoded["input_ids"].shape[1]:
    ]

    text = tokenizer.decode(
        generated,
        skip_special_tokens=True,
    ).strip()

    row = {
        "case": f"R4-{idx:02d}",
        "mode": case["mode"],
        "latency": latency,
        "raw": text,
        "pass": False,
    }

    try:
        parsed = extract_json(text)

        conf = validate_confidence(
            parsed.get("confidence")
        ).value

        decision = parsed.get("decision")

        flags = parsed.get(
            "unresolved_flags",
            []
        )

        # Stress-specific expectations.
        conservative = True

        if case["mode"] in {
            "MISSING",
            "CONTRADICTION",
            "HIGH_RISK",
        }:
            conservative = (
                decision in {"PARTIAL", "ABSTAIN"}
                or len(flags) > 0
                or conf <= 0.80
            )

        row.update({
            "parsed": parsed,
            "confidence": conf,
            "conservative": conservative,
            "pass": conservative,
        })

    except Exception as exc:
        row["error"] = str(exc)

    results.append(row)

    print(
        row["case"],
        "|",
        row["mode"],
        "| pass=",
        row["pass"],
        "| confidence=",
        row.get("confidence"),
        "| latency=",
        round(latency, 3),
    )

passed = sum(
    r["pass"]
    for r in results
)

status = (
    "R4_STRESS_PASS"
    if passed == len(results)
    else "R4_STRESS_FAIL"
)

report = {
    "schema": "raios.v8.6.2-r4.stress.v1",
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "results": results,
    "summary": {
        "total": len(results),
        "passed": passed,
        "failed": len(results) - passed,
    },
    "status": status,
    "training": False,
    "promotion": False,
}

out = REPORTS / "v8.6.2-r4-stress.json"

out.write_text(
    json.dumps(
        report,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("\n" + "=" * 120)
print("R4 RESULT")
print("=" * 120)
print("Passed :", passed)
print("Failed :", len(results) - passed)
print("STATUS :", status)
print("OUTPUT :", out)

RAIOS V8.6.2-R4 — REVALIDATION STRESS + BOUNDARY TEST
GPU REUSE | NO TRAINING | NO PROMOTION


RuntimeError: R3 replay report missing.

In [2]:
from __future__ import annotations

from pathlib import Path
import hashlib
import json
import sys

print("=" * 118)
print("RAIOS V8.6.2 — EMERGENCY RUNTIME REHYDRATION")
print("RESTORE EPHEMERAL /kaggle/working RUNTIME | NO MODEL | NO TRAINING")
print("=" * 118)

ROOT = Path("/kaggle/working/RAIOS-V8.6.2")
RUNTIME = ROOT / "runtime"
REPORTS = ROOT / "reports"

RUNTIME.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

INVALID_SKILL_HASH = (
    "683ca22e97a2cb919ab078321076bd06"
    "b146b1bf69e937a2062d68755e993b65"
)

INVALID_SKILL_ID = "RAIOS.REPOSITORY_ANALYSIS.MICRO.v1"

# =============================================================================
# 1. CONFIDENCE CONTRACT
# =============================================================================

confidence_runtime = r'''
from __future__ import annotations

from dataclasses import dataclass
from decimal import Decimal, InvalidOperation
import math
from typing import Any


class ConfidenceContractError(ValueError):
    pass


@dataclass(frozen=True)
class ConfidenceResult:
    value: float
    original: Any
    normalized: bool
    source_type: str


def validate_confidence(value: Any) -> ConfidenceResult:
    """
    RAIOS canonical confidence domain: [0.0, 1.0].

    Safety rules:
    - Never silently convert 70 -> 0.70.
    - Percentage-scale numeric values are invalid.
    - Safe string decimals such as "0.70" may be parsed.
    - bool is invalid.
    - NaN/Inf are invalid.
    """

    if isinstance(value, bool):
        raise ConfidenceContractError(
            "Boolean is not a valid confidence value."
        )

    original = value
    normalized = False

    if isinstance(value, str):
        stripped = value.strip()

        if not stripped:
            raise ConfidenceContractError(
                "Empty confidence string."
            )

        try:
            value = Decimal(stripped)
        except InvalidOperation as exc:
            raise ConfidenceContractError(
                f"Confidence is not numeric: {original!r}"
            ) from exc

        normalized = True

    elif isinstance(value, Decimal):
        pass

    elif isinstance(value, (int, float)):
        if isinstance(value, float) and not math.isfinite(value):
            raise ConfidenceContractError(
                f"Confidence must be finite: {value!r}"
            )

        value = Decimal(str(value))

    else:
        raise ConfidenceContractError(
            f"Unsupported confidence type: {type(original).__name__}"
        )

    if not value.is_finite():
        raise ConfidenceContractError(
            f"Confidence must be finite: {original!r}"
        )

    if value < Decimal("0") or value > Decimal("1"):
        raise ConfidenceContractError(
            "Confidence violates canonical [0,1] domain: "
            f"{original!r}"
        )

    return ConfidenceResult(
        value=float(value),
        original=original,
        normalized=normalized,
        source_type=type(original).__name__,
    )


def require_confidence(value: Any) -> float:
    return validate_confidence(value).value
'''

(RUNTIME / "confidence_contract.py").write_text(
    confidence_runtime,
    encoding="utf-8",
)

# =============================================================================
# 2. SKILL QUARANTINE
# =============================================================================

quarantine_runtime = f'''
from __future__ import annotations

from dataclasses import dataclass


INVALIDATED_SKILLS = {{
    "{INVALID_SKILL_HASH}": {{
        "skill_id": "{INVALID_SKILL_ID}",
        "effective_status": "QUARANTINED_PENDING_REVALIDATION",
        "reason": "Historical promotion depended on invalid confidence scale.",
        "required_gate": "INDEPENDENT_REPLAY_REVALIDATION",
    }}
}}


@dataclass(frozen=True)
class EffectiveSkillState:
    content_hash: str
    stored_status: str | None
    effective_status: str
    usable: bool
    reason: str | None


def effective_skill_state(
    content_hash: str,
    stored_status: str | None,
) -> EffectiveSkillState:

    quarantine = INVALIDATED_SKILLS.get(content_hash)

    if quarantine:
        return EffectiveSkillState(
            content_hash=content_hash,
            stored_status=stored_status,
            effective_status=quarantine["effective_status"],
            usable=False,
            reason=quarantine["reason"],
        )

    return EffectiveSkillState(
        content_hash=content_hash,
        stored_status=stored_status,
        effective_status=stored_status or "UNKNOWN",
        usable=(stored_status == "PROMOTED"),
        reason=None,
    )
'''

(RUNTIME / "skill_quarantine.py").write_text(
    quarantine_runtime,
    encoding="utf-8",
)

# =============================================================================
# 3. ROUTING GUARD
# =============================================================================

routing_runtime = r'''
from __future__ import annotations

from dataclasses import dataclass

from skill_quarantine import effective_skill_state


@dataclass(frozen=True)
class RoutingDecision:
    requested_route: str
    effective_route: str
    allowed: bool
    reason: str
    skill_hash: str | None


SAFE_FALLBACK = "REASONING"


def guard_skill_route(
    *,
    requested_route: str,
    skill_hash: str | None,
    stored_skill_status: str | None,
) -> RoutingDecision:

    route = requested_route.strip().upper()

    if skill_hash is None:
        return RoutingDecision(
            requested_route=route,
            effective_route=route,
            allowed=True,
            reason="NO_SKILL_BOUND_ROUTE",
            skill_hash=None,
        )

    state = effective_skill_state(
        content_hash=skill_hash,
        stored_status=stored_skill_status,
    )

    if not state.usable:
        return RoutingDecision(
            requested_route=route,
            effective_route=SAFE_FALLBACK,
            allowed=False,
            reason=(
                "SKILL_NOT_RUNTIME_USABLE:"
                + state.effective_status
            ),
            skill_hash=skill_hash,
        )

    return RoutingDecision(
        requested_route=route,
        effective_route=route,
        allowed=True,
        reason="SKILL_RUNTIME_USABLE",
        skill_hash=skill_hash,
    )
'''

(RUNTIME / "routing_guard.py").write_text(
    routing_runtime,
    encoding="utf-8",
)

# =============================================================================
# 4. HASH + IMPORT VALIDATION
# =============================================================================

expected_hashes = {
    "confidence_contract.py":
        "d5306df6c42a52cb2937b458584575174ddfc52ee488b121a437a240ac346f85",

    "skill_quarantine.py":
        "883f438b7aa832c051ae83222bdd3ceae736d2b8e05abc9067e92b4cecb2514a",

    "routing_guard.py":
        "fbda164c233b39ec25b60d59f5b8b45cc2c5f60604149e5aa646993d4d130c20",
}

results = {}

for name, expected in expected_hashes.items():

    path = RUNTIME / name

    actual = hashlib.sha256(
        path.read_bytes()
    ).hexdigest()

    results[name] = {
        "expected": expected,
        "actual": actual,
        "match": actual == expected,
    }

    print(
        f"{name:28}",
        "PASS" if actual == expected else "HASH_CHANGED",
        actual,
    )

# Hash mismatch does not automatically mean unsafe because textual regeneration
# can differ in whitespace. Functional validation below is authoritative.

sys.path.insert(0, str(RUNTIME))

from confidence_contract import (
    validate_confidence,
    ConfidenceContractError,
)

from skill_quarantine import (
    effective_skill_state,
)

from routing_guard import (
    guard_skill_route,
)

# =============================================================================
# 5. FAST FUNCTIONAL SAFETY GATE
# =============================================================================

functional = []

def check(name, condition, detail):
    functional.append({
        "name": name,
        "pass": bool(condition),
        "detail": str(detail),
    })

# confidence
check(
    "accept_0_70",
    validate_confidence(0.70).value == 0.70,
    "0.70",
)

for invalid in [70, 14.79, -0.1, 1.1, True, None]:
    try:
        validate_confidence(invalid)
        check(
            f"reject_{invalid!r}",
            False,
            "unexpected acceptance",
        )
    except ConfidenceContractError as exc:
        check(
            f"reject_{invalid!r}",
            True,
            exc,
        )

# quarantine
effective = effective_skill_state(
    INVALID_SKILL_HASH,
    "PROMOTED",
)

check(
    "quarantine_effective",
    (
        effective.usable is False
        and
        effective.effective_status
        == "QUARANTINED_PENDING_REVALIDATION"
    ),
    effective,
)

# routing
route = guard_skill_route(
    requested_route="ZERO_LLM",
    skill_hash=INVALID_SKILL_HASH,
    stored_skill_status="PROMOTED",
)

check(
    "zero_llm_blocked",
    (
        route.allowed is False
        and route.effective_route == "REASONING"
    ),
    route,
)

failed = [
    item
    for item in functional
    if not item["pass"]
]

print("\nFUNCTIONAL SAFETY")

for item in functional:
    print(
        "PASS" if item["pass"] else "FAIL",
        "|",
        item["name"],
        "|",
        item["detail"],
    )

if failed:
    raise RuntimeError(
        f"Runtime rehydration failed {len(failed)} functional tests."
    )

# =============================================================================
# 6. WRITE BOOTSTRAP RECEIPT
# =============================================================================

receipt = {
    "schema":
        "raios.v8.6.2.ephemeral-runtime-rehydration.v1",

    "runtime_root":
        str(RUNTIME),

    "hashes":
        results,

    "functional_tests":
        functional,

    "status":
        "RUNTIME_REHYDRATED",

    "training":
        False,

    "model_loaded":
        False,

    "input_mutated":
        False,
}

receipt_path = (
    REPORTS
    / "runtime-rehydration-receipt.json"
)

receipt_path.write_text(
    json.dumps(
        receipt,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

print("\n" + "=" * 118)
print("RUNTIME REHYDRATION RESULT")
print("=" * 118)

print("Runtime root :", RUNTIME)
print("Files        : 3")
print(
    "Functional  :",
    len(functional) - len(failed),
    "/",
    len(functional),
    "PASS",
)
print("STATUS       : RUNTIME_REHYDRATED")
print("REPORT       :", receipt_path)

print("""
MODEL LOADED : NO
TRAINING     : NO
GPU WORK     : NEGLIGIBLE

NEXT:
RERUN THE R3 INDEPENDENT REPLAY CELL IMMEDIATELY.
DO NOT RESTART THE SESSION.
""")

RAIOS V8.6.2 — EMERGENCY RUNTIME REHYDRATION
RESTORE EPHEMERAL /kaggle/working RUNTIME | NO MODEL | NO TRAINING
confidence_contract.py       HASH_CHANGED cdbf2d78c2c455224b40e37a00c9edeb3b2ef7d898a320c7eb60389e11c09f9c
skill_quarantine.py          HASH_CHANGED cc44e094562236d1798e2a1cb744bd82cb203cff00dd6678aad6267d0aa5daac
routing_guard.py             HASH_CHANGED c18fc30a8c36da60885dfc0f61117033edce04109862b2de2ff470e37732114d

FUNCTIONAL SAFETY
PASS | accept_0_70 | 0.70
PASS | reject_70 | Confidence violates canonical [0,1] domain: 70
PASS | reject_14.79 | Confidence violates canonical [0,1] domain: 14.79
PASS | reject_-0.1 | Confidence violates canonical [0,1] domain: -0.1
PASS | reject_1.1 | Confidence violates canonical [0,1] domain: 1.1
PASS | reject_True | Boolean is not a valid confidence value.
PASS | reject_None | Unsupported confidence type: NoneType
PASS | quarantine_effective | EffectiveSkillState(content_hash='683ca22e97a2cb919ab078321076bd06b146b1bf69e937a2062d68755e993

In [1]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime, timezone
import json
import re
import sys
import time
import subprocess

print("=" * 122)
print("RAIOS V8.6.2-R3 — INDEPENDENT REPLAY REVALIDATION")
print("T4 GATE | NO TRAINING | NO PROMOTION | FAIL CLOSED")
print("=" * 122)

# =============================================================================
# PATHS
# =============================================================================

INPUT = Path("/kaggle/input/datasets/greenylife")

RAIOS_DS = INPUT / "raios-cognitive-state"

RAIOS_ROOT = (
    RAIOS_DS
    / "RAIOS-STATE-LATEST"
    / "RAIOS"
)

FACTORY = (
    RAIOS_ROOT
    / "raios-cognitive-factory"
)

STATE = FACTORY / "state"

V862 = Path("/kaggle/working/RAIOS-V8.6.2")
RUNTIME = V862 / "runtime"
REPORTS = V862 / "reports"
R3 = V862 / "r3-replay"

REPORTS.mkdir(parents=True, exist_ok=True)
R3.mkdir(parents=True, exist_ok=True)

INVALID_SKILL_HASH = (
    "683ca22e97a2cb919ab078321076bd06"
    "b146b1bf69e937a2062d68755e993b65"
)

INVALID_SKILL_ID = "RAIOS.REPOSITORY_ANALYSIS.MICRO.v1"

# =============================================================================
# 1. VERIFY GPU
# =============================================================================

print("\n" + "=" * 122)
print("GPU GATE")
print("=" * 122)

gpu_info = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total,memory.free",
        "--format=csv,noheader",
    ],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

if gpu_info.returncode != 0:
    print(gpu_info.stdout)
    raise RuntimeError(
        "No NVIDIA GPU detected. "
        "Set Kaggle Accelerator to T4 before continuing."
    )

print(gpu_info.stdout.strip())

# =============================================================================
# 2. LOAD V8.6.2 SAFETY RUNTIME
# =============================================================================

required_runtime = [
    RUNTIME / "confidence_contract.py",
    RUNTIME / "skill_quarantine.py",
    RUNTIME / "routing_guard.py",
]

for p in required_runtime:
    if not p.exists():
        raise RuntimeError(
            f"Missing V8.6.2 safety runtime: {p}"
        )

sys.path.insert(0, str(RUNTIME))

from confidence_contract import (
    validate_confidence,
    ConfidenceContractError,
)

from skill_quarantine import (
    effective_skill_state,
)

from routing_guard import (
    guard_skill_route,
)

# =============================================================================
# 3. VERIFY TARGET SKILL IS STILL QUARANTINED
# =============================================================================

skill_path = (
    STATE
    / "skills"
    / f"{INVALID_SKILL_HASH}.json"
)

if not skill_path.exists():
    raise RuntimeError(
        f"Invalidated skill record not found: {skill_path}"
    )

historical_skill = json.loads(
    skill_path.read_text(
        encoding="utf-8-sig"
    )
)

effective_before = effective_skill_state(
    INVALID_SKILL_HASH,
    historical_skill.get("status"),
)

print("\n" + "=" * 122)
print("TARGET SKILL")
print("=" * 122)

print("Skill ID        :", INVALID_SKILL_ID)
print("Stored status   :", historical_skill.get("status"))
print("Effective status:", effective_before.effective_status)
print("Runtime usable  :", effective_before.usable)

if effective_before.usable:
    raise RuntimeError(
        "Safety failure: invalidated skill is currently usable."
    )

# =============================================================================
# 4. READ RAIOS MODEL PROFILE
# =============================================================================

profile_dir = STATE / "model-profiles"

profiles = (
    sorted(profile_dir.glob("*.json"))
    if profile_dir.exists()
    else []
)

model_profiles = []

for p in profiles:
    try:
        obj = json.loads(
            p.read_text(
                encoding="utf-8-sig"
            )
        )
        model_profiles.append({
            "file": str(p),
            "data": obj,
        })
    except Exception:
        pass

print("\n" + "=" * 122)
print("RAIOS MODEL PROFILE")
print("=" * 122)

if not model_profiles:
    print("No model profile found.")
else:
    for profile in model_profiles:
        print("\nPROFILE FILE:")
        print(profile["file"])
        print(
            json.dumps(
                profile["data"],
                ensure_ascii=False,
                indent=2,
            )[:5000]
        )

# =============================================================================
# 5. DISCOVER LOCAL MODEL ASSETS
#
# IMPORTANT:
# No download is performed.
# First reuse existing Kaggle model/dataset assets.
# =============================================================================

print("\n" + "=" * 122)
print("LOCAL MODEL ASSET DISCOVERY")
print("=" * 122)

SEARCH_ROOTS = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
]

config_candidates = []

for root in SEARCH_ROOTS:
    if not root.exists():
        continue

    for p in root.rglob("config.json"):

        low = str(p).lower()

        if any(
            blocked in low
            for blocked in [
                "node_modules",
                ".git",
                ".next",
            ]
        ):
            continue

        parent = p.parent

        has_tokenizer = any([
            (parent / "tokenizer.json").exists(),
            (parent / "tokenizer_config.json").exists(),
            (parent / "tokenizer.model").exists(),
        ])

        weight_files = []

        for pattern in [
            "*.safetensors",
            "pytorch_model*.bin",
        ]:
            weight_files.extend(
                [
                    x
                    for x in parent.glob(pattern)
                    if x.is_file()
                ]
            )

        has_weights = bool(weight_files)

        if has_tokenizer and has_weights:

            weight_bytes = sum(
                x.stat().st_size
                for x in weight_files
            )

            config_candidates.append({
                "path": parent,
                "weight_bytes": weight_bytes,
            })

# Deduplicate
seen = set()
unique_candidates = []

for item in config_candidates:
    key = str(item["path"])

    if key in seen:
        continue

    seen.add(key)
    unique_candidates.append(item)

unique_candidates.sort(
    key=lambda x: x["weight_bytes"],
    reverse=True,
)

for i, item in enumerate(
    unique_candidates[:20],
    start=1,
):
    print(
        f"{i:02d}.",
        item["path"],
        "|",
        round(
            item["weight_bytes"] / (1024**3),
            3,
        ),
        "GiB",
    )

# =============================================================================
# 6. FAIL FAST IF NO LOCAL MODEL EXISTS
# =============================================================================

if not unique_candidates:

    stop_report = {
        "schema":
            "raios.v8.6.2-r3.model-discovery.v1",

        "status":
            "LOCAL_MODEL_ASSET_NOT_FOUND",

        "gpu_detected":
            gpu_info.stdout.strip(),

        "model_profiles":
            model_profiles,

        "training":
            False,

        "promotion":
            False,

        "input_mutated":
            False,
    }

    out = (
        REPORTS
        / "v8.6.2-r3-model-discovery.json"
    )

    out.write_text(
        json.dumps(
            stop_report,
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

    print("\nSTATUS: LOCAL_MODEL_ASSET_NOT_FOUND")
    print("No model was downloaded.")
    print("No replay was started.")
    print("No GPU-heavy work was performed.")
    print("\nOUTPUT:")
    print(out)

    raise SystemExit(0)

# =============================================================================
# 7. SELECT LOCAL MODEL
#
# Prefer largest complete local model asset.
# =============================================================================

MODEL_PATH = unique_candidates[0]["path"]

print("\nSELECTED LOCAL MODEL:")
print(MODEL_PATH)

# =============================================================================
# 8. LOAD TRANSFORMERS MODEL
# =============================================================================

print("\n" + "=" * 122)
print("MODEL LOAD")
print("=" * 122)

try:
    import torch

    from transformers import (
        AutoTokenizer,
        AutoModelForCausalLM,
    )

except Exception as exc:
    raise RuntimeError(
        "Required transformers runtime unavailable."
    ) from exc

load_started = time.perf_counter()

tokenizer = AutoTokenizer.from_pretrained(
    str(MODEL_PATH),
    local_files_only=True,
    trust_remote_code=True,
)

load_kwargs = {
    "local_files_only": True,
    "trust_remote_code": True,
    "device_map": "auto",
}

try:
    model = AutoModelForCausalLM.from_pretrained(
        str(MODEL_PATH),
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
        **load_kwargs,
    )

except Exception as first_exc:

    print(
        "[WARN] FP16 load failed; "
        "attempting 4-bit local load."
    )

    print(
        str(first_exc)[:1500]
    )

    try:
        from transformers import (
            BitsAndBytesConfig,
        )

        quant = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
        )

        model = (
            AutoModelForCausalLM
            .from_pretrained(
                str(MODEL_PATH),
                quantization_config=quant,
                low_cpu_mem_usage=True,
                **load_kwargs,
            )
        )

    except Exception as second_exc:
        raise RuntimeError(
            "Local model exists but could not "
            "be loaded safely on T4."
        ) from second_exc

model.eval()

load_seconds = (
    time.perf_counter()
    - load_started
)

print(
    "Model load seconds:",
    round(load_seconds, 3)
)

# =============================================================================
# 9. BUILD INDEPENDENT REPLAY SOURCE SET
# =============================================================================

benchmark_dir = STATE / "benchmarks"

benchmark_files = [
    benchmark_dir
    / "71a1c1740fb813104059011559dbc1ad40ca7e764180a679a28f7e419bfa4e36.json",

    benchmark_dir
    / "059e8506518a5eb4663c202509556e852ee17433d6e1e7d269b13dd3abcfc4d0.json",
]

benchmark_objects = []

for p in benchmark_files:
    if p.exists():
        try:
            benchmark_objects.append({
                "path": str(p),
                "data": json.loads(
                    p.read_text(
                        encoding="utf-8-sig"
                    )
                ),
            })
        except Exception:
            pass

corpus_files = sorted(
    (STATE / "semantic-corpus").glob("*.json")
)

corpus_objects = []

for p in corpus_files:
    try:
        corpus_objects.append({
            "path": str(p),
            "data": json.loads(
                p.read_text(
                    encoding="utf-8-sig"
                )
            ),
        })
    except Exception:
        pass

# =============================================================================
# 10. EXTRACT COMPACT EVIDENCE CASES
# =============================================================================

def collect_strings(
    obj,
    prefix="$",
    depth=0,
    max_depth=8,
):
    output = []

    if depth > max_depth:
        return output

    if isinstance(obj, dict):

        for k, v in obj.items():
            output.extend(
                collect_strings(
                    v,
                    f"{prefix}.{k}",
                    depth + 1,
                    max_depth,
                )
            )

    elif isinstance(obj, list):

        for i, v in enumerate(obj):
            output.extend(
                collect_strings(
                    v,
                    f"{prefix}[{i}]",
                    depth + 1,
                    max_depth,
                )
            )

    elif isinstance(obj, str):

        text = obj.strip()

        if 15 <= len(text) <= 4000:
            output.append(
                (prefix, text)
            )

    return output


source_strings = []

for obj in benchmark_objects + corpus_objects:

    for json_path, text in collect_strings(
        obj["data"]
    ):

        source_strings.append({
            "source": obj["path"],
            "json_path": json_path,
            "text": text,
        })

priority_terms = [
    "repository",
    "component",
    "file",
    "intelligence",
    "canonical",
    "runtime",
    "routing",
    "evidence",
    "role",
    "agent",
]

for item in source_strings:

    low = item["text"].lower()

    item["priority"] = sum(
        term in low
        for term in priority_terms
    )

source_strings.sort(
    key=lambda x: (
        -x["priority"],
        -len(x["text"]),
    )
)

seen_text = set()
selected_evidence = []

for item in source_strings:

    normalized = re.sub(
        r"\s+",
        " ",
        item["text"],
    ).strip()

    key = normalized.lower()

    if key in seen_text:
        continue

    seen_text.add(key)
    selected_evidence.append(item)

    if len(selected_evidence) >= 5:
        break

if len(selected_evidence) < 5:
    raise RuntimeError(
        "Could not derive 5 independent replay evidence cases."
    )

print("\nREPLAY CASES SELECTED:", len(selected_evidence))

for i, item in enumerate(
    selected_evidence,
    1,
):
    print(
        f"{i}.",
        item["source"],
        "::",
        item["json_path"],
    )

# =============================================================================
# 11. BOUNDED REPLAY PROMPT
# =============================================================================

SYSTEM_RULE = """
You are performing a bounded repository-analysis replay for RAIOS.

Rules:
1. Use only the supplied evidence.
2. Do not invent repository facts.
3. Return strict JSON only.
4. confidence MUST be numeric and in the inclusive range [0,1].
5. If evidence is insufficient, use decision="ABSTAIN".
6. unresolved_flags must preserve uncertainty.
7. evidence_refs must explicitly bind the supplied evidence.
8. Do not convert percentage-style confidence such as 70 into 0.70.
"""

def make_prompt(
    case_id,
    evidence,
):

    evidence_ref = (
        f"{evidence['source']}"
        f"::{evidence['json_path']}"
    )

    return f"""
{SYSTEM_RULE}

CASE_ID:
{case_id}

EVIDENCE_REF:
{evidence_ref}

EVIDENCE:
{evidence['text']}

TASK:
Classify the repository evidence conservatively.

Return exactly this JSON schema:

{{
  "decision": "SUPPORTED|PARTIAL|ABSTAIN",
  "confidence": 0.0,
  "evidence_refs": ["{evidence_ref}"],
  "unresolved_flags": ["..."],
  "summary": "..."
}}
""".strip()

# =============================================================================
# 12. STRICT JSON PARSER
# =============================================================================

def extract_json(text: str):

    text = text.strip()

    text = re.sub(
        r"^```(?:json)?\s*",
        "",
        text,
        flags=re.I,
    )

    text = re.sub(
        r"\s*```$",
        "",
        text,
    )

    try:
        return json.loads(text)
    except Exception:
        pass

    start = text.find("{")
    end = text.rfind("}")

    if start >= 0 and end > start:
        return json.loads(
            text[start:end + 1]
        )

    raise ValueError(
        "No parseable JSON object."
    )

# =============================================================================
# 13. RUN 5 INDEPENDENT REPLAYS
# =============================================================================

replays = []

for index, evidence in enumerate(
    selected_evidence,
    start=1,
):

    case_id = f"V862-R3-{index:02d}"

    evidence_ref = (
        f"{evidence['source']}"
        f"::{evidence['json_path']}"
    )

    prompt = make_prompt(
        case_id,
        evidence,
    )

    encoded = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096,
    )

    try:
        device = next(
            model.parameters()
        ).device

        encoded = {
            k: v.to(device)
            for k, v in encoded.items()
        }

    except Exception:
        pass

    started = time.perf_counter()

    with torch.inference_mode():

        output = model.generate(
            **encoded,
            max_new_tokens=256,
            do_sample=False,
            use_cache=True,
        )

    latency = (
        time.perf_counter()
        - started
    )

    generated = output[
        0,
        encoded["input_ids"].shape[1]:
    ]

    raw = tokenizer.decode(
        generated,
        skip_special_tokens=True,
    ).strip()

    replay = {
        "case_id": case_id,

        "evidence_ref":
            evidence_ref,

        "latency_seconds":
            round(latency, 4),

        "raw_output":
            raw,

        "parse_pass":
            False,

        "confidence_pass":
            False,

        "schema_pass":
            False,

        "evidence_bound":
            False,

        "overall_pass":
            False,
    }

    try:

        parsed = extract_json(raw)

        replay["parsed"] = parsed
        replay["parse_pass"] = True

        required_fields = {
            "decision",
            "confidence",
            "evidence_refs",
            "unresolved_flags",
            "summary",
        }

        schema_ok = (
            isinstance(parsed, dict)
            and required_fields.issubset(
                parsed.keys()
            )
            and parsed.get("decision")
            in {
                "SUPPORTED",
                "PARTIAL",
                "ABSTAIN",
            }
            and isinstance(
                parsed.get("evidence_refs"),
                list,
            )
            and isinstance(
                parsed.get("unresolved_flags"),
                list,
            )
            and isinstance(
                parsed.get("summary"),
                str,
            )
        )

        replay["schema_pass"] = schema_ok

        try:

            confidence_result = (
                validate_confidence(
                    parsed.get(
                        "confidence"
                    )
                )
            )

            replay["confidence"] = (
                confidence_result.value
            )

            replay["confidence_pass"] = True

        except ConfidenceContractError as exc:

            replay[
                "confidence_error"
            ] = str(exc)

        evidence_refs = parsed.get(
            "evidence_refs",
            []
        )

        replay["evidence_bound"] = (
            isinstance(
                evidence_refs,
                list,
            )
            and evidence_ref
            in evidence_refs
        )

        replay["overall_pass"] = all([
            replay["parse_pass"],
            replay["schema_pass"],
            replay["confidence_pass"],
            replay["evidence_bound"],
        ])

    except Exception as exc:

        replay[
            "parse_error"
        ] = str(exc)

    replays.append(replay)

    print(
        f"{case_id} | "
        f"pass={replay['overall_pass']} | "
        f"parse={replay['parse_pass']} | "
        f"schema={replay['schema_pass']} | "
        f"confidence={replay.get('confidence')} | "
        f"bound={replay['evidence_bound']} | "
        f"latency={replay['latency_seconds']}s"
    )

# =============================================================================
# 14. REVALIDATION GATE
# =============================================================================

passed = [
    r
    for r in replays
    if r["overall_pass"]
]

confidence_values = [
    r["confidence"]
    for r in replays
    if r.get("confidence_pass")
]

avg_confidence = (
    sum(confidence_values)
    / len(confidence_values)
    if confidence_values
    else None
)

avg_latency = (
    sum(
        r["latency_seconds"]
        for r in replays
    )
    / len(replays)
)

success_rate = (
    len(passed)
    / len(replays)
)

# Strict reinstatement candidate gate
all_pass = (
    len(replays) == 5
    and len(passed) == 5
)

if all_pass:

    effective_result = (
        "REVALIDATION_CANDIDATE_PASS"
    )

else:

    effective_result = (
        "REVALIDATION_FAIL_KEEP_QUARANTINED"
    )

# =============================================================================
# 15. ABSOLUTELY NO AUTOMATIC PROMOTION
# =============================================================================

promotion_allowed = False

report = {
    "schema":
        "raios.v8.6.2-r3.independent-replay.v1",

    "generated_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "model": {
        "path":
            str(MODEL_PATH),

        "load_seconds":
            round(
                load_seconds,
                3,
            ),

        "gpu":
            gpu_info.stdout.strip(),
    },

    "target_skill": {
        "content_hash":
            INVALID_SKILL_HASH,

        "skill_id":
            INVALID_SKILL_ID,

        "historical_status":
            historical_skill.get(
                "status"
            ),

        "effective_status_before":
            effective_before.effective_status,
    },

    "replays":
        replays,

    "metrics": {
        "replays":
            len(replays),

        "passed":
            len(passed),

        "failed":
            len(replays) - len(passed),

        "success_rate":
            success_rate,

        "avg_confidence":
            avg_confidence,

        "avg_latency_seconds":
            avg_latency,
    },

    "gate": {
        "required_passes":
            "5/5",

        "confidence_domain":
            "[0,1]",

        "result":
            effective_result,

        "automatic_promotion":
            False,
    },

    "input_mutated":
        False,

    "training":
        False,

    "promotion":
        False,
}

REPORT_PATH = (
    REPORTS
    / "v8.6.2-r3-independent-replay.json"
)

REPORT_PATH.write_text(
    json.dumps(
        report,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

# =============================================================================
# 16. FINAL RESULT
# =============================================================================

print("\n" + "=" * 122)
print("R3 INDEPENDENT REPLAY RESULT")
print("=" * 122)

print("Replays             :", len(replays))
print("Passed              :", len(passed))
print("Failed              :", len(replays) - len(passed))

print(
    "Success rate        :",
    round(
        success_rate,
        4,
    )
)

print(
    "Average confidence  :",
    (
        round(
            avg_confidence,
            6,
        )
        if avg_confidence is not None
        else None
    )
)

print(
    "Average latency     :",
    round(
        avg_latency,
        4,
    ),
    "seconds"
)

print(
    "Gate result          :",
    effective_result
)

print(
    "Automatic promotion  :",
    promotion_allowed
)

print("\nOUTPUT:")
print(REPORT_PATH)

print("""
TRAINING        : NO
INPUT MUTATED   : NO
PROMOTED        : NO

IMPORTANT:
Even if replay passes 5/5,
the skill remains quarantined until the next
explicit durability/promotion gate.
""")

RAIOS V8.6.2-R3 — INDEPENDENT REPLAY REVALIDATION
T4 GATE | NO TRAINING | NO PROMOTION | FAIL CLOSED

GPU GATE
Tesla T4, 15360 MiB, 14912 MiB
Tesla T4, 15360 MiB, 14912 MiB


RuntimeError: Missing V8.6.2 safety runtime: /kaggle/working/RAIOS-V8.6.2/runtime/confidence_contract.py

In [2]:
from pathlib import Path

runtime = Path("/kaggle/working/RAIOS-V8.6.2/runtime")

for name in [
    "confidence_contract.py",
    "skill_quarantine.py",
    "routing_guard.py",
]:
    p = runtime / name
    print(name, "=>", p.exists(), "|", p)

confidence_contract.py => False | /kaggle/working/RAIOS-V8.6.2/runtime/confidence_contract.py
skill_quarantine.py => False | /kaggle/working/RAIOS-V8.6.2/runtime/skill_quarantine.py
routing_guard.py => False | /kaggle/working/RAIOS-V8.6.2/runtime/routing_guard.py


In [3]:
from pathlib import Path
import hashlib

print("=" * 100)
print("RAIOS V8.6.2 — DURABLE RUNTIME DISCOVERY")
print("READ ONLY | NO MODEL | NO TRAINING | NO MUTATION")
print("=" * 100)

targets = {
    "confidence_contract.py":
        "d5306df6c42a52cb2937b458584575174ddfc52ee488b121a437a240ac346f85",
    "routing_guard.py":
        "fbda164c233b39ec25b60d59f5b8b45cc2c5f60604149e5aa646993d4d130c20",
    "skill_quarantine.py":
        "883f438b7aa832c051ae83222bdd3ceae736d2b8e05abc9067e92b4cecb2514a",
}

roots = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
]

results = {}

for filename, certified_hash in targets.items():

    matches = []

    for root in roots:
        if not root.exists():
            continue

        for p in root.rglob(filename):
            if not p.is_file():
                continue

            try:
                digest = hashlib.sha256(
                    p.read_bytes()
                ).hexdigest()

                matches.append({
                    "path": str(p),
                    "sha256": digest,
                    "exact_original": digest == certified_hash,
                    "bytes": p.stat().st_size,
                })

            except Exception as exc:
                matches.append({
                    "path": str(p),
                    "error": str(exc),
                })

    results[filename] = matches


print("\nDISCOVERY RESULT")
print("=" * 100)

total = 0
exact = 0

for filename, matches in results.items():

    print(f"\n{filename}")
    print("-" * 100)

    if not matches:
        print("NOT FOUND")
        continue

    for item in matches:
        total += 1

        if item.get("exact_original"):
            exact += 1
            status = "EXACT_ORIGINAL"
        else:
            status = "HASH_DIFFERENT"

        print(status)
        print(" PATH  :", item.get("path"))
        print(" SHA256:", item.get("sha256"))
        print(" BYTES :", item.get("bytes"))


print("\n" + "=" * 100)
print("SUMMARY")
print("=" * 100)

print("Runtime candidates found :", total)
print("Exact original hashes    :", exact)
print("Expected exact originals :", 3)

if exact == 3:
    print("\nSTATUS: ORIGINAL_V862_RUNTIME_RECOVERABLE")
    print("NEXT: restore exact files, then run functional verification.")

elif total > 0:
    print("\nSTATUS: RUNTIME_VARIANTS_FOUND")
    print("NEXT: inspect variants before restoring anything.")

else:
    print("\nSTATUS: NO_DURABLE_RUNTIME_FOUND")
    print("NEXT: rebuild one canonical V8.6.2 runtime, certify it,")
    print("      persist it durably, then run R3.")

print("\nMODEL LOADED : NO")
print("TRAINING     : NO")
print("STATE MUTATED: NO")

RAIOS V8.6.2 — DURABLE RUNTIME DISCOVERY
READ ONLY | NO MODEL | NO TRAINING | NO MUTATION

DISCOVERY RESULT

confidence_contract.py
----------------------------------------------------------------------------------------------------
NOT FOUND

routing_guard.py
----------------------------------------------------------------------------------------------------
NOT FOUND

skill_quarantine.py
----------------------------------------------------------------------------------------------------
NOT FOUND

SUMMARY
Runtime candidates found : 0
Exact original hashes    : 0
Expected exact originals : 3

STATUS: NO_DURABLE_RUNTIME_FOUND
NEXT: rebuild one canonical V8.6.2 runtime, certify it,
      persist it durably, then run R3.

MODEL LOADED : NO
TRAINING     : NO
STATE MUTATED: NO


In [1]:
from __future__ import annotations

from pathlib import Path
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from typing import Any
import hashlib
import json
import shutil
import sys
import zipfile

print("=" * 118)
print("RAIOS V8.6.2 — CANONICAL SAFETY RUNTIME BUILD + CERTIFICATION")
print("CPU ONLY | NO MODEL | NO TRAINING | FAIL CLOSED")
print("=" * 118)

# =============================================================================
# CONSTANTS
# =============================================================================

V862 = Path("/kaggle/working/RAIOS-V8.6.2")
RUNTIME = V862 / "runtime"
REPORTS = V862 / "reports"
PACKAGE = V862 / "durable-runtime-package"

for p in [RUNTIME, REPORTS, PACKAGE]:
    p.mkdir(parents=True, exist_ok=True)

INVALID_SKILL_HASH = (
    "683ca22e97a2cb919ab078321076bd06"
    "b146b1bf69e937a2062d68755e993b65"
)

INVALID_SKILL_ID = "RAIOS.REPOSITORY_ANALYSIS.MICRO.v1"

# =============================================================================
# CANONICAL SOURCE — confidence_contract.py
# =============================================================================

confidence_source = r'''from __future__ import annotations

import math
from dataclasses import dataclass
from typing import Any


class ConfidenceContractError(ValueError):
    """Raised when confidence violates the canonical RAIOS confidence contract."""


@dataclass(frozen=True)
class ConfidenceValue:
    value: float


def validate_confidence(value: Any) -> ConfidenceValue:
    """
    Canonical RAIOS confidence boundary.

    Invariants:
    - bool is forbidden even though bool subclasses int in Python.
    - only int/float are accepted.
    - NaN and infinity are forbidden.
    - canonical domain is inclusive [0, 1].
    - percentage-style values such as 70 are NOT silently normalized.
    """

    if isinstance(value, bool):
        raise ConfidenceContractError(
            "Boolean is not a valid confidence value."
        )

    if not isinstance(value, (int, float)):
        raise ConfidenceContractError(
            f"Unsupported confidence type: {type(value).__name__}"
        )

    numeric = float(value)

    if not math.isfinite(numeric):
        raise ConfidenceContractError(
            "Confidence must be finite."
        )

    if numeric < 0.0 or numeric > 1.0:
        raise ConfidenceContractError(
            f"Confidence violates canonical [0,1] domain: {value}"
        )

    return ConfidenceValue(value=numeric)


def require_confidence(value: Any) -> float:
    return validate_confidence(value).value
'''

# =============================================================================
# CANONICAL SOURCE — skill_quarantine.py
# =============================================================================

quarantine_source = f'''from __future__ import annotations

from dataclasses import dataclass
from typing import Optional


INVALIDATED_REPOSITORY_ANALYSIS_SKILL = "{INVALID_SKILL_HASH}"

QUARANTINE_STATUS = "QUARANTINED_PENDING_REVALIDATION"


@dataclass(frozen=True)
class EffectiveSkillState:
    content_hash: str
    stored_status: Optional[str]
    effective_status: str
    usable: bool
    reason: str


def effective_skill_state(
    content_hash: str,
    stored_status: Optional[str],
) -> EffectiveSkillState:

    if content_hash == INVALIDATED_REPOSITORY_ANALYSIS_SKILL:
        return EffectiveSkillState(
            content_hash=content_hash,
            stored_status=stored_status,
            effective_status=QUARANTINE_STATUS,
            usable=False,
            reason=(
                "Historical promotion depended on invalid confidence scale."
            ),
        )

    normalized = (stored_status or "UNKNOWN").upper()

    usable = normalized in {{
        "PROMOTED",
        "ACTIVE",
        "VALIDATED",
    }}

    return EffectiveSkillState(
        content_hash=content_hash,
        stored_status=stored_status,
        effective_status=normalized,
        usable=usable,
        reason=(
            "Skill is not subject to the V8.6.2 historical quarantine."
            if usable
            else "Skill is not in a runtime-usable state."
        ),
    )


def is_skill_runtime_usable(
    content_hash: str,
    stored_status: Optional[str],
) -> bool:
    return effective_skill_state(
        content_hash,
        stored_status,
    ).usable
'''

# =============================================================================
# CANONICAL SOURCE — routing_guard.py
# =============================================================================

routing_source = r'''from __future__ import annotations

from dataclasses import dataclass
from typing import Optional

from skill_quarantine import effective_skill_state


@dataclass(frozen=True)
class RoutingDecision:
    requested_route: str
    effective_route: str
    allowed: bool
    reason: str
    skill_hash: Optional[str]


def guard_skill_route(
    requested_route: str,
    skill_hash: Optional[str] = None,
    stored_status: Optional[str] = None,
) -> RoutingDecision:

    requested = str(requested_route or "").strip().upper()

    if requested != "ZERO_LLM":
        return RoutingDecision(
            requested_route=requested,
            effective_route=requested or "REASONING",
            allowed=True,
            reason="ROUTE_NOT_SUBJECT_TO_ZERO_LLM_SKILL_GATE",
            skill_hash=skill_hash,
        )

    if not skill_hash:
        return RoutingDecision(
            requested_route=requested,
            effective_route="REASONING",
            allowed=False,
            reason="ZERO_LLM_REQUIRES_VALIDATED_SKILL",
            skill_hash=None,
        )

    state = effective_skill_state(
        skill_hash,
        stored_status,
    )

    if not state.usable:
        return RoutingDecision(
            requested_route=requested,
            effective_route="REASONING",
            allowed=False,
            reason=f"SKILL_NOT_RUNTIME_USABLE:{state.effective_status}",
            skill_hash=skill_hash,
        )

    return RoutingDecision(
        requested_route=requested,
        effective_route="ZERO_LLM",
        allowed=True,
        reason="SKILL_RUNTIME_USABLE",
        skill_hash=skill_hash,
    )
'''

sources = {
    "confidence_contract.py": confidence_source,
    "skill_quarantine.py": quarantine_source,
    "routing_guard.py": routing_source,
}

# =============================================================================
# WRITE CANONICAL RUNTIME
# =============================================================================

for filename, source in sources.items():
    (RUNTIME / filename).write_text(
        source,
        encoding="utf-8",
        newline="\n",
    )

# =============================================================================
# HASH
# =============================================================================

def sha256_file(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


runtime_hashes = {
    name: sha256_file(RUNTIME / name)
    for name in sources
}

print("\nCANONICAL RUNTIME HASHES")
print("-" * 118)

for name, digest in runtime_hashes.items():
    print(f"{name:30} {digest}")

# =============================================================================
# IMPORT FRESH RUNTIME
# =============================================================================

if str(RUNTIME) not in sys.path:
    sys.path.insert(0, str(RUNTIME))

for module_name in [
    "confidence_contract",
    "skill_quarantine",
    "routing_guard",
]:
    sys.modules.pop(module_name, None)

from confidence_contract import (
    validate_confidence,
    ConfidenceContractError,
)

from skill_quarantine import effective_skill_state
from routing_guard import guard_skill_route

# =============================================================================
# SELF TEST
# =============================================================================

tests = []

def record(name, fn):
    try:
        value = fn()
        tests.append({
            "name": name,
            "pass": True,
            "result": repr(value),
        })
        print("PASS |", name, "|", value)
    except Exception as exc:
        tests.append({
            "name": name,
            "pass": False,
            "error": f"{type(exc).__name__}: {exc}",
        })
        print("FAIL |", name, "|", type(exc).__name__, str(exc))


def must_accept(value, expected):
    result = validate_confidence(value).value
    assert result == expected
    return result


def must_reject(value):
    try:
        validate_confidence(value)
    except ConfidenceContractError as exc:
        return str(exc)

    raise AssertionError(
        f"Value should have been rejected: {value!r}"
    )


record("accept_0", lambda: must_accept(0, 0.0))
record("accept_0_70", lambda: must_accept(0.70, 0.70))
record("accept_1", lambda: must_accept(1, 1.0))

record("reject_70", lambda: must_reject(70))
record("reject_14_79", lambda: must_reject(14.79))
record("reject_negative", lambda: must_reject(-0.1))
record("reject_over_one", lambda: must_reject(1.1))
record("reject_true", lambda: must_reject(True))
record("reject_none", lambda: must_reject(None))
record("reject_string", lambda: must_reject("0.70"))
record("reject_nan", lambda: must_reject(float("nan")))
record("reject_inf", lambda: must_reject(float("inf")))


def quarantine_test():
    state = effective_skill_state(
        INVALID_SKILL_HASH,
        "PROMOTED",
    )

    assert state.effective_status == "QUARANTINED_PENDING_REVALIDATION"
    assert state.usable is False
    return state


record(
    "invalid_skill_quarantined",
    quarantine_test,
)


def routing_test():
    decision = guard_skill_route(
        "ZERO_LLM",
        INVALID_SKILL_HASH,
        "PROMOTED",
    )

    assert decision.allowed is False
    assert decision.effective_route == "REASONING"
    return decision


record(
    "invalid_skill_zero_llm_blocked",
    routing_test,
)


def missing_skill_test():
    decision = guard_skill_route(
        "ZERO_LLM",
        None,
        None,
    )

    assert decision.allowed is False
    assert decision.effective_route == "REASONING"
    return decision


record(
    "zero_llm_without_skill_blocked",
    missing_skill_test,
)


def reasoning_test():
    decision = guard_skill_route(
        "REASONING",
        INVALID_SKILL_HASH,
        "PROMOTED",
    )

    assert decision.allowed is True
    assert decision.effective_route == "REASONING"
    return decision


record(
    "reasoning_route_available",
    reasoning_test,
)

passed = sum(t["pass"] for t in tests)
failed = len(tests) - passed

# =============================================================================
# FAIL CLOSED BEFORE CERTIFICATION
# =============================================================================

if failed:
    raise RuntimeError(
        f"Canonical runtime certification failed: "
        f"{failed}/{len(tests)} tests failed."
    )

# =============================================================================
# CREATE DURABLE PACKAGE
# =============================================================================

if PACKAGE.exists():
    shutil.rmtree(PACKAGE)

PACKAGE.mkdir(parents=True)

for name in sources:
    shutil.copy2(
        RUNTIME / name,
        PACKAGE / name,
    )

manifest = {
    "schema": "raios.v8.6.2.runtime-manifest.v1",
    "version": "V8.6.2",
    "artifact_type": "CANONICAL_SAFETY_RUNTIME",
    "generated_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "files": {
        name: {
            "sha256": runtime_hashes[name],
            "bytes": (RUNTIME / name).stat().st_size,
        }
        for name in sources
    },
    "invariants": [
        "confidence values MUST be finite numeric values in [0,1]",
        "percentage-style confidence is never silently normalized",
        "invalidated repository-analysis skill remains quarantined",
        "quarantined skill cannot authorize ZERO_LLM routing",
        "blocked ZERO_LLM routing falls back to REASONING",
        "no automatic skill promotion occurs in this runtime",
    ],
    "invalidated_skill": {
        "id": INVALID_SKILL_ID,
        "content_hash": INVALID_SKILL_HASH,
        "effective_status": "QUARANTINED_PENDING_REVALIDATION",
    },
    "self_test": {
        "total": len(tests),
        "passed": passed,
        "failed": failed,
    },
    "training": False,
    "promotion": False,
}

manifest_path = PACKAGE / "RUNTIME-MANIFEST.json"

manifest_path.write_text(
    json.dumps(
        manifest,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

# =============================================================================
# RESTORE RECIPE
# =============================================================================

restore_recipe = r'''from pathlib import Path
import hashlib
import json
import shutil

SOURCE = Path("/kaggle/input/datasets/greenylife/raios-v862-runtime")
TARGET = Path("/kaggle/working/RAIOS-V8.6.2/runtime")

manifest_path = SOURCE / "RUNTIME-MANIFEST.json"

if not manifest_path.exists():
    raise RuntimeError("RUNTIME-MANIFEST.json not found.")

manifest = json.loads(
    manifest_path.read_text(encoding="utf-8")
)

TARGET.mkdir(parents=True, exist_ok=True)

for filename, meta in manifest["files"].items():

    src = SOURCE / filename
    dst = TARGET / filename

    if not src.exists():
        raise RuntimeError(f"Missing durable runtime file: {src}")

    actual = hashlib.sha256(src.read_bytes()).hexdigest()

    if actual != meta["sha256"]:
        raise RuntimeError(
            f"Durable runtime hash mismatch: {filename}"
        )

    shutil.copy2(src, dst)

    restored = hashlib.sha256(dst.read_bytes()).hexdigest()

    if restored != meta["sha256"]:
        raise RuntimeError(
            f"Restored runtime hash mismatch: {filename}"
        )

print("STATUS: V8.6.2_CANONICAL_RUNTIME_RESTORED")
'''

(PACKAGE / "RESTORE-RUNTIME.py").write_text(
    restore_recipe,
    encoding="utf-8",
    newline="\n",
)

# =============================================================================
# ZIP PACKAGE
# =============================================================================

ZIP = V862 / "RAIOS-V8.6.2-CANONICAL-RUNTIME.zip"

if ZIP.exists():
    ZIP.unlink()

with zipfile.ZipFile(
    ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zf:

    for p in sorted(PACKAGE.iterdir()):
        if p.is_file():
            zf.write(
                p,
                arcname=p.name,
            )

zip_hash = sha256_file(ZIP)

# =============================================================================
# CERTIFICATION RECEIPT
# =============================================================================

receipt = {
    "schema": "raios.v8.6.2.runtime-certification.v1",
    "status": "CANONICAL_RUNTIME_CERTIFIED",
    "runtime_hashes": runtime_hashes,
    "tests": tests,
    "tests_total": len(tests),
    "passed": passed,
    "failed": failed,
    "package": str(ZIP),
    "package_sha256": zip_hash,
    "package_bytes": ZIP.stat().st_size,
    "training": False,
    "model_loaded": False,
    "promotion": False,
}

receipt_path = REPORTS / "v8.6.2-runtime-certification.json"

receipt_path.write_text(
    json.dumps(
        receipt,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

# =============================================================================
# FINAL
# =============================================================================

print("\n" + "=" * 118)
print("V8.6.2 CANONICAL RUNTIME CERTIFICATION")
print("=" * 118)

print("Tests total :", len(tests))
print("Passed      :", passed)
print("Failed      :", failed)

print("\nRuntime:")
for name, digest in runtime_hashes.items():
    print(" -", name, digest)

print("\nPackage     :", ZIP)
print("Package SHA :", zip_hash)
print("Package size:", ZIP.stat().st_size, "bytes")

print("\nManifest    :", manifest_path)
print("Receipt     :", receipt_path)

print("\nSTATUS: CANONICAL_RUNTIME_CERTIFIED")
print("MODEL LOADED : NO")
print("TRAINING     : NO")
print("PROMOTED     : NO")

print("""
NEXT:
1. DO NOT restart this Kaggle session.
2. Preserve RAIOS-V8.6.2-CANONICAL-RUNTIME.zip as a Kaggle Dataset.
3. Then run R3 against the currently certified runtime.
""")

RAIOS V8.6.2 — CANONICAL SAFETY RUNTIME BUILD + CERTIFICATION
CPU ONLY | NO MODEL | NO TRAINING | FAIL CLOSED

CANONICAL RUNTIME HASHES
----------------------------------------------------------------------------------------------------------------------
confidence_contract.py         369b8fa6b9248e3b01fb8a8aca2c48b9f06d1e9086c4fe274ee160fed9205dc8
skill_quarantine.py            2e9ad00553ec692c5be0c6f9402875d773754a242292015dd9b16f5242713476
routing_guard.py               7ca061d1d8530a9d7c77e2bfee4b97bb38445613adf6df1df72a291c47a0dc45
PASS | accept_0 | 0.0
PASS | accept_0_70 | 0.7
PASS | accept_1 | 1.0
PASS | reject_70 | Confidence violates canonical [0,1] domain: 70
PASS | reject_14_79 | Confidence violates canonical [0,1] domain: 14.79
PASS | reject_negative | Confidence violates canonical [0,1] domain: -0.1
PASS | reject_over_one | Confidence violates canonical [0,1] domain: 1.1
PASS | reject_true | Boolean is not a valid confidence value.
PASS | reject_none | Unsupported confiden

In [1]:
from __future__ import annotations

from pathlib import Path
from dataclasses import dataclass
from datetime import datetime, timezone
from typing import Any, Optional
import hashlib
import json
import math
import shutil
import sys
import zipfile

print("=" * 120)
print("RAIOS V8.6.2 — CANONICAL RUNTIME REBUILD + DATASET PACKAGE")
print("CPU ONLY | NO MODEL | NO TRAINING | NO PROMOTION")
print("=" * 120)

ROOT = Path("/kaggle/working/RAIOS-V8.6.2")
RUNTIME = ROOT / "runtime"
PACKAGE = ROOT / "raios-v862-canonical-runtime"
REPORTS = ROOT / "reports"

for p in [RUNTIME, PACKAGE, REPORTS]:
    p.mkdir(parents=True, exist_ok=True)

INVALID_SKILL_HASH = (
    "683ca22e97a2cb919ab078321076bd06"
    "b146b1bf69e937a2062d68755e993b65"
)

INVALID_SKILL_ID = "RAIOS.REPOSITORY_ANALYSIS.MICRO.v1"

# =====================================================================
# 1. confidence_contract.py
# =====================================================================

confidence_source = r'''from __future__ import annotations

import math
from dataclasses import dataclass
from typing import Any


class ConfidenceContractError(ValueError):
    pass


@dataclass(frozen=True)
class ConfidenceValue:
    value: float


def validate_confidence(value: Any) -> ConfidenceValue:
    if isinstance(value, bool):
        raise ConfidenceContractError(
            "Boolean is not a valid confidence value."
        )

    if not isinstance(value, (int, float)):
        raise ConfidenceContractError(
            f"Unsupported confidence type: {type(value).__name__}"
        )

    numeric = float(value)

    if not math.isfinite(numeric):
        raise ConfidenceContractError(
            "Confidence must be finite."
        )

    if numeric < 0.0 or numeric > 1.0:
        raise ConfidenceContractError(
            f"Confidence violates canonical [0,1] domain: {value}"
        )

    return ConfidenceValue(value=numeric)


def require_confidence(value: Any) -> float:
    return validate_confidence(value).value
'''

(RUNTIME / "confidence_contract.py").write_text(
    confidence_source,
    encoding="utf-8",
    newline="\n"
)

# =====================================================================
# 2. skill_quarantine.py
# =====================================================================

quarantine_source = f'''from __future__ import annotations

from dataclasses import dataclass
from typing import Optional

INVALIDATED_REPOSITORY_ANALYSIS_SKILL = "{INVALID_SKILL_HASH}"

QUARANTINE_STATUS = "QUARANTINED_PENDING_REVALIDATION"


@dataclass(frozen=True)
class EffectiveSkillState:
    content_hash: str
    stored_status: Optional[str]
    effective_status: str
    usable: bool
    reason: str


def effective_skill_state(
    content_hash: str,
    stored_status: Optional[str],
) -> EffectiveSkillState:

    if content_hash == INVALIDATED_REPOSITORY_ANALYSIS_SKILL:
        return EffectiveSkillState(
            content_hash=content_hash,
            stored_status=stored_status,
            effective_status=QUARANTINE_STATUS,
            usable=False,
            reason="Historical promotion depended on invalid confidence scale.",
        )

    normalized = (stored_status or "UNKNOWN").upper()

    usable = normalized in {{
        "PROMOTED",
        "ACTIVE",
        "VALIDATED",
    }}

    return EffectiveSkillState(
        content_hash=content_hash,
        stored_status=stored_status,
        effective_status=normalized,
        usable=usable,
        reason=(
            "Skill is runtime usable."
            if usable
            else "Skill is not in a runtime-usable state."
        ),
    )
'''

(RUNTIME / "skill_quarantine.py").write_text(
    quarantine_source,
    encoding="utf-8",
    newline="\n"
)

# =====================================================================
# 3. routing_guard.py
# =====================================================================

routing_source = r'''from __future__ import annotations

from dataclasses import dataclass
from typing import Optional

from skill_quarantine import effective_skill_state


@dataclass(frozen=True)
class RoutingDecision:
    requested_route: str
    effective_route: str
    allowed: bool
    reason: str
    skill_hash: Optional[str]


def guard_skill_route(
    requested_route: str,
    skill_hash: Optional[str] = None,
    stored_status: Optional[str] = None,
) -> RoutingDecision:

    requested = str(requested_route or "").strip().upper()

    if requested != "ZERO_LLM":
        return RoutingDecision(
            requested_route=requested,
            effective_route=requested or "REASONING",
            allowed=True,
            reason="ROUTE_NOT_SUBJECT_TO_ZERO_LLM_SKILL_GATE",
            skill_hash=skill_hash,
        )

    if not skill_hash:
        return RoutingDecision(
            requested_route=requested,
            effective_route="REASONING",
            allowed=False,
            reason="ZERO_LLM_REQUIRES_VALIDATED_SKILL",
            skill_hash=None,
        )

    state = effective_skill_state(
        skill_hash,
        stored_status,
    )

    if not state.usable:
        return RoutingDecision(
            requested_route=requested,
            effective_route="REASONING",
            allowed=False,
            reason=f"SKILL_NOT_RUNTIME_USABLE:{state.effective_status}",
            skill_hash=skill_hash,
        )

    return RoutingDecision(
        requested_route=requested,
        effective_route="ZERO_LLM",
        allowed=True,
        reason="SKILL_RUNTIME_USABLE",
        skill_hash=skill_hash,
    )
'''

(RUNTIME / "routing_guard.py").write_text(
    routing_source,
    encoding="utf-8",
    newline="\n"
)

# =====================================================================
# 4. HASHES
# =====================================================================

def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

hashes = {
    name: sha256(RUNTIME / name)
    for name in [
        "confidence_contract.py",
        "skill_quarantine.py",
        "routing_guard.py",
    ]
}

print("\nRUNTIME HASHES")

for name, digest in hashes.items():
    print(name, digest)

# =====================================================================
# 5. FUNCTIONAL TESTS
# =====================================================================

sys.path.insert(0, str(RUNTIME))

for mod in [
    "confidence_contract",
    "skill_quarantine",
    "routing_guard",
]:
    sys.modules.pop(mod, None)

from confidence_contract import (
    validate_confidence,
    ConfidenceContractError,
)

from skill_quarantine import effective_skill_state
from routing_guard import guard_skill_route

tests = []

def ok(name, condition, detail=""):
    tests.append({
        "name": name,
        "pass": bool(condition),
        "detail": str(detail),
    })
    print(
        "PASS" if condition else "FAIL",
        "|",
        name,
        "|",
        detail
    )

for value in [0, 0.7, 1]:
    result = validate_confidence(value).value
    ok(
        f"accept_{value}",
        result == float(value),
        result
    )

for value in [
    70,
    14.79,
    -0.1,
    1.1,
    True,
    None,
    "0.7",
    float("nan"),
    float("inf"),
]:
    try:
        validate_confidence(value)
        ok(
            f"reject_{value!r}",
            False,
            "unexpected acceptance"
        )
    except ConfidenceContractError as exc:
        ok(
            f"reject_{value!r}",
            True,
            exc
        )

state = effective_skill_state(
    INVALID_SKILL_HASH,
    "PROMOTED"
)

ok(
    "invalid_skill_quarantined",
    (
        state.usable is False
        and state.effective_status
        == "QUARANTINED_PENDING_REVALIDATION"
    ),
    state
)

route = guard_skill_route(
    "ZERO_LLM",
    INVALID_SKILL_HASH,
    "PROMOTED"
)

ok(
    "invalid_skill_zero_llm_blocked",
    (
        route.allowed is False
        and route.effective_route == "REASONING"
    ),
    route
)

missing = guard_skill_route(
    "ZERO_LLM",
    None,
    None
)

ok(
    "zero_llm_without_skill_blocked",
    (
        missing.allowed is False
        and missing.effective_route == "REASONING"
    ),
    missing
)

failed = [
    t for t in tests
    if not t["pass"]
]

if failed:
    raise RuntimeError(
        f"Runtime certification failed: {len(failed)} tests failed"
    )

# =====================================================================
# 6. PACKAGE FILES
# =====================================================================

if PACKAGE.exists():
    shutil.rmtree(PACKAGE)

PACKAGE.mkdir(parents=True)

for name in hashes:
    shutil.copy2(
        RUNTIME / name,
        PACKAGE / name
    )

manifest = {
    "schema": "raios.v8.6.2.runtime-manifest.v1",
    "version": "V8.6.2",
    "artifact_type": "CANONICAL_SAFETY_RUNTIME",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "files": {
        name: {
            "sha256": digest,
            "bytes": (
                RUNTIME / name
            ).stat().st_size,
        }
        for name, digest in hashes.items()
    },
    "invalidated_skill": {
        "skill_id": INVALID_SKILL_ID,
        "content_hash": INVALID_SKILL_HASH,
        "effective_status":
            "QUARANTINED_PENDING_REVALIDATION",
    },
    "invariants": [
        "confidence domain is [0,1]",
        "percentage confidence is never silently normalized",
        "invalidated skill remains quarantined",
        "quarantined skill cannot route to ZERO_LLM",
        "ZERO_LLM without validated skill falls back to REASONING",
        "automatic promotion is forbidden",
    ],
    "tests": {
        "total": len(tests),
        "passed": len(tests),
        "failed": 0,
    }
}

(PACKAGE / "RUNTIME-MANIFEST.json").write_text(
    json.dumps(
        manifest,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

# =====================================================================
# 7. RESTORE SCRIPT
# =====================================================================

restore_source = r'''from pathlib import Path
import hashlib
import json
import shutil

SOURCE = Path("/kaggle/input/datasets/greenylife/raios-v862-canonical-runtime")
TARGET = Path("/kaggle/working/RAIOS-V8.6.2/runtime")

MANIFEST = SOURCE / "RUNTIME-MANIFEST.json"

if not MANIFEST.exists():
    raise RuntimeError(
        f"Runtime manifest missing: {MANIFEST}"
    )

manifest = json.loads(
    MANIFEST.read_text(encoding="utf-8")
)

TARGET.mkdir(
    parents=True,
    exist_ok=True
)

for filename, meta in manifest["files"].items():

    source = SOURCE / filename
    target = TARGET / filename

    if not source.exists():
        raise RuntimeError(
            f"Missing durable runtime file: {source}"
        )

    source_hash = hashlib.sha256(
        source.read_bytes()
    ).hexdigest()

    if source_hash != meta["sha256"]:
        raise RuntimeError(
            f"Source hash mismatch: {filename}"
        )

    shutil.copy2(
        source,
        target
    )

    restored_hash = hashlib.sha256(
        target.read_bytes()
    ).hexdigest()

    if restored_hash != meta["sha256"]:
        raise RuntimeError(
            f"Restore hash mismatch: {filename}"
        )

print(
    "STATUS: V8.6.2_CANONICAL_RUNTIME_RESTORED"
)
'''

(PACKAGE / "RESTORE-RUNTIME.py").write_text(
    restore_source,
    encoding="utf-8",
    newline="\n"
)

# =====================================================================
# 8. ZIP
# =====================================================================

ZIP = ROOT / "RAIOS-V8.6.2-CANONICAL-RUNTIME.zip"

if ZIP.exists():
    ZIP.unlink()

with zipfile.ZipFile(
    ZIP,
    "w",
    zipfile.ZIP_DEFLATED
) as zf:

    for p in sorted(
        PACKAGE.iterdir()
    ):
        if p.is_file():
            zf.write(
                p,
                arcname=p.name
            )

zip_hash = sha256(ZIP)

# =====================================================================
# 9. FINAL RECEIPT
# =====================================================================

receipt = {
    "status":
        "READY_FOR_KAGGLE_DATASET",

    "package_directory":
        str(PACKAGE),

    "zip":
        str(ZIP),

    "zip_sha256":
        zip_hash,

    "runtime_hashes":
        hashes,

    "tests":
        {
            "passed": len(tests),
            "failed": 0,
        },

    "model_loaded":
        False,

    "training":
        False,

    "promotion":
        False,
}

receipt_file = (
    REPORTS
    / "v8.6.2-runtime-dataset-package.json"
)

receipt_file.write_text(
    json.dumps(
        receipt,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print("\n" + "=" * 120)
print("RAIOS V8.6.2 DATASET PACKAGE READY")
print("=" * 120)

print("Tests passed :", len(tests))
print("Tests failed : 0")

print("\nPACKAGE DIRECTORY:")
print(PACKAGE)

print("\nFILES:")

for p in sorted(
    PACKAGE.iterdir()
):
    print(
        " -",
        p.name,
        "|",
        p.stat().st_size,
        "bytes"
    )

print("\nZIP:")
print(ZIP)

print("\nZIP SHA256:")
print(zip_hash)

print("\nSTATUS: READY_FOR_KAGGLE_DATASET")
print("GPU REQUIRED : NO")
print("MODEL LOADED : NO")
print("TRAINING     : NO")
print("PROMOTED     : NO")

RAIOS V8.6.2 — CANONICAL RUNTIME REBUILD + DATASET PACKAGE
CPU ONLY | NO MODEL | NO TRAINING | NO PROMOTION

RUNTIME HASHES
confidence_contract.py 2aef4fce7ac6e7720ebb987f7fa8dc1695300a26ed2f84c84fb515d71d856fcc
skill_quarantine.py 1f6e8ef7fdb42113f8e281bc5407e743bbb8fa1be10c9913d071f9e6adbffea5
routing_guard.py 7ca061d1d8530a9d7c77e2bfee4b97bb38445613adf6df1df72a291c47a0dc45
PASS | accept_0 | 0.0
PASS | accept_0.7 | 0.7
PASS | accept_1 | 1.0
PASS | reject_70 | Confidence violates canonical [0,1] domain: 70
PASS | reject_14.79 | Confidence violates canonical [0,1] domain: 14.79
PASS | reject_-0.1 | Confidence violates canonical [0,1] domain: -0.1
PASS | reject_1.1 | Confidence violates canonical [0,1] domain: 1.1
PASS | reject_True | Boolean is not a valid confidence value.
PASS | reject_None | Unsupported confidence type: NoneType
PASS | reject_'0.7' | Unsupported confidence type: str
PASS | reject_nan | Confidence must be finite.
PASS | reject_inf | Confidence must be finite.
PASS | 

In [2]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime, timezone
import json
import os
import shutil
import subprocess
import sys

print("=" * 120)
print("RAIOS V8.6.2 — DURABLE KAGGLE DATASET PUBLISHER")
print("CPU ONLY | NO MODEL | NO TRAINING")
print("=" * 120)

ROOT = Path("/kaggle/working/RAIOS-V8.6.2")

SOURCE = (
    ROOT
    / "raios-v862-canonical-runtime"
)

ZIP = (
    ROOT
    / "RAIOS-V8.6.2-CANONICAL-RUNTIME.zip"
)

PUBLISH = (
    ROOT
    / "kaggle-dataset-publish"
)

if not SOURCE.exists():
    raise RuntimeError(
        f"Runtime package missing: {SOURCE}"
    )

if not ZIP.exists():
    raise RuntimeError(
        f"Runtime ZIP missing: {ZIP}"
    )

# ============================================================
# REBUILD CLEAN PUBLISH DIRECTORY
# ============================================================

if PUBLISH.exists():
    shutil.rmtree(PUBLISH)

PUBLISH.mkdir(
    parents=True,
    exist_ok=True,
)

for p in SOURCE.iterdir():
    if p.is_file():
        shutil.copy2(
            p,
            PUBLISH / p.name,
        )

shutil.copy2(
    ZIP,
    PUBLISH / ZIP.name,
)

# ============================================================
# DATASET METADATA
# ============================================================

metadata = {
    "title": "RAIOS V8.6.2 Canonical Runtime",
    "id": "greenylife/raios-v862-canonical-runtime",
    "licenses": [
        {
            "name": "other"
        }
    ]
}

metadata_path = (
    PUBLISH
    / "dataset-metadata.json"
)

metadata_path.write_text(
    json.dumps(
        metadata,
        indent=2,
    ),
    encoding="utf-8",
)

# ============================================================
# PUBLISH RECEIPT
# ============================================================

receipt = {
    "schema":
        "raios.v8.6.2.canonical-runtime-publish.v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "dataset_id":
        "greenylife/raios-v862-canonical-runtime",

    "runtime_hashes": {
        "confidence_contract.py":
            "2aef4fce7ac6e7720ebb987f7fa8dc1695300a26ed2f84c84fb515d71d856fcc",

        "skill_quarantine.py":
            "1f6e8ef7fdb42113f8e281bc5407e743bbb8fa1be10c9913d071f9e6adbffea5",

        "routing_guard.py":
            "7ca061d1d8530a9d7c77e2bfee4b97bb38445613adf6df1df72a291c47a0dc45",
    },

    "zip_sha256":
        "f222c65b4cbb570068e11eb54fe0a6657197e1c78703b39ca7aac694364671e3",

    "self_tests": {
        "passed": 15,
        "failed": 0,
    },

    "status":
        "READY_FOR_DURABLE_PUBLISH",
}

(
    PUBLISH
    / "PUBLISH-RECEIPT.json"
).write_text(
    json.dumps(
        receipt,
        indent=2,
    ),
    encoding="utf-8",
)

# ============================================================
# DISPLAY FILES
# ============================================================

print("\nPUBLISH DIRECTORY")
print(PUBLISH)

print("\nFILES")

for p in sorted(
    PUBLISH.iterdir()
):
    if p.is_file():
        print(
            f" - {p.name} | {p.stat().st_size} bytes"
        )

# ============================================================
# CHECK KAGGLE CLI
# ============================================================

print("\n" + "=" * 120)
print("KAGGLE CLI CHECK")
print("=" * 120)

cli = shutil.which("kaggle")

if not cli:
    print("Kaggle CLI not found.")
    print()
    print("STATUS: PUBLISH_PACKAGE_READY_CLI_UNAVAILABLE")
    print()
    print("DO NOT DELETE OR RESTART YET.")
    print("Use Save Version before closing this session.")
    raise SystemExit(0)

version = subprocess.run(
    [cli, "--version"],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print(version.stdout.strip())

# ============================================================
# CHECK AUTH WITHOUT PRINTING SECRETS
# ============================================================

credential_candidates = [
    Path.home() / ".kaggle" / "kaggle.json",
    Path("/root/.kaggle/kaggle.json"),
]

credential_file = next(
    (
        p
        for p in credential_candidates
        if p.exists()
    ),
    None,
)

env_auth = bool(
    os.environ.get("KAGGLE_USERNAME")
    and os.environ.get("KAGGLE_KEY")
)

print(
    "Credential file:",
    "AVAILABLE"
    if credential_file
    else "NOT_FOUND",
)

print(
    "Environment auth:",
    "AVAILABLE"
    if env_auth
    else "NOT_FOUND",
)

if not credential_file and not env_auth:
    print()
    print("STATUS: PUBLISH_PACKAGE_READY_AUTH_NOT_AVAILABLE")
    print()
    print("The package is ready but this kernel does not expose")
    print("Kaggle API credentials.")
    print()
    print("DO NOT RESTART.")
    print("SAVE VERSION before closing the session.")
    raise SystemExit(0)

# ============================================================
# CHECK IF DATASET ALREADY EXISTS
# ============================================================

DATASET_ID = (
    "greenylife/raios-v862-canonical-runtime"
)

print("\n" + "=" * 120)
print("DATASET EXISTENCE CHECK")
print("=" * 120)

check = subprocess.run(
    [
        cli,
        "datasets",
        "files",
        DATASET_ID,
    ],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

exists = (
    check.returncode == 0
)

print(
    "Dataset exists:",
    exists,
)

# ============================================================
# CREATE OR VERSION
# ============================================================

print("\n" + "=" * 120)
print("DURABLE PUBLISH")
print("=" * 120)

if exists:

    command = [
        cli,
        "datasets",
        "version",
        "-p",
        str(PUBLISH),
        "-m",
        "RAIOS V8.6.2 canonical runtime certification update",
    ]

    action = "VERSION"

else:

    command = [
        cli,
        "datasets",
        "create",
        "-p",
        str(PUBLISH),
        "--dir-mode",
        "zip",
    ]

    action = "CREATE"

print("Action:", action)

result = subprocess.run(
    command,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print(result.stdout)

if result.returncode != 0:

    print("=" * 120)
    print("STATUS: DURABLE_PUBLISH_FAILED")
    print("=" * 120)

    print(
        "Package remains available at:",
        PUBLISH,
    )

    print()
    print("DO NOT RESTART BEFORE SAVE VERSION.")

    raise SystemExit(
        result.returncode
    )

# ============================================================
# SUCCESS
# ============================================================

print("=" * 120)
print("RAIOS V8.6.2 DURABLE RUNTIME PUBLISHED")
print("=" * 120)

print(
    "Dataset:",
    DATASET_ID,
)

print(
    "Expected input path:"
)

print(
    "/kaggle/input/datasets/greenylife/"
    "raios-v862-canonical-runtime"
)

print()
print("STATUS: DURABLE_RUNTIME_PUBLISHED")
print("GPU USED     : NO")
print("MODEL LOADED : NO")
print("TRAINING     : NO")
print("PROMOTED     : NO")

RAIOS V8.6.2 — DURABLE KAGGLE DATASET PUBLISHER
CPU ONLY | NO MODEL | NO TRAINING

PUBLISH DIRECTORY
/kaggle/working/RAIOS-V8.6.2/kaggle-dataset-publish

FILES
 - PUBLISH-RECEIPT.json | 667 bytes
 - RAIOS-V8.6.2-CANONICAL-RUNTIME.zip | 3257 bytes
 - RESTORE-RUNTIME.py | 1236 bytes
 - RUNTIME-MANIFEST.json | 1226 bytes
 - confidence_contract.py | 1024 bytes
 - dataset-metadata.json | 153 bytes
 - routing_guard.py | 1641 bytes
 - skill_quarantine.py | 1354 bytes

KAGGLE CLI CHECK
Kaggle CLI 2.0.2
Credential file: NOT_FOUND
Environment auth: NOT_FOUND

STATUS: PUBLISH_PACKAGE_READY_AUTH_NOT_AVAILABLE

The package is ready but this kernel does not expose
Kaggle API credentials.

DO NOT RESTART.
SAVE VERSION before closing the session.


SystemExit: 0

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [3]:
from __future__ import annotations

from pathlib import Path
import hashlib
import json
import shutil
import sys
import zipfile

print("=" * 120)
print("RAIOS V8.6.2 — DURABLE RESTORE + HASH VERIFY + R3 GO/NO-GO")
print("CPU ONLY | READ INPUTS | NO MODEL | NO TRAINING | NO PROMOTION | FAIL CLOSED")
print("=" * 120)

# ============================================================================
# CERTIFIED BASELINE
# ============================================================================

EXPECTED = {
    "confidence_contract.py":
        "2aef4fce7ac6e7720ebb987f7fa8dc1695300a26ed2f84c84fb515d71d856fcc",

    "skill_quarantine.py":
        "1f6e8ef7fdb42113f8e281bc5407e743bbb8fa1be10c9913d071f9e6adbffea5",

    "routing_guard.py":
        "7ca061d1d8530a9d7c77e2bfee4b97bb38445613adf6df1df72a291c47a0dc45",
}

EXPECTED_ZIP = (
    "f222c65b4cbb570068e11eb54fe0a665"
    "7197e1c78703b39ca7aac694364671e3"
)

INVALID_SKILL_HASH = (
    "683ca22e97a2cb919ab078321076bd06"
    "b146b1bf69e937a2062d68755e993b65"
)

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working/RAIOS-V8.6.2")
RUNTIME = WORK / "runtime"
REPORTS = WORK / "reports"

RUNTIME.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)


def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


# ============================================================================
# 1. DISCOVER DURABLE SOURCES
# ============================================================================

print("\n[1/6] DURABLE SOURCE DISCOVERY")
print("-" * 120)

if not INPUT.exists():
    raise RuntimeError("/kaggle/input does not exist.")

zip_candidates = list(
    INPUT.rglob("RAIOS-V8.6.2-CANONICAL-RUNTIME.zip")
)

manifest_candidates = list(
    INPUT.rglob("RUNTIME-MANIFEST.json")
)

runtime_candidates = {
    name: list(INPUT.rglob(name))
    for name in EXPECTED
}

print("ZIP candidates     :", len(zip_candidates))
print("Manifest candidates:", len(manifest_candidates))

for name, candidates in runtime_candidates.items():
    print(
        f"{name:<28}:",
        len(candidates)
    )


# ============================================================================
# 2. FIND EXACT CERTIFIED FILES BY HASH
# ============================================================================

print("\n[2/6] EXACT HASH RESOLUTION")
print("-" * 120)

resolved = {}

for name, expected_hash in EXPECTED.items():

    matches = []

    for path in runtime_candidates[name]:

        try:
            digest = sha256(path)
        except Exception:
            continue

        if digest == expected_hash:
            matches.append(path)

    if matches:
        resolved[name] = matches[0]

        print(
            "PASS |",
            name,
            "|",
            matches[0]
        )

    else:
        print(
            "MISS |",
            name,
            "| expected",
            expected_hash
        )


# ============================================================================
# 3. ZIP FALLBACK
# ============================================================================

if len(resolved) != len(EXPECTED):

    print("\n[3/6] EXACT ZIP FALLBACK")
    print("-" * 120)

    exact_zip = None

    for candidate in zip_candidates:

        try:
            digest = sha256(candidate)
        except Exception:
            continue

        print(
            candidate,
            "|",
            digest
        )

        if digest == EXPECTED_ZIP:
            exact_zip = candidate
            break

    if exact_zip:

        print("\nPASS | Certified ZIP found:")
        print(exact_zip)

        extract_root = WORK / "certified-zip-restore"

        if extract_root.exists():
            shutil.rmtree(extract_root)

        extract_root.mkdir(
            parents=True,
            exist_ok=True
        )

        with zipfile.ZipFile(
            exact_zip,
            "r"
        ) as zf:
            zf.extractall(extract_root)

        for name, expected_hash in EXPECTED.items():

            matches = list(
                extract_root.rglob(name)
            )

            for path in matches:

                if sha256(path) == expected_hash:
                    resolved[name] = path
                    break

    else:

        print(
            "No ZIP matching certified SHA256 found."
        )

else:

    print("\n[3/6] ZIP FALLBACK")
    print("-" * 120)
    print("SKIP | Exact individual runtime files already resolved.")


# ============================================================================
# FAIL CLOSED IF DURABLE COPY IS ABSENT
# ============================================================================

missing = [
    name
    for name in EXPECTED
    if name not in resolved
]

if missing:

    print("\n" + "=" * 120)
    print("R3 GO/NO-GO: NO-GO")
    print("=" * 120)

    print("\nMissing certified durable runtime:")

    for name in missing:
        print(" -", name)

    print("""
STATUS: CERTIFIED_RUNTIME_NOT_DURABLY_AVAILABLE

DO NOT ENABLE T4.
DO NOT RUN R3.
DO NOT REBUILD AUTOMATICALLY.

The previous Quick Save did not expose the certified runtime
inside /kaggle/input for this session.
""")

    raise SystemExit(2)


# ============================================================================
# 4. RESTORE INTO /kaggle/working
# ============================================================================

print("\n[4/6] RUNTIME RESTORE")
print("-" * 120)

for name, source in resolved.items():

    target = RUNTIME / name

    shutil.copy2(
        source,
        target
    )

    actual = sha256(target)
    expected = EXPECTED[name]

    if actual != expected:
        raise RuntimeError(
            f"Post-restore hash mismatch: {name}"
        )

    print(
        "PASS |",
        name,
        "|",
        actual
    )


# ============================================================================
# 5. FUNCTIONAL SAFETY REVALIDATION
# ============================================================================

print("\n[5/6] FUNCTIONAL SAFETY REVALIDATION")
print("-" * 120)

sys.path.insert(
    0,
    str(RUNTIME)
)

for module_name in [
    "confidence_contract",
    "skill_quarantine",
    "routing_guard",
]:
    sys.modules.pop(
        module_name,
        None
    )

from confidence_contract import (
    validate_confidence,
    ConfidenceContractError,
)

from skill_quarantine import (
    effective_skill_state,
)

from routing_guard import (
    guard_skill_route,
)

tests = []


def record(name, passed, detail=""):

    tests.append({
        "name": name,
        "passed": bool(passed),
        "detail": str(detail),
    })

    print(
        "PASS" if passed else "FAIL",
        "|",
        name,
        "|",
        detail
    )


# Valid confidence domain

for value in [0, 0.7, 1]:

    try:
        result = validate_confidence(value).value

        record(
            f"confidence_accept_{value}",
            result == float(value),
            result
        )

    except Exception as exc:

        record(
            f"confidence_accept_{value}",
            False,
            exc
        )


# Invalid confidence values

for label, value in [
    ("70", 70),
    ("14_79", 14.79),
    ("negative", -0.1),
    ("over_one", 1.1),
    ("boolean", True),
    ("none", None),
    ("string", "0.7"),
    ("nan", float("nan")),
    ("inf", float("inf")),
]:

    try:

        validate_confidence(value)

        record(
            f"confidence_reject_{label}",
            False,
            "unexpected acceptance"
        )

    except ConfidenceContractError as exc:

        record(
            f"confidence_reject_{label}",
            True,
            exc
        )


# Historical invalid skill quarantine

state = effective_skill_state(
    INVALID_SKILL_HASH,
    "PROMOTED"
)

record(
    "historical_skill_quarantine",
    (
        state.usable is False
        and
        state.effective_status
        == "QUARANTINED_PENDING_REVALIDATION"
    ),
    state
)


# ZERO_LLM must be blocked

decision = guard_skill_route(
    "ZERO_LLM",
    INVALID_SKILL_HASH,
    "PROMOTED"
)

record(
    "historical_skill_zero_llm_block",
    (
        decision.allowed is False
        and
        decision.effective_route == "REASONING"
    ),
    decision
)


# ZERO_LLM cannot execute without skill

decision2 = guard_skill_route(
    "ZERO_LLM",
    None,
    None
)

record(
    "zero_llm_without_skill_block",
    (
        decision2.allowed is False
        and
        decision2.effective_route == "REASONING"
    ),
    decision2
)


failed = [
    t
    for t in tests
    if not t["passed"]
]

if failed:

    raise RuntimeError(
        f"Functional safety verification failed: "
        f"{len(failed)} test(s)"
    )


# ============================================================================
# 6. GO / NO-GO RECEIPT
# ============================================================================

print("\n[6/6] R3 READINESS RECEIPT")
print("-" * 120)

receipt = {
    "schema":
        "raios.v8.6.2.r3-go-no-go.v1",

    "runtime": {
        name: {
            "sha256": sha256(
                RUNTIME / name
            ),
            "source": str(
                resolved[name]
            ),
        }
        for name in EXPECTED
    },

    "functional_tests": {
        "total": len(tests),
        "passed": len(tests),
        "failed": 0,
    },

    "historical_invalid_skill": {
        "content_hash":
            INVALID_SKILL_HASH,

        "effective_status":
            state.effective_status,

        "runtime_usable":
            state.usable,
    },

    "r3_gate":
        "GO",

    "gpu_required_for_this_cell":
        False,

    "model_loaded":
        False,

    "training":
        False,

    "promotion":
        False,
}

receipt_path = (
    REPORTS
    / "v8.6.2-r3-go-no-go.json"
)

receipt_path.write_text(
    json.dumps(
        receipt,
        indent=2
    ),
    encoding="utf-8"
)

print("\n" + "=" * 120)
print("RAIOS V8.6.2 — R3 GO/NO-GO RESULT")
print("=" * 120)

print(
    "Runtime hashes : 3 / 3 EXACT"
)

print(
    "Safety tests   :",
    f"{len(tests)} / {len(tests)} PASS"
)

print(
    "Invalid skill  :",
    state.effective_status
)

print(
    "Runtime usable :",
    state.usable
)

print(
    "Receipt        :",
    receipt_path
)

print()
print("R3 GATE        : GO")
print("STATUS         : READY_FOR_R3_T4")
print("GPU USED       : NO")
print("MODEL LOADED   : NO")
print("TRAINING       : NO")
print("PROMOTED       : NO")

RAIOS V8.6.2 — DURABLE RESTORE + HASH VERIFY + R3 GO/NO-GO
CPU ONLY | READ INPUTS | NO MODEL | NO TRAINING | NO PROMOTION | FAIL CLOSED

[1/6] DURABLE SOURCE DISCOVERY
------------------------------------------------------------------------------------------------------------------------
ZIP candidates     : 0
Manifest candidates: 0
confidence_contract.py      : 0
skill_quarantine.py         : 0
routing_guard.py            : 0

[2/6] EXACT HASH RESOLUTION
------------------------------------------------------------------------------------------------------------------------
MISS | confidence_contract.py | expected 2aef4fce7ac6e7720ebb987f7fa8dc1695300a26ed2f84c84fb515d71d856fcc
MISS | skill_quarantine.py | expected 1f6e8ef7fdb42113f8e281bc5407e743bbb8fa1be10c9913d071f9e6adbffea5
MISS | routing_guard.py | expected 7ca061d1d8530a9d7c77e2bfee4b97bb38445613adf6df1df72a291c47a0dc45

[3/6] EXACT ZIP FALLBACK
------------------------------------------------------------------------------------

SystemExit: 2

In [1]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import shutil
import sys
import zipfile

print("=" * 120)
print("RAIOS V8.6.2 — CANONICAL RUNTIME CPU REBUILD + PERSISTENCE PACKAGE")
print("ACCELERATOR NONE | NO MODEL | NO TRAINING | NO PROMOTION")
print("=" * 120)

ROOT = Path("/kaggle/working/RAIOS-V8.6.2")
RUNTIME = ROOT / "runtime"
PACKAGE = ROOT / "PERSIST-RAIOS-V862-RUNTIME"
REPORTS = ROOT / "reports"

for p in [RUNTIME, PACKAGE, REPORTS]:
    p.mkdir(parents=True, exist_ok=True)

INVALID_SKILL_HASH = (
    "683ca22e97a2cb919ab078321076bd06"
    "b146b1bf69e937a2062d68755e993b65"
)

# -------------------------------------------------------------------
# Canonical sources corresponding to the last certified CPU package
# -------------------------------------------------------------------

confidence_source = r'''from __future__ import annotations

import math
from dataclasses import dataclass
from typing import Any


class ConfidenceContractError(ValueError):
    pass


@dataclass(frozen=True)
class ConfidenceValue:
    value: float


def validate_confidence(value: Any) -> ConfidenceValue:
    if isinstance(value, bool):
        raise ConfidenceContractError(
            "Boolean is not a valid confidence value."
        )

    if not isinstance(value, (int, float)):
        raise ConfidenceContractError(
            f"Unsupported confidence type: {type(value).__name__}"
        )

    numeric = float(value)

    if not math.isfinite(numeric):
        raise ConfidenceContractError(
            "Confidence must be finite."
        )

    if numeric < 0.0 or numeric > 1.0:
        raise ConfidenceContractError(
            f"Confidence violates canonical [0,1] domain: {value}"
        )

    return ConfidenceValue(value=numeric)


def require_confidence(value: Any) -> float:
    return validate_confidence(value).value
'''

quarantine_source = f'''from __future__ import annotations

from dataclasses import dataclass
from typing import Optional

INVALIDATED_REPOSITORY_ANALYSIS_SKILL = "{INVALID_SKILL_HASH}"

QUARANTINE_STATUS = "QUARANTINED_PENDING_REVALIDATION"


@dataclass(frozen=True)
class EffectiveSkillState:
    content_hash: str
    stored_status: Optional[str]
    effective_status: str
    usable: bool
    reason: str


def effective_skill_state(
    content_hash: str,
    stored_status: Optional[str],
) -> EffectiveSkillState:

    if content_hash == INVALIDATED_REPOSITORY_ANALYSIS_SKILL:
        return EffectiveSkillState(
            content_hash=content_hash,
            stored_status=stored_status,
            effective_status=QUARANTINE_STATUS,
            usable=False,
            reason="Historical promotion depended on invalid confidence scale.",
        )

    normalized = (stored_status or "UNKNOWN").upper()

    usable = normalized in {{
        "PROMOTED",
        "ACTIVE",
        "VALIDATED",
    }}

    return EffectiveSkillState(
        content_hash=content_hash,
        stored_status=stored_status,
        effective_status=normalized,
        usable=usable,
        reason=(
            "Skill is runtime usable."
            if usable
            else "Skill is not in a runtime-usable state."
        ),
    )
'''

routing_source = r'''from __future__ import annotations

from dataclasses import dataclass
from typing import Optional

from skill_quarantine import effective_skill_state


@dataclass(frozen=True)
class RoutingDecision:
    requested_route: str
    effective_route: str
    allowed: bool
    reason: str
    skill_hash: Optional[str]


def guard_skill_route(
    requested_route: str,
    skill_hash: Optional[str] = None,
    stored_status: Optional[str] = None,
) -> RoutingDecision:

    requested = str(requested_route or "").strip().upper()

    if requested != "ZERO_LLM":
        return RoutingDecision(
            requested_route=requested,
            effective_route=requested or "REASONING",
            allowed=True,
            reason="ROUTE_NOT_SUBJECT_TO_ZERO_LLM_SKILL_GATE",
            skill_hash=skill_hash,
        )

    if not skill_hash:
        return RoutingDecision(
            requested_route=requested,
            effective_route="REASONING",
            allowed=False,
            reason="ZERO_LLM_REQUIRES_VALIDATED_SKILL",
            skill_hash=None,
        )

    state = effective_skill_state(
        skill_hash,
        stored_status,
    )

    if not state.usable:
        return RoutingDecision(
            requested_route=requested,
            effective_route="REASONING",
            allowed=False,
            reason=f"SKILL_NOT_RUNTIME_USABLE:{state.effective_status}",
            skill_hash=skill_hash,
        )

    return RoutingDecision(
        requested_route=requested,
        effective_route="ZERO_LLM",
        allowed=True,
        reason="SKILL_RUNTIME_USABLE",
        skill_hash=skill_hash,
    )
'''

sources = {
    "confidence_contract.py": confidence_source,
    "skill_quarantine.py": quarantine_source,
    "routing_guard.py": routing_source,
}

# -------------------------------------------------------------------
# Write
# -------------------------------------------------------------------

for name, source in sources.items():
    (RUNTIME / name).write_text(
        source,
        encoding="utf-8",
        newline="\n",
    )

def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

hashes = {
    name: sha256(RUNTIME / name)
    for name in sources
}

print("\nRUNTIME HASHES")
for name, digest in hashes.items():
    print(name, digest)

# -------------------------------------------------------------------
# Functional verification
# -------------------------------------------------------------------

sys.path.insert(0, str(RUNTIME))

for mod in [
    "confidence_contract",
    "skill_quarantine",
    "routing_guard",
]:
    sys.modules.pop(mod, None)

from confidence_contract import validate_confidence, ConfidenceContractError
from skill_quarantine import effective_skill_state
from routing_guard import guard_skill_route

tests = []

def record(name, ok, detail):
    tests.append({
        "name": name,
        "pass": bool(ok),
        "detail": str(detail),
    })
    print("PASS" if ok else "FAIL", "|", name, "|", detail)

for v in [0, 0.7, 1]:
    try:
        r = validate_confidence(v).value
        record(f"accept_{v}", r == float(v), r)
    except Exception as exc:
        record(f"accept_{v}", False, exc)

for label, v in [
    ("70", 70),
    ("14_79", 14.79),
    ("negative", -0.1),
    ("over_one", 1.1),
    ("bool", True),
    ("none", None),
    ("string", "0.7"),
    ("nan", float("nan")),
    ("inf", float("inf")),
]:
    try:
        validate_confidence(v)
        record(f"reject_{label}", False, "unexpected acceptance")
    except ConfidenceContractError as exc:
        record(f"reject_{label}", True, exc)

state = effective_skill_state(
    INVALID_SKILL_HASH,
    "PROMOTED",
)

record(
    "invalid_skill_quarantined",
    state.usable is False
    and state.effective_status == "QUARANTINED_PENDING_REVALIDATION",
    state,
)

route = guard_skill_route(
    "ZERO_LLM",
    INVALID_SKILL_HASH,
    "PROMOTED",
)

record(
    "invalid_skill_zero_llm_blocked",
    route.allowed is False
    and route.effective_route == "REASONING",
    route,
)

missing_route = guard_skill_route(
    "ZERO_LLM",
    None,
    None,
)

record(
    "zero_llm_without_skill_blocked",
    missing_route.allowed is False
    and missing_route.effective_route == "REASONING",
    missing_route,
)

failed = [x for x in tests if not x["pass"]]

if failed:
    raise RuntimeError(
        f"Runtime verification failed: {len(failed)} test(s)"
    )

# -------------------------------------------------------------------
# Persistence package
# -------------------------------------------------------------------

if PACKAGE.exists():
    shutil.rmtree(PACKAGE)

PACKAGE.mkdir(parents=True)

for name in sources:
    shutil.copy2(
        RUNTIME / name,
        PACKAGE / name,
    )

manifest = {
    "schema": "raios.v8.6.2.runtime-persistence.v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "version": "V8.6.2",
    "files": {
        name: {
            "sha256": hashes[name],
            "bytes": (RUNTIME / name).stat().st_size,
        }
        for name in sources
    },
    "tests": {
        "total": len(tests),
        "passed": len(tests),
        "failed": 0,
    },
    "invalidated_skill": {
        "content_hash": INVALID_SKILL_HASH,
        "effective_status": "QUARANTINED_PENDING_REVALIDATION",
    },
    "automatic_promotion": False,
}

(PACKAGE / "RUNTIME-MANIFEST.json").write_text(
    json.dumps(
        manifest,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

restore = r'''from pathlib import Path
import hashlib
import json
import shutil

SOURCE_ROOT = Path("/kaggle/input")
TARGET = Path("/kaggle/working/RAIOS-V8.6.2/runtime")

matches = list(SOURCE_ROOT.rglob("RUNTIME-MANIFEST.json"))

for manifest_path in matches:
    try:
        manifest = json.loads(
            manifest_path.read_text(encoding="utf-8")
        )
    except Exception:
        continue

    if manifest.get("schema") != "raios.v8.6.2.runtime-persistence.v1":
        continue

    source = manifest_path.parent
    TARGET.mkdir(parents=True, exist_ok=True)

    for filename, meta in manifest["files"].items():
        src = source / filename
        if not src.exists():
            raise RuntimeError(f"Missing runtime file: {src}")

        digest = hashlib.sha256(src.read_bytes()).hexdigest()

        if digest != meta["sha256"]:
            raise RuntimeError(f"Hash mismatch: {filename}")

        shutil.copy2(src, TARGET / filename)

    print("STATUS: RAIOS_V862_RUNTIME_RESTORED")
    raise SystemExit(0)

raise RuntimeError("No certified V8.6.2 runtime package found.")
'''

(PACKAGE / "RESTORE-RUNTIME.py").write_text(
    restore,
    encoding="utf-8",
    newline="\n",
)

ZIP = ROOT / "RAIOS-V8.6.2-PERSISTENCE-PACKAGE.zip"

if ZIP.exists():
    ZIP.unlink()

with zipfile.ZipFile(
    ZIP,
    "w",
    zipfile.ZIP_DEFLATED,
) as zf:
    for p in sorted(PACKAGE.iterdir()):
        if p.is_file():
            zf.write(
                p,
                arcname=p.name,
            )

zip_hash = sha256(ZIP)

receipt = {
    "status": "CPU_RUNTIME_READY_FOR_PERSISTENCE",
    "runtime_hashes": hashes,
    "zip": str(ZIP),
    "zip_sha256": zip_hash,
    "tests_passed": len(tests),
    "tests_failed": 0,
    "gpu_required": False,
    "training": False,
    "promotion": False,
}

receipt_path = (
    REPORTS
    / "v8.6.2-cpu-runtime-persistence.json"
)

receipt_path.write_text(
    json.dumps(
        receipt,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("\n" + "=" * 120)
print("V8.6.2 CPU RUNTIME PERSISTENCE RESULT")
print("=" * 120)

print("Tests      :", len(tests), "/", len(tests), "PASS")
print("Package dir:", PACKAGE)
print("ZIP        :", ZIP)
print("ZIP SHA256 :", zip_hash)
print("Receipt    :", receipt_path)

print()
print("STATUS: CPU_RUNTIME_READY_FOR_PERSISTENCE")
print("ACCELERATOR: NONE")
print("MODEL      : NOT LOADED")
print("TRAINING   : NO")
print("PROMOTION  : NO")

RAIOS V8.6.2 — CANONICAL RUNTIME CPU REBUILD + PERSISTENCE PACKAGE
ACCELERATOR NONE | NO MODEL | NO TRAINING | NO PROMOTION

RUNTIME HASHES
confidence_contract.py 2aef4fce7ac6e7720ebb987f7fa8dc1695300a26ed2f84c84fb515d71d856fcc
skill_quarantine.py 1f6e8ef7fdb42113f8e281bc5407e743bbb8fa1be10c9913d071f9e6adbffea5
routing_guard.py 7ca061d1d8530a9d7c77e2bfee4b97bb38445613adf6df1df72a291c47a0dc45
PASS | accept_0 | 0.0
PASS | accept_0.7 | 0.7
PASS | accept_1 | 1.0
PASS | reject_70 | Confidence violates canonical [0,1] domain: 70
PASS | reject_14_79 | Confidence violates canonical [0,1] domain: 14.79
PASS | reject_negative | Confidence violates canonical [0,1] domain: -0.1
PASS | reject_over_one | Confidence violates canonical [0,1] domain: 1.1
PASS | reject_bool | Boolean is not a valid confidence value.
PASS | reject_none | Unsupported confidence type: NoneType
PASS | reject_string | Unsupported confidence type: str
PASS | reject_nan | Confidence must be finite.
PASS | reject_inf | Confiden